This notebook can be run in Google Colab by setting `colab = True` in the first cell of **Setup** below. The second cell will then install all relevant libraries to the current Colab session, which will take several minutes. Partway though, Google Colab will ask you to restart the session -- click "Cancel" and the code will continue running. At the end of the cell, it will restart automatically, after which you can run the following cells (no need to rerun the first two cells).

If running locally or on another service, make sure all libraries and models are installed according to the instructions on the [github Readme](https://github.com/lu-liang-geo/Sam-Lidar/blob/main/README.md).


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lu-liang-geo/UAV_Tree_Detection/blob/main/notebooks/NEON_Tree_Evaluation_EDA.ipynb)

# Setup

In [ ]:
# If running in colab, set colab = True, else set colab = False.
colab = True

In [ ]:
if colab == True:
    # Install DeepForest, Detectree2, SAM 2, and this github
    !pip install deepforest                                                         # DeepForest
    !pip install 'git+https://github.com/PatBall1/detectree2.git'                   # Detectree2
    !pip install 'git+https://github.com/facebookresearch/segment-anything-2.git'   # SAM 2
    !pip install 'git+https://github.com/lu-liang-geo/Sam-Lidar.git'                # our code

    # Download weights for Detectree2 and SAM 2
    !mkdir /content/sam2_weights
    !wget -q -P /content/sam2_weights https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

    !mkdir /content/detectree2_weights
    !wget -q -P /content/detectree2_weights https://zenodo.org/records/12773341/files/230103_randresize_full.pth

    # Download annotated data
    !pip install zenodo_get
    !zenodo_get https://zenodo.org/records/15133614
    !unzip '/content/data.zip' -d '/content'

    # Restart runtime (continue to next cell after this)
    quit()

In [ ]:
# Same as first block, necessary after the colab session restarts if colab == True
colab = True

In [ ]:
# Import standard libraries
import os
import torch
import timeit
import pickle
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.optimize import linear_sum_assignment
from IPython.display import display

In [ ]:
# Import installed libraries
import supervision as sv

from deepforest import main

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from detectron2.engine import DefaultPredictor
from detectree2.models.train import setup_cfg

from sam_lidar import (
    NEONTreeDataset,
    rasterize_lidar,
    lidar_filter,
    sample_points,
    segment_boxes,
    segment_box_lidar,
    custom_nms,
    per_tree_metrics,
    per_tree_std
)

In [ ]:
# Set device, either CPU or CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# Load SAM Model. We're currently using SAM-2
if colab == True:
    sam_weights = "/content/sam2_weights/sam2_hiera_large.pt"
else:
    sam_weights = "PATH/TO/SAM2/WEIGHTS.pt"
    
sam_model = build_sam2("sam2_hiera_l.yaml", sam_weights, device=device)
sam_predictor = SAM2ImagePredictor(sam_model)

In [ ]:
# Load Detectree2 Model
if colab == True:
    detectree_weights = "/content/detectree2_weights/230103_randresize_full.pth"
else:
    detectree_weights = "PATH/TO/DETECTREE2/WEIGHTS.pth"

cfg = setup_cfg(update_model=detectree_weights)
cfg.MODEL.DEVICE = device.type
detectree_model=DefaultPredictor(cfg)

In [ ]:
# Load DeepForest Model
deepforest_model = main.deepforest()
deepforest_model.use_release()
deepforest_model.to(device)
print()

In [ ]:
# Define dataset
if colab == True:
    rgb_path = "/content/Sam-Lidar/data/RGB"
    lidar_path = "/content/Sam-Lidar/data/LiDAR"
    ann_path = "/content/Sam-Lidar/data/Annotations"
else:
    rgb_path = "PATH/TO/RGB/FOLDER"
    lidar_path = "PATH/TO/LIDAR/FOLDER"
    ann_path = "PATH/TO/ANNOTATION/FOLDER"

dataset = NEONTreeDataset(rgb_folder=rgb_path, lidar_folder=lidar_path ann_folder=ann_path)

# Set Image Paths / Random Seed

In [ ]:
# Set random seed for reproducibility. Seed used for paper is 1.
seed = 1

# Define test images and classifications
test_imgs = [('2018_SJER_3_252000_4106000_image_234', 'Low'), ('2018_SJER_3_254000_4108000_image_700', 'Low'),
             ('2018_TEAK_3_318000_4107000_image_718', 'Low'), ('DSNY_002_2018', 'Low'), ('JERC_055_2018', 'Low'),
             ('OSBS_051_2019', 'Low'), ('ABBY_063_2019', 'Medium'), ('BONA_018_2019', 'Medium'), ('DSNY_025_2018', 'Medium'),
             ('SJER_025_2018', 'Medium'), ('TEAK_049_2018', 'Medium'), ('WREF_072_2019', 'Medium'), ('ABBY_076_2019', 'High'),
             ('BART_050_2019', 'High'), ('BONA_005_2019', 'High'), ('DELA_047_2019', 'High'), ('LENO_066_2019', 'High'),
             ('SERC_004_2019', 'High')]

# Define metrics dictionary that will be updated below
metrics = dict()

Print number of trees in each density class.

In [ ]:
trees = {'Low':dict(), 'Medium':dict(), 'High':dict()}

for filename, density in test_imgs:
  img = dataset.get_image(filename)
  true_boxes = img['box']
  trees[density][filename] = len(true_boxes)

total_trees = 0
for density in ['Low','Medium','High']:
  density_trees = 0
  print(density)
  for filename, number in trees[density].items():
    print(filename, ':', number)
    density_trees += number
    total_trees += number
  print()
  print(density, 'Trees :', density_trees)
  print()
print('Total Trees :', total_trees)

# Methods

## Generating Bounding Boxes

In [ ]:
boxes = dict()
for method in ['Manual','DeepForest','Detectree2','Baseline']:
    boxes[method] = dict()
    for density in ['Low','Medium','High']:
        boxes[method][density] = dict()

for filename, density in test_imgs:
    # Get image
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()

    # Collect hand-drawn bounding boxes
    manual_boxes = img['box']

    # Generate automatic bounding boxes using DeepForest
    deepforest_preds = deepforest_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
    # Filter with confidence threshold of 0.2
    if deepforest_preds is not None:
        deepforest_preds = deepforest_preds[deepforest_preds['score'] >= 0.2]
        deepforest_scores = deepforest_preds['score'].to_numpy()
        deepforest_boxes = deepforest_preds[['xmin','ymin','xmax','ymax']].to_numpy()
    else:
        deepforest_scores = np.empty((0))
        deepforest_boxes = np.empty((0,4))

  # Generate automatic bounding boxes using Detectree2
    detectree_preds = detectree_model(bgr_img)
    if detectree_preds is not None:
        detectree_boxes = detectree_preds['instances'].pred_boxes.tensor.cpu().numpy()
        detectree_scores = detectree_preds['instances'].scores.cpu().numpy()
        detectree_classes = detectree_preds['instances'].pred_classes.cpu().numpy()
        baseline_masks = detectree_preds['instances'].pred_masks.cpu().numpy()
        # Filter with confidence threshold of 0.2
        detectree_idx = detectree_scores >= 0.2
        detectree_boxes = detectree_boxes[detectree_idx]
        detectree_scores = detectree_scores[detectree_idx]
        detectree_classes = detectree_classes[detectree_idx]
        baseline_masks = baseline_masks[detectree_idx]
    else:
        detectree_boxes = np.empty((0,4))
        detectree_scores = np.empty((0))
        detectree_classes = np.empty((0))
        baseline_masks = np.empty(0,400,400)

    # Create detections objects for boxes
    manual_detections = sv.Detections(xyxy=manual_boxes, confidence=np.ones(len(manual_boxes)),
                                      class_id=np.zeros(len(manual_boxes), dtype='int'))
    deepforest_detections = sv.Detections(xyxy=deepforest_boxes, confidence=deepforest_scores,
                                          class_id=np.zeros(len(deepforest_boxes), dtype='int'))
    detectree_detections = sv.Detections(xyxy=detectree_boxes, confidence=detectree_scores,
                                         class_id=detectree_classes)
    baseline_detections = sv.Detections(xyxy=detectree_boxes, confidence=detectree_scores,
                                             class_id=detectree_classes)

    # Apply custom non-max suppression to automatic boxes
    deepforest_idx = custom_nms(deepforest_detections, threshold=0.5)
    detectree_idx = custom_nms(detectree_detections, threshold=0.5)
    deepforest_detections = deepforest_detections[deepforest_idx]
    detectree_detections = detectree_detections[detectree_idx]
    baseline_detections = baseline_detections[detectree_idx]

    # Store predicted bounding boxes
    boxes['Manual'][density][filename] = {'Boxes':manual_detections}
    boxes['DeepForest'][density][filename] = {'Boxes':deepforest_detections}
    boxes['Detectree2'][density][filename] = {'Boxes':detectree_detections}
    boxes['Baseline'][density][filename] = {'Boxes':baseline_detections, 'Masks':baseline_masks}

## Incorporating LiDAR Data

The cell above in **Generating Bounding Boxes** must be run first.

In [ ]:
# Set number of positive and negative LiDAR points to sample. Number used in paper is 10 positive and 10 negative points.
pos = 10
neg = 10

lidar = dict()
for method in ['Manual','DeepForest','Detectree2','Baseline']:
    lidar[method] = dict()
    for density in ['Low','Medium','High']:
        lidar[method][density] = dict()

for filename, density in test_imgs:
    # Get image
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()

    # Retrieve predicted bounding boxes, scores, and classes
    manual_boxes = boxes['Manual'][density][filename]['Boxes'].xyxy
    deepforest_boxes = boxes['DeepForest'][density][filename]['Boxes'].xyxy
    detectree_boxes = boxes['Detectree2'][density][filename]['Boxes'].xyxy
    baseline_boxes = boxes['Baseline'][density][filename]['Boxes'].xyxy

    deepforest_scores = boxes['DeepForest'][density][filename]['Boxes'].confidence
    detectree_scores = boxes['Detectree2'][density][filename]['Boxes'].confidence
    baseline_scores = boxes['Baseline'][density][filename]['Boxes'].confidence

    detectree_classes = boxes['Detectree2'][density][filename]['Boxes'].class_id
    baseline_classes = boxes['Baseline'][density][filename]['Boxes'].confidence

    baseline_masks = boxes['Baseline'][density][filename]['Masks']

    # Use boxes to label LiDAR points
    manual_coords, manual_labels = rasterize_lidar(lidar_path, rgb_path, filename, boxes=manual_boxes)
    deepforest_coords, deepforest_labels = rasterize_lidar(lidar_path, rgb_path, filename, boxes=deepforest_boxes)
    detectree_coords, detectree_labels = rasterize_lidar(lidar_path, rgb_path, filename, boxes=detectree_boxes)

    # Use labeled LiDAR points to filter boxes
    idx = lidar_filter(deepforest_boxes, deepforest_coords, deepforest_labels, threshold=0.2)
    deepforest_boxes, deepforest_labels = deepforest_boxes[idx], deepforest_labels[idx]
    deepforest_scores = deepforest_scores[idx]

    idx = lidar_filter(detectree_boxes, detectree_coords, detectree_labels, threshold=0.2)
    detectree_boxes, detectree_labels = detectree_boxes[idx], detectree_labels[idx],
    detectree_scores, detectree_classes = detectree_scores[idx], detectree_classes[idx]

    baseline_boxes, baseline_masks = baseline_boxes[idx], baseline_masks[idx]
    baseline_scores, baseline_classes = baseline_scores[idx], baseline_classes[idx]

    # Create detections objects
    manual_detections = sv.Detections(xyxy=manual_boxes, confidence=np.ones(len(manual_boxes)), 
                                      class_id=np.zeros(len(manual_boxes), dtype='int'))
    deepforest_detections = sv.Detections(xyxy=deepforest_boxes, confidence=deepforest_scores, 
                                          class_id=np.zeros(len(deepforest_boxes), dtype='int'))
    detectree_detections = sv.Detections(xyxy=detectree_boxes, confidence=detectree_scores, 
                                         class_id=detectree_classes)
    baseline_detections = sv.Detections(xyxy=baseline_boxes, confidence=baseline_scores,
                                        class_id=baseline_classes)

    # Sample LiDAR points for SAM-2. Parameters used in paper are distance_weight=True, neg_sample_spread=1, seed=1 (set above).
    manual_coords, manual_labels = sample_points(manual_coords, manual_labels, pos_samples=pos, neg_samples=neg, 
                                                 distance_weight=True, neg_sample_spread=1, seed=seed)
    if len(deepforest_boxes) > 0:
        deepforest_coords, deepforest_labels = sample_points(deepforest_coords, deepforest_labels, pos_samples=pos, neg_samples=neg, 
                                                             distance_weight=True, neg_sample_spread=1, seed=seed)
    else:
        deepforest_coords, deepforest_labels = None, None

    if len(detectree_boxes) > 0:
        detectree_coords, detectree_labels = sample_points(detectree_coords, detectree_labels, pos_samples=pos, neg_samples=neg, 
                                                           distance_weight=True, neg_sample_spread=1, seed=seed)
    else:
        detectree_coords, detectree_labels = None, None
        
    lidar['Manual'][density][filename] = {'Boxes':manual_detections, 'LiDAR':(manual_coords, manual_labels)}
    lidar['DeepForest'][density][filename] = {'Boxes':deepforest_detections, 'LiDAR':(deepforest_coords, deepforest_labels)}
    lidar['Detectree2'][density][filename] = {'Boxes':detectree_detections, 'LiDAR':(detectree_coords, detectree_labels)}
    lidar['Baseline'][density][filename] = {'Boxes':baseline_detections, 'Masks':baseline_masks}

## Prompting SAM

The cells above, **Generating Bounding Boxes** and **Incorporating LiDAR Data** must be run first.

In [ ]:
for filename, density in test_imgs:
    # Get image
    img = dataset.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()
    sam_predictor.set_image(rgb_img)

    # Prompt SAM with Bounding Boxes
    manual_boxes = boxes['Manual'][density][filename]['Boxes']
    deepforest_boxes = boxes['DeepForest'][density][filename]['Boxes']
    detectree_boxes = boxes['Detectree2'][density][filename]['Boxes']

    manual_masks, manual_scores = segment_boxes(sam_predictor, manual_boxes.xyxy)
    if len(deepforest_boxes) > 0:
        deepforest_masks, deepforest_scores = segment_boxes(sam_predictor, deepforest_boxes.xyxy)
    else:
        deepforest_masks, deepforest_scores = np.empty((0,400,400)), np.empty((0))
    if len(detectree_boxes) > 0:
        detectree_masks, detectree_scores = segment_boxes(sam_predictor, detectree_boxes.xyxy)
    else:
        detectree_masks, detectree_scores = np.empty((0,400,400)), np.empty((0))

    # Filter masks with confidence threshold of 0.2
    idx = manual_scores >= 0.2
    boxes['Manual'][density][filename]['Boxes'] = manual_boxes[idx]
    boxes['Manual'][density][filename]['Masks'] = manual_masks[idx]

    idx = deepforest_scores >= 0.2
    boxes['DeepForest'][density][filename]['Boxes'] = deepforest_boxes[idx]
    boxes['DeepForest'][density][filename]['Masks'] = deepforest_masks[idx]

    idx = detectree_scores >= 0.2
    boxes['Detectree2'][density][filename]['Boxes'] = detectree_boxes[idx]
    boxes['Detectree2'][density][filename]['Masks'] = detectree_masks[idx]

    # Prompt SAM with Bounding Boxes and LiDAR
    manual_boxes = lidar['Manual'][density][filename]['Boxes']
    deepforest_boxes = lidar['DeepForest'][density][filename]['Boxes']
    detectree_boxes = lidar['Detectree2'][density][filename]['Boxes']

    manual_coords, manual_labels = lidar['Manual'][density][filename]['LiDAR']
    deepforest_coords, deepforest_labels = lidar['DeepForest'][density][filename]['LiDAR']
    detectree_coords, detectree_labels = lidar['Detectree2'][density][filename]['LiDAR']

  # Segment using SAM-2 on bounding boxes and LiDAR. In paper, iterative=True (default)
    manual_masks, manual_scores = segment_box_lidar(sam_predictor, manual_boxes.xyxy, manual_coords, manual_labels)
    if len(deepforest_boxes) > 0:
        deepforest_masks, deepforest_scores = segment_box_lidar(sam_predictor, deepforest_boxes.xyxy, deepforest_coords, deepforest_labels)
    else:
        deepforest_masks, deepforest_scores = np.empty((0,400,400)), np.empty((0))
    if len(detectree_boxes) > 0:
        detectree_masks, detectree_scores = segment_box_lidar(sam_predictor, detectree_boxes.xyxy, detectree_coords, detectree_labels)
    else:
        detectree_masks, detectree_scores = np.empty((0,400,400)), np.empty((0))

    # Filter masks with confidence threshold of 0.2
    idx = manual_scores >= 0.2
    lidar['Manual'][density][filename]['Boxes'] = manual_boxes[idx]
    lidar['Manual'][density][filename]['LiDAR'] = (manual_coords[idx], manual_labels[idx])
    lidar['Manual'][density][filename]['Masks'] = manual_masks[idx]

    idx = deepforest_scores >= 0.2
    lidar['DeepForest'][density][filename]['Boxes'] = deepforest_boxes[idx]
    lidar['DeepForest'][density][filename]['LiDAR'] = (deepforest_coords[idx], deepforest_labels[idx])
    lidar['DeepForest'][density][filename]['Masks'] = deepforest_masks[idx]

    idx = detectree_scores >= 0.2
    lidar['Detectree2'][density][filename]['Boxes'] = detectree_boxes[idx]
    lidar['Detectree2'][density][filename]['LiDAR'] = (detectree_coords[idx], detectree_labels[idx])
    lidar['Detectree2'][density][filename]['Masks'] = detectree_masks[idx]

In [ ]:
#@title Bounding Box Predictions

for filename, density in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect hand-drawn bounding boxes
  hand_boxes_1 = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_preds = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  if df_preds is not None:
    df_preds = df_preds[df_preds['score'] >= 0.2]
    df_scores_1 = df_preds['score'].to_numpy()
    df_boxes_1 = df_preds[['xmin','ymin','xmax','ymax']].to_numpy()
  else:
    df_scores_1 = np.empty((0))
    df_boxes_1 = np.empty((0,4))

  # Generate automatic bounding boxes using Detectree2
  dt_preds = dt_model(bgr_img)
  if dt_preds is not None:
    dt_boxes_1 = dt_preds['instances'].pred_boxes.tensor.cpu().numpy()
    dt_scores_1 = dt_preds['instances'].scores.cpu().numpy()
    dt_classes_1 = dt_preds['instances'].pred_classes.cpu().numpy()
    dt_cnn_masks = dt_preds['instances'].pred_masks.cpu().numpy()
    # Filter with confidence threshold of 0.2
    dt_idx = dt_scores_1 >= 0.2
    dt_boxes_1 = dt_boxes_1[dt_idx]
    dt_scores_1 = dt_scores_1[dt_idx]
    dt_classes_1 = dt_classes_1[dt_idx]
    dt_cnn_masks = dt_cnn_masks[dt_idx]
  else:
    dt_boxes_1 = np.empty((0,4))
    dt_scores_1 = np.empty((0))
    dt_classes_1 = np.empty((0))
    dt_cnn_masks = np.empty(0,400,400)

  # Create detections objects for boxes
  hand_detections_1 = sv.Detections(xyxy=hand_boxes_1, confidence=np.ones(len(hand_boxes_1)),
                                    class_id=np.zeros(len(hand_boxes_1), dtype='int'))
  df_detections_1 = sv.Detections(xyxy=df_boxes_1, confidence=df_scores_1,
                                  class_id=np.zeros(len(df_boxes_1), dtype='int'))
  dt_detections_1 = sv.Detections(xyxy=dt_boxes_1, confidence=dt_scores_1,
                                  class_id=dt_classes_1)
  dt_cnn_detections_1 = sv.Detections(xyxy=dt_boxes_1, mask=dt_cnn_masks, confidence=dt_scores_1,
                                      class_id=dt_classes_1)

  # Apply custom non-max suppression to automatic boxes
  df_idx = custom_nms(df_detections_1, threshold=0.5)
  dt_idx = custom_nms(dt_detections_1, threshold=0.5)
  df_detections_1 = df_detections_1[df_idx]
  dt_detections_1 = dt_detections_1[dt_idx]
  dt_cnn_detections_1 = dt_cnn_detections_1[dt_idx]

  # Extract boxes and scores for convenience
  hand_boxes_1, hand_scores_1 = hand_detections_1.xyxy, hand_detections_1.confidence
  df_boxes_1, df_scores_1 = df_detections_1.xyxy, df_detections_1.confidence
  dt_boxes_1, dt_scores_1 = dt_detections_1.xyxy, dt_detections_1.confidence

  # Segment using SAM-2 on bounding boxes
  hand_masks_1, hand_confidence_1 = segment_boxes(sam_predictor, hand_boxes_1)
  if len(df_boxes_1) > 0:
    df_masks_1, df_confidence_1 = segment_boxes(sam_predictor, df_boxes_1)
  else:
    df_masks_1, df_confidence_2 = np.empty((0,400,400)), np.empty((0))
  if len(dt_boxes_1) > 0:
    dt_masks_1, dt_confidence_1 = segment_boxes(sam_predictor, dt_boxes_1)
  else:
    dt_masks_1, dt_confidence_1 = np.empty((0,400,400)), np.empty((0))

  # Assign masks and confidence values to detections
  hand_detections_1.mask, hand_detections_1.confidence = hand_masks_1, hand_confidence_1
  df_detections_1.mask, df_detections_1.confidence = df_masks_1, df_confidence_1
  dt_detections_1.mask, dt_detections_1.confidence = dt_masks_1, dt_confidence_1

  # Filter masks with confidence threshold of 0.2
  hand_idx = hand_confidence_1 >= 0.2
  df_idx = df_confidence_1 >= 0.2
  dt_idx = dt_confidence_1 >= 0.2
  hand_detections_1 = hand_detections_1[hand_idx]
  df_detections_1 = df_detections_1[df_idx]
  dt_detections_1 = dt_detections_1[dt_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=hand_boxes_1)
  df_coords, df_labels = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=df_boxes_1)
  dt_coords, dt_labels = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=dt_boxes_1)

  # Use labeled LiDAR points to filter boxes
  df_idx = lidar_filter(df_boxes_1, df_coords, df_labels, threshold=0.2)
  dt_idx = lidar_filter(dt_boxes_1, dt_coords, dt_labels, threshold=0.2)
  df_boxes_2, df_labels, df_scores_2 = df_boxes_1[df_idx], df_labels[df_idx], df_scores_1[df_idx]
  dt_boxes_2, dt_labels, dt_scores_2, dt_classes_2 = dt_boxes_1[dt_idx], dt_labels[dt_idx], dt_scores_1[dt_idx], dt_classes_1[dt_idx]

  # Create detections objects
  hand_detections_2 = sv.Detections(xyxy=hand_boxes_1, confidence=np.ones(len(hand_boxes_1)),
                                    class_id=np.zeros(len(hand_boxes_1), dtype='int'))
  df_detections_2 = sv.Detections(xyxy=df_boxes_2, confidence=df_scores_2,
                                  class_id=np.zeros(len(df_boxes_2), dtype='int'))
  dt_detections_2 = sv.Detections(xyxy=dt_boxes_2, confidence=dt_scores_2,
                                  class_id=dt_classes_2)
  dt_cnn_detections_2 = dt_cnn_detections_1[dt_idx]

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  hand_coords, hand_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes_2) > 0:
    df_coords, df_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes_2) > 0:
    dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2 on bounding boxes and LiDAR points
  hand_masks_2, hand_confidence_2 = segment_box_points(sam_predictor, hand_boxes_1, hand_coords, hand_labels, iterative=True)
  if len(df_boxes_2) > 0:
    df_masks_2, df_confidence_2 = segment_box_points(sam_predictor, df_boxes_2, df_coords, df_labels, iterative=True)
  else:
    df_masks_2, df_confidence_2 = np.empty((0,400,400)), np.empty((0))
  if len(dt_boxes_2) > 0:
    dt_masks_2, dt_confidence_2 = segment_box_points(sam_predictor, dt_boxes_2, dt_coords, dt_labels, iterative=True)
  else:
    dt_masks_2, dt_confidence_2 = np.empty((0,400,400)), np.empty((0))

  # Assign masks and confidence values to detections
  hand_detections_2.mask, hand_detections_2.confidence = hand_masks_2, hand_confidence_2
  df_detections_2.mask, df_detections_2.confidence = df_masks_2, df_confidence_2
  dt_detections_2.mask, dt_detections_2.confidence = dt_masks_2, dt_confidence_2

  # Filter masks with confidence threshold of 0.2
  hand_idx = hand_confidence_2 >= 0.2
  df_idx = df_confidence_2 >= 0.2
  dt_idx = dt_confidence_2 >= 0.2
  hand_detections_2 = hand_detections_2[hand_idx]
  df_detections_2 = df_detections_2[df_idx]
  dt_detections_2 = dt_detections_2[dt_idx]

  # Save Detections
  with sv.CSVSink(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', density, filename+'.csv')) as sink:
    sink.append(hand_detections_1, {'method':'Manual Boxes'})
    sink.append(hand_detections_2, {'method':'Manual + LiDAR'})
    sink.append(df_detections_1, {'method':'DeepForest Boxes'})
    sink.append(df_detections_2, {'method':'DeepForest + LiDAR'})
    sink.append(dt_detections_1, {'method':'Detectree2 Boxes'})
    sink.append(dt_detections_2, {'method':'Detectree2 + LiDAR'})
    sink.append(dt_cnn_detections_1, {'method':'Baseline'})
    sink.append(dt_cnn_detections_2, {'method':'Baseline Filtered'})

  # Save Masks
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', density, filename+'.npy'), 'wb') as f:
    if len(hand_detections_1) > 0:
      np.save(f, hand_detections_1.mask)
    if len(hand_detections_2) > 0:
      np.save(f, hand_detections_2.mask)
    if len(df_detections_1) > 0:
      np.save(f, df_detections_1.mask)
    if len(df_detections_2) > 0:
      np.save(f, df_detections_2.mask)
    if len(dt_detections_1) > 0:
      np.save(f, dt_detections_1.mask)
    if len(dt_detections_2) > 0:
      np.save(f, dt_detections_2.mask)
    if len(dt_cnn_detections_1) > 0:
      np.save(f, dt_cnn_detections_1.mask)
    if len(dt_cnn_detections_2) > 0:
      np.save(f, dt_cnn_detections_2.mask)

In [ ]:
#@title Save Ground Truths
'''
for filename, difficulty in test_imgs:
  img = ds.get_image(filename)
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes, true_masks = true_boxes[box_idx], true_masks[mask_idx]
  true_detections = sv.Detections(xyxy=true_boxes, mask=true_masks, confidence=np.ones(len(true_boxes)),
                                               class_id=np.zeros(len(true_boxes), dtype='int'))

  # Save Detections
  with sv.CSVSink(os.path.join(annotation_folder, difficulty, filename+'.csv')) as sink:
    sink.append(true_detections, {})

  # Save Masks
  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'wb') as f:
    if len(true_detections) > 0:
      np.save(f, true_detections.mask)
'''
print()

In [ ]:
#@title Per-Tree Averages
'''
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

difficulties = ['Easy','Medium', 'Hard']

columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)

preds = {difficulty:{method:dict() for method in methods} for difficulty in difficulties}
truths = {difficulty:dict() for difficulty in difficulties}

for filename, difficulty in test_imgs:
  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths[difficulty][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.csv'), index_col='method')
    for method in methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)

# Per Image Metrics
index = [(difficulty, method, filename) for method in methods for filename, difficulty in test_imgs]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Filename'])
metrics_images = pd.DataFrame(index=index, columns=columns)

for filename, difficulty in test_imgs:
  true_detections = truths[difficulty][filename]
  for method in methods:
    pred_detections = preds[difficulty][method][filename]
    metrics_images.loc[difficulty, method, filename] = per_tree_metrics([true_detections], [pred_detections])
metrics_images.to_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'Avg by Image.csv'))

# Per Difficulty Metrics
index = [(difficulty, method) for difficulty in difficulties for method in methods]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method'])
metrics_difficulty = pd.DataFrame(index=index, columns=columns)

for diff in difficulties:
  for method in methods:
    true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs if difficulty==diff]
    pred_detections = [preds[difficulty][method][filename] for filename, difficulty in test_imgs if difficulty==diff]
    metrics_difficulty.loc[diff,method] = per_tree_metrics(true_detections, pred_detections)
metrics_difficulty.to_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'Avg by Difficulty.csv'))

# Per Method Metrics
metrics_methods = pd.DataFrame(index=methods, columns=columns)

for method in methods:
  true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs]
  pred_detections = [preds[difficulty][method][filename] for filename, difficulty in test_imgs]
  metrics_methods.loc[method] = per_tree_metrics(true_detections, pred_detections)
metrics_methods.to_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'Avg by Method.csv'))
'''
metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'Avg by Method.csv'), index_col=[0], header=[0,1])
metrics_difficulty = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'Avg by Difficulty.csv'), index_col=[0,1], header=[0,1])
metrics_images = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'Avg by Image.csv'), index_col=[0,1,2], header=[0,1])

display(metrics_methods.style.format('{:.1%}'))
print()
display(metrics_difficulty.style.format('{:.1%}'))
print()
display(metrics_images.style.format('{:.1%}'))

In [ ]:
#@title Per Tree Standard Deviations (Pixel-based metrics)

methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

difficulties = ['Easy','Medium', 'Hard']

columns = [('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)

preds = {difficulty:{method:dict() for method in methods} for difficulty in difficulties}
truths = {difficulty:dict() for difficulty in difficulties}

for filename, difficulty in test_imgs:
  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths[difficulty][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.csv'), index_col='method')
    for method in methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)

# Per Image Metrics
index = [(difficulty, method, filename) for method in methods for filename, difficulty in test_imgs]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Filename'])
std_images = pd.DataFrame(index=index, columns=columns)

for filename, difficulty in test_imgs:
  true_detections = truths[difficulty][filename]
  for method in methods:
    pred_detections = preds[difficulty][method][filename]
    std_images.loc[difficulty, method, filename] = per_tree_std([true_detections], [pred_detections])
std_images.to_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'STD by Image.csv'))

# Per Difficulty Metrics
index = [(difficulty, method) for difficulty in difficulties for method in methods]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method'])
std_difficulty = pd.DataFrame(index=index, columns=columns)

for diff in difficulties:
  for method in methods:
    true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs if difficulty==diff]
    pred_detections = [preds[difficulty][method][filename] for filename, difficulty in test_imgs if difficulty==diff]
    std_difficulty.loc[diff,method] = per_tree_std(true_detections, pred_detections)
std_difficulty.to_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'STD by Difficulty.csv'))

# Per Method Metrics
std_methods = pd.DataFrame(index=methods, columns=columns)

for method in methods:
  true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs]
  pred_detections = [preds[difficulty][method][filename] for filename, difficulty in test_imgs]
  std_methods.loc[method] = per_tree_std(true_detections, pred_detections)
std_methods.to_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'STD by Method.csv'))

std_methods = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'STD by Method.csv'), index_col=[0], header=[0,1])
std_difficulty = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'STD by Difficulty.csv'), index_col=[0,1], header=[0,1])
std_images = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', 'Per Tree Metrics', 'STD by Image.csv'), index_col=[0,1,2], header=[0,1])

display(std_methods.style.format('{:.1%}'))
print()
display(std_difficulty.style.format('{:.1%}'))
print()
display(std_images.style.format('{:.1%}'))

In [ ]:
#@title Example Images

methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

fig, axs = plt.subplots(nrows=4, ncols=4, figsize=(7.5,7.5))

for i, (filename, difficulty, method_1, method_2) in enumerate([('2018_SJER_3_254000_4108000_image_700', 'Easy', 'Manual Boxes', 'Manual + LiDAR'),
                                                                ('WREF_072_2019', 'Medium', 'DeepForest Boxes', 'DeepForest + LiDAR'),
                                                                ('DELA_047_2019', 'Hard', 'Detectree2 Boxes', 'Detectree2 + LiDAR'),
                                                                ('SJER_025_2018', 'Medium', 'Baseline', 'Baseline Filtered')]):
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()

  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.csv'), index_col='method')
    for method in methods:
      mask = np.load(f)
      if method == method_1:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
        preds_1 = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
      elif method == method_2:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
        preds_2 = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
      else:
        continue

  box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.BLUE)
  poly_annotator = sv.PolygonAnnotator(thickness=2, color=sv.Color.BLUE)
  mask_annotator = sv.MaskAnnotator(opacity=1, color=sv.Color.WHITE)
  white_img = np.ones((400,400,3), dtype=np.uint8)

  # Visualize
  prompt_img = box_annotator.annotate(bgr_img.copy(), detections=preds_1)[:,:,::-1]
  pred_img_1 = mask_annotator.annotate(white_img.copy(), detections=preds_1)
  pred_img_1 = poly_annotator.annotate(pred_img_1, detections=preds_1)[:,:,::-1]
  pred_img_2 = mask_annotator.annotate(white_img.copy(), detections=preds_2)
  pred_img_2 = poly_annotator.annotate(pred_img_2, detections=preds_2)[:,:,::-1]
  true_img = mask_annotator.annotate(white_img.copy(), detections=truths)
  true_img = poly_annotator.annotate(true_img, detections=truths)[:,:,::-1]

  axs[i,0].imshow(prompt_img)
  axs[i,1].imshow(pred_img_1)
  axs[i,2].imshow(pred_img_2)
  axs[i,3].imshow(true_img)

for i in range(4):
  for j in range(4):
    axs[i,j].set_frame_on(False)
    axs[i,j].get_xaxis().set_ticks([])
    axs[i,j].get_yaxis().set_ticks([])

axs[0,0].set_title('Prompt')
axs[0,1].set_title('Predictions')
axs[0,2].set_title('Predictions (+LiDAR)')
axs[0,3].set_title('Ground Truth')

plt.tight_layout()
plt.savefig(f'qualitative_segmentation.tif', dpi=500)
plt.show()

# Other Experiments

In [ ]:
#@title LiDAR Preprocessing

lidar_unfiltered = '/content/drive/MyDrive/Sam+LiDAR/LiDAR/Labeled/Without LiDAR filter'
lidar_original = '/content/weecology-NeonTreeEvaluation-d0b90bc/evaluation/LiDAR'

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_preds = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  if df_preds is not None:
    df_preds = df_preds[df_preds['score'] >= 0.2]
    df_scores = df_preds['score'].to_numpy()
    df_boxes = df_preds[['xmin','ymin','xmax','ymax']].to_numpy()
  else:
    df_scores = np.empty((0))
    df_boxes = np.empty((0,4))

  # Generate automatic bounding boxes using Detectree2
  dt_preds = dt_model(bgr_img)
  if dt_preds is not None:
    dt_boxes = dt_preds['instances'].pred_boxes.tensor.cpu().numpy()
    dt_scores = dt_preds['instances'].scores.cpu().numpy()
    dt_classes = dt_preds['instances'].pred_classes.cpu().numpy()
    dt_cnn_masks = dt_preds['instances'].pred_masks.cpu().numpy()
    # Filter with confidence threshold of 0.2
    dt_idx = dt_scores >= 0.2
    dt_boxes = dt_boxes[dt_idx]
    dt_scores = dt_scores[dt_idx]
    dt_classes = dt_classes[dt_idx]
    dt_cnn_masks = dt_cnn_masks[dt_idx]
  else:
    dt_boxes = np.empty((0,4))
    dt_scores = np.empty((0))
    dt_classes = np.empty((0))
    dt_cnn_masks = np.empty(0,400,400)

  # Create detections objects for boxes
  hand_detections = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                    class_id=np.zeros(len(hand_boxes), dtype='int'))
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores,
                                  class_id=np.zeros(len(df_boxes), dtype='int'))
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores,
                                  class_id=dt_classes)
  dt_cnn_detections = sv.Detections(xyxy=dt_boxes, mask=dt_cnn_masks, confidence=dt_scores,
                                      class_id=dt_classes)

  # Apply custom non-max suppression to automatic boxes
  df_idx = custom_nms(df_detections, threshold=0.5)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  df_detections = df_detections[df_idx]
  dt_detections = dt_detections[dt_idx]
  dt_cnn_detections = dt_cnn_detections[dt_idx]

  # Extract boxes and scores for convenience
  hand_boxes, hand_scores = hand_detections.xyxy, hand_detections.confidence
  df_boxes, df_scores = df_detections.xyxy, df_detections.confidence
  dt_boxes, dt_scores = dt_detections.xyxy, dt_detections.confidence

  # Use boxes to label unfiltered LiDAR points
  hand_coords_1, hand_labels_1 = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=hand_boxes)
  df_coords_1, df_labels_1 = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=df_boxes)
  dt_coords_1, dt_labels_1 = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=dt_boxes)

  # Use unfiltered LiDAR points to filter boxes
  df_idx = lidar_filter(df_boxes, df_coords_1, df_labels_1, threshold=0.2)
  dt_idx = lidar_filter(dt_boxes, dt_coords_1, dt_labels_1, threshold=0.2)
  df_boxes_1, df_labels_1, df_scores_1 = df_boxes[df_idx], df_labels_1[df_idx], df_scores[df_idx]
  dt_boxes_1, dt_labels_1, dt_scores_1, dt_classes_1 = dt_boxes[dt_idx], dt_labels_1[dt_idx], dt_scores[dt_idx], dt_classes[dt_idx]

  # Create detections objects
  hand_detections_1 = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                    class_id=np.zeros(len(hand_boxes), dtype='int'))
  df_detections_1 = sv.Detections(xyxy=df_boxes_1, confidence=df_scores_1,
                                  class_id=np.zeros(len(df_boxes_1), dtype='int'))
  dt_detections_1 = sv.Detections(xyxy=dt_boxes_1, confidence=dt_scores_1,
                                  class_id=dt_classes_1)
  dt_cnn_detections_1 = dt_cnn_detections[dt_idx]

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  hand_coords_1, hand_labels_1 = sample_points(hand_coords_1, hand_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes_1) > 0:
    df_coords_1, df_labels_1 = sample_points(df_coords_1, df_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes_1) > 0:
    dt_coords_1, dt_labels_1 = sample_points(dt_coords_1, dt_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2 on bounding boxes and LiDAR points
  hand_masks_1, hand_confidence_1 = segment_box_points(sam_predictor, hand_boxes, hand_coords_1, hand_labels_1, iterative=True)
  if len(df_boxes_1) > 0:
    df_masks_1, df_confidence_1 = segment_box_points(sam_predictor, df_boxes_1, df_coords_1, df_labels_1, iterative=True)
  else:
    df_masks_1, df_confidence_1 = np.empty((0,400,400)), np.empty((0))
  if len(dt_boxes_1) > 0:
    dt_masks_1, dt_confidence_1 = segment_box_points(sam_predictor, dt_boxes_1, dt_coords_1, dt_labels_1, iterative=True)
  else:
    dt_masks_1, dt_confidence_1 = np.empty((0,400,400)), np.empty((0))

  # Assign masks and confidence values to detections
  hand_detections_1.mask, hand_detections_1.confidence = hand_masks_1, hand_confidence_1
  df_detections_1.mask, df_detections_1.confidence = df_masks_1, df_confidence_1
  dt_detections_1.mask, dt_detections_1.confidence = dt_masks_1, dt_confidence_1

  # Save Detections
  with sv.CSVSink(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.csv')) as sink:
    sink.append(hand_detections_1, {'method':'Manual + LiDAR'})
    sink.append(df_detections_1, {'method':'DeepForest + LiDAR'})
    sink.append(dt_detections_1, {'method':'Detectree2 + LiDAR'})
    sink.append(dt_cnn_detections_1, {'method':'Baseline Filtered'})

  # Save Masks
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.npy'), 'wb') as f:
    if len(hand_detections_1) > 0:
      np.save(f, hand_detections_1.mask)
    if len(df_detections_1) > 0:
      np.save(f, df_detections_1.mask)
    if len(dt_detections_1) > 0:
      np.save(f, dt_detections_1.mask)
    if len(dt_cnn_detections_1) > 0:
      np.save(f, dt_cnn_detections_1.mask)

  # Use boxes to label original LiDAR points
  hand_coords_2, hand_labels_2 = rasterize_lidar_old(lidar_original, rgb_folder, filename, boxes=hand_boxes)
  df_coords_2, df_labels_2 = rasterize_lidar_old(lidar_original, rgb_folder, filename, boxes=df_boxes)
  dt_coords_2, dt_labels_2 = rasterize_lidar_old(lidar_original, rgb_folder, filename, boxes=dt_boxes)

  # Use original LiDAR points to filter boxes
  df_idx = lidar_filter(df_boxes, df_coords_2, df_labels_2, threshold=0.2)
  dt_idx = lidar_filter(dt_boxes, dt_coords_2, dt_labels_2, threshold=0.2)
  df_boxes_2, df_labels_2, df_scores_2 = df_boxes[df_idx], df_labels_2[df_idx], df_scores[df_idx]
  dt_boxes_2, dt_labels_2, dt_scores_2, dt_classes_2 = dt_boxes[dt_idx], dt_labels_2[dt_idx], dt_scores[dt_idx], dt_classes[dt_idx]

  # Create detections objects
  hand_detections_2 = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                    class_id=np.zeros(len(hand_boxes), dtype='int'))
  df_detections_2 = sv.Detections(xyxy=df_boxes_2, confidence=df_scores_2,
                                  class_id=np.zeros(len(df_boxes_2), dtype='int'))
  dt_detections_2 = sv.Detections(xyxy=dt_boxes_2, confidence=dt_scores_2,
                                  class_id=dt_classes_2)
  dt_cnn_detections_2 = dt_cnn_detections[dt_idx]

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  hand_coords_2, hand_labels_2 = sample_points(hand_coords_2, hand_labels_2, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes_2) > 0:
    df_coords_2, df_labels_2 = sample_points(df_coords_2, df_labels_2, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes_2) > 0:
    dt_coords_2, dt_labels_2 = sample_points(dt_coords_2, dt_labels_2, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2 on bounding boxes and LiDAR points
  hand_masks_2, hand_confidence_2 = segment_box_points(sam_predictor, hand_boxes, hand_coords_2, hand_labels_2, iterative=True)
  if len(df_boxes_2) > 0:
    df_masks_2, df_confidence_2 = segment_box_points(sam_predictor, df_boxes_2, df_coords_2, df_labels_2, iterative=True)
  else:
    df_masks_2, df_confidence_2 = np.empty((0,400,400)), np.empty((0))
  if len(dt_boxes_2) > 0:
    dt_masks_2, dt_confidence_2 = segment_box_points(sam_predictor, dt_boxes_2, dt_coords_2, dt_labels_2, iterative=True)
  else:
    dt_masks_2, dt_confidence_2 = np.empty((0,400,400)), np.empty((0))

  # Assign masks and confidence values to detections
  hand_detections_2.mask, hand_detections_2.confidence = hand_masks_2, hand_confidence_2
  df_detections_2.mask, df_detections_2.confidence = df_masks_2, df_confidence_2
  dt_detections_2.mask, dt_detections_2.confidence = dt_masks_2, dt_confidence_2

  # Save Detections
  with sv.CSVSink(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Original', difficulty, filename+'.csv')) as sink:
    sink.append(hand_detections_2, {'method':'Manual + LiDAR'})
    sink.append(df_detections_2, {'method':'DeepForest + LiDAR'})
    sink.append(dt_detections_2, {'method':'Detectree2 + LiDAR'})
    sink.append(dt_cnn_detections_2, {'method':'Baseline Filtered'})

  # Save Masks
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Original', difficulty, filename+'.npy'), 'wb') as f:
    if len(hand_detections_2) > 0:
      np.save(f, hand_detections_2.mask)
    if len(df_detections_2) > 0:
      np.save(f, df_detections_2.mask)
    if len(dt_detections_2) > 0:
      np.save(f, dt_detections_2.mask)
    if len(dt_cnn_detections_2) > 0:
      np.save(f, dt_cnn_detections_2.mask)

In [ ]:
#@title Calculate Per-Tree Metrics
'''
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

lidar_methods = ['Manual + LiDAR', 'DeepForest + LiDAR', 'Detectree2 + LiDAR', 'Baseline Filtered']

difficulties = ['Easy','Medium', 'Hard']

lidar_preprocessing = ['Threshold', 'Classified', 'Unfiltered']

columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)

original_preds = {difficulty:{method:dict() for method in lidar_methods} for difficulty in difficulties}
classified_preds = {difficulty:{method:dict() for method in methods} for difficulty in difficulties}
unfiltered_preds = {difficulty:{method:dict() for method in lidar_methods} for difficulty in difficulties}
truths = {difficulty:dict() for difficulty in difficulties}

for filename, difficulty in test_imgs:
  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths[difficulty][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Height Threshold', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Height Threshold', difficulty, filename+'.csv'), index_col='method')
    for method in lidar_methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      original_preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR', difficulty, filename+'.csv'), index_col='method')
    for method in methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      classified_preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.csv'), index_col='method')
    for method in lidar_methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      unfiltered_preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)

# Per Image Metrics
index = [(difficulty, method, filename, process) for method in lidar_methods for filename, difficulty in test_imgs for process in lidar_preprocessing]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Filename', 'LiDAR'])
metrics_images = pd.DataFrame(index=index, columns=columns)

for filename, difficulty in test_imgs:
  true_detections = truths[difficulty][filename]
  for method in lidar_methods:
    original_detections = original_preds[difficulty][method][filename]
    classified_detections = classified_preds[difficulty][method][filename]
    unfiltered_detections = unfiltered_preds[difficulty][method][filename]
    metrics_images.loc[difficulty, method, filename, 'Threshold'] = per_tree_metrics([true_detections], [original_detections])
    metrics_images.loc[difficulty, method, filename, 'Classified'] = per_tree_metrics([true_detections], [classified_detections])
    metrics_images.loc[difficulty, method, filename, 'Unfiltered'] = per_tree_metrics([true_detections], [unfiltered_detections])
metrics_images.to_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Tree Metrics', 'Avg by Image.csv'))

# Per Difficulty Metrics
index = [(difficulty, method, process) for difficulty in difficulties for method in lidar_methods for process in lidar_preprocessing]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'LiDAR'])
metrics_difficulties = pd.DataFrame(index=index, columns=columns)

for diff in difficulties:
  for method in lidar_methods:
    true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs if difficulty==diff]
    original_detections = [original_preds[difficulty][method][filename] for filename, difficulty in test_imgs if difficulty==diff]
    classified_detections = [classified_preds[difficulty][method][filename] for filename, difficulty in test_imgs if difficulty==diff]
    unfiltered_detections = [unfiltered_preds[difficulty][method][filename] for filename, difficulty in test_imgs if difficulty==diff]
    metrics_difficulties.loc[diff, method, 'Threshold'] = per_tree_metrics(true_detections, original_detections)
    metrics_difficulties.loc[diff, method, 'Classified'] = per_tree_metrics(true_detections, classified_detections)
    metrics_difficulties.loc[diff, method, 'Unfiltered'] = per_tree_metrics(true_detections, unfiltered_detections)
metrics_difficulties.to_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Tree Metrics', 'Avg by Difficulty.csv'))

# Per Method Metrics
index = [(method, process) for method in lidar_methods for process in lidar_preprocessing]
index = pd.MultiIndex.from_tuples(index, names=['Method', 'LiDAR'])
metrics_methods = pd.DataFrame(index=index, columns=columns)

for method in lidar_methods:
  true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs]
  original_detections = [original_preds[difficulty][method][filename] for filename, difficulty in test_imgs]
  classified_detections = [classified_preds[difficulty][method][filename] for filename, difficulty in test_imgs]
  unfiltered_detections = [unfiltered_preds[difficulty][method][filename] for filename, difficulty in test_imgs]
  metrics_methods.loc[method, 'Threshold'] = per_tree_metrics(true_detections, original_detections)
  metrics_methods.loc[method, 'Classified'] = per_tree_metrics(true_detections, classified_detections)
  metrics_methods.loc[method, 'Unfiltered'] = per_tree_metrics(true_detections, unfiltered_detections)
metrics_methods.to_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Tree Metrics', 'Avg by Method.csv'))
'''
metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Tree Metrics', 'Avg by Method.csv'), index_col=[0,1], header=[0,1])
metrics_difficulties = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Tree Metrics', 'Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])
metrics_images = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Tree Metrics', 'Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])

display(metrics_methods.style.format('{:.1%}'))
print()
display(metrics_difficulties.style.format('{:.1%}'))
print()
display(metrics_images.style.format('{:.1%}'))

In [ ]:
#@title Calculate Per-Image Metrics
'''
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

lidar_methods = ['Manual + LiDAR', 'DeepForest + LiDAR', 'Detectree2 + LiDAR', 'Baseline Filtered']

difficulties = ['Easy','Medium', 'Hard']

lidar_preprocessing = ['Threshold', 'Classified', 'Unfiltered']

columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)

original_preds = {difficulty:{method:dict() for method in lidar_methods} for difficulty in difficulties}
classified_preds = {difficulty:{method:dict() for method in methods} for difficulty in difficulties}
unfiltered_preds = {difficulty:{method:dict() for method in lidar_methods} for difficulty in difficulties}
truths = {difficulty:dict() for difficulty in difficulties}

for filename, difficulty in test_imgs:
  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths[difficulty][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Height Threshold', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Height Threshold', difficulty, filename+'.csv'), index_col='method')
    for method in lidar_methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      original_preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR', difficulty, filename+'.csv'), index_col='method')
    for method in methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      classified_preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.csv'), index_col='method')
    for method in lidar_methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      unfiltered_preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)

# Per Image Metrics
index = [(difficulty, method, filename, process) for method in lidar_methods for filename, difficulty in test_imgs for process in lidar_preprocessing]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Filename', 'LiDAR'])
metrics_images = pd.DataFrame(index=index, columns=columns)

for filename, difficulty in test_imgs:
  true_detections = truths[difficulty][filename]
  for method in lidar_methods:
    original_detections = original_preds[difficulty][method][filename]
    classified_detections = classified_preds[difficulty][method][filename]
    unfiltered_detections = unfiltered_preds[difficulty][method][filename]

    true_boxes, true_masks = true_detections.xyxy, true_detections.mask
    original_boxes, original_masks = original_detections.xyxy, original_detections.mask
    classified_boxes, classified_masks = classified_detections.xyxy, classified_detections.mask
    unfiltered_boxes, unfiltered_masks = unfiltered_detections.xyxy, unfiltered_detections.mask

    original_detect_metrics = detection_metrics(true_masks, original_masks)
    classified_detect_metrics = detection_metrics(true_masks, classified_masks)
    unfiltered_detect_metrics = detection_metrics(true_masks, unfiltered_masks)

    true_idx, original_idx = hungarian_matching(true_boxes, original_boxes, threshold=0.5)
    original_iou = compute_iou(true_masks[true_idx], original_masks[original_idx], reduction='mean')
    original_segment_metrics = segmentation_metrics(true_masks[true_idx], original_masks[original_idx], reduction='mean')

    true_idx, classified_idx = hungarian_matching(true_boxes, classified_boxes, threshold=0.5)
    classified_iou = compute_iou(true_masks[true_idx], classified_masks[classified_idx], reduction='mean')
    classified_segment_metrics = segmentation_metrics(true_masks[true_idx], classified_masks[classified_idx], reduction='mean')

    true_idx, unfiltered_idx = hungarian_matching(true_boxes, unfiltered_boxes, threshold=0.5)
    unfiltered_iou = compute_iou(true_masks[true_idx], unfiltered_masks[unfiltered_idx], reduction='mean')
    unfiltered_segment_metrics = segmentation_metrics(true_masks[true_idx], unfiltered_masks[unfiltered_idx], reduction='mean')

    metrics_images.loc[difficulty, method, filename, 'Threshold'] = original_detect_metrics + original_segment_metrics + (original_iou,)
    metrics_images.loc[difficulty, method, filename, 'Classified'] = classified_detect_metrics + classified_segment_metrics + (classified_iou,)
    metrics_images.loc[difficulty, method, filename, 'Unfiltered'] = unfiltered_detect_metrics + unfiltered_segment_metrics + (unfiltered_iou,)
metrics_images.to_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Image Metrics', 'Avg by Image.csv'))

# Per Difficulty Metrics
index = [(difficulty, method, process) for difficulty in difficulties for method in lidar_methods for process in lidar_preprocessing]
metrics_difficulties = metrics_images.groupby(by=['Difficulty','Method','LiDAR']).mean().reindex(index)
metrics_difficulties.to_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Image Metrics', 'Avg by Difficulty.csv'))

# Per Method Metrics
index = [(method, process) for method in lidar_methods for process in lidar_preprocessing]
metrics_methods = metrics_images.groupby(by=['Method','LiDAR']).mean().reindex(index)
metrics_methods.to_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Image Metrics', 'Avg by Method.csv'))
'''
metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Image Metrics', 'Avg by Method.csv'), index_col=[0,1], header=[0,1])
metrics_difficulties = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Image Metrics', 'Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])
metrics_images = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Per Image Metrics', 'Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])

display(metrics_methods.style.format('{:.1%}'))
print()
display(metrics_difficulties.style.format('{:.1%}'))
print()
display(metrics_images.style.format('{:.1%}'))

In [ ]:
#@title LiDAR Preprocessing Image

lidar_unfiltered = '/content/drive/MyDrive/Sam+LiDAR/LiDAR/Labeled/Without LiDAR filter'
lidar_original = '/content/weecology-NeonTreeEvaluation-d0b90bc/evaluation/LiDAR'

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12, 4))

for filename, difficulty in [('2018_TEAK_3_318000_4107000_image_718', 'Easy')]:
  for i, folder in enumerate([lidar_original, lidar_folder, lidar_unfiltered]):
    # Get image
    img = ds.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()
    sam_predictor.set_image(rgb_img)

    # Get ground truths
    true_boxes = img['annotation']
    true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))
    box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
    true_boxes, true_masks = true_boxes[box_idx], true_masks[mask_idx]
    true_detections = sv.Detections(xyxy=true_boxes, mask=true_masks, confidence=np.ones(len(true_boxes)),
                                                class_id=np.zeros(len(true_boxes), dtype='int'))

    # Collect hand-drawn bounding boxes
    hand_boxes = img['annotation']

    # Create detections objects for boxes
    hand_detections = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                      class_id=np.zeros(len(hand_boxes), dtype='int'))

    # Extract boxes and scores for convenience
    hand_boxes, hand_scores = hand_detections.xyxy, hand_detections.confidence

    # Use boxes to label unfiltered LiDAR points
    hand_coords_1, hand_labels_1 = rasterize_lidar(folder, rgb_folder, filename, boxes=hand_boxes)

    # Create detections objects
    hand_detections_1 = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                      class_id=np.zeros(len(hand_boxes), dtype='int'))

    # Sample LiDAR points for SAM-2
    pos = 10
    neg = 10
    hand_coords_1, hand_labels_1 = sample_points(hand_coords_1, hand_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

    # Segment using SAM-2 on bounding boxes and LiDAR points
    hand_masks_1, hand_confidence_1 = segment_box_points(sam_predictor, hand_boxes, hand_coords_1, hand_labels_1, iterative=True)

    # Assign masks and confidence values to detections
    hand_detections_1.mask, hand_detections_1.confidence = hand_masks_1, hand_confidence_1

    # Plot Images
    poly_annotator = sv.PolygonAnnotator(thickness=2, color=sv.Color.RED)
    mask_annotator = sv.MaskAnnotator()
    true_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections)

    hand_img = poly_annotator.annotate(true_img.copy(), detections=true_detections)
    hand_img = mask_annotator.annotate(hand_img, detections=hand_detections_1)[:,:,::-1]

    axs[i].imshow(hand_img)

axs[0].set_title('Height Threshold', fontsize=14, y=0.98)
axs[1].set_title('Classification', fontsize=14, y=0.98)
axs[2].set_title('Classification (No Filter)', fontsize=14, y=0.98)

for i in range(3):
  axs[i].set_frame_on(False)
  axs[i].get_xaxis().set_ticks([])
  axs[i].get_yaxis().set_ticks([])
plt.tight_layout()
plt.savefig(f'{filename} LiDAR Preprocessing.jpg')
plt.show()

In [ ]:
#@title Random Seed Sampling

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_preds = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  if df_preds is not None:
    df_preds = df_preds[df_preds['score'] >= 0.2]
    df_scores = df_preds['score'].to_numpy()
    df_boxes = df_preds[['xmin','ymin','xmax','ymax']].to_numpy()
  else:
    df_scores = np.empty((0))
    df_boxes = np.empty((0,4))

  # Generate automatic bounding boxes using Detectree2
  dt_preds = dt_model(bgr_img)
  if dt_preds is not None:
    dt_boxes = dt_preds['instances'].pred_boxes.tensor.cpu().numpy()
    dt_scores = dt_preds['instances'].scores.cpu().numpy()
    dt_classes = dt_preds['instances'].pred_classes.cpu().numpy()
    dt_cnn_masks = dt_preds['instances'].pred_masks.cpu().numpy()
    # Filter with confidence threshold of 0.2
    dt_idx = dt_scores >= 0.2
    dt_boxes = dt_boxes[dt_idx]
    dt_scores = dt_scores[dt_idx]
    dt_classes = dt_classes[dt_idx]
    dt_cnn_masks = dt_cnn_masks[dt_idx]
  else:
    dt_boxes = np.empty((0,4))
    dt_scores = np.empty((0))
    dt_classes = np.empty((0))
    dt_cnn_masks = np.empty(0,400,400)

  # Create detections objects for boxes
  hand_detections = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                    class_id=np.zeros(len(hand_boxes), dtype='int'))
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores,
                                  class_id=np.zeros(len(df_boxes), dtype='int'))
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores,
                                  class_id=dt_classes)
  dt_cnn_detections = sv.Detections(xyxy=dt_boxes, mask=dt_cnn_masks, confidence=dt_scores,
                                      class_id=dt_classes)

  # Apply custom non-max suppression to automatic boxes
  df_idx = custom_nms(df_detections, threshold=0.5)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  df_detections = df_detections[df_idx]
  dt_detections = dt_detections[dt_idx]
  dt_cnn_detections = dt_cnn_detections[dt_idx]

  # Extract boxes and scores for convenience
  hand_boxes, hand_scores = hand_detections.xyxy, hand_detections.confidence
  df_boxes, df_scores = df_detections.xyxy, df_detections.confidence
  dt_boxes, dt_scores = dt_detections.xyxy, dt_detections.confidence

  for i in range(4,6):
    # Use boxes to label unfiltered LiDAR points
    hand_coords_1, hand_labels_1 = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=hand_boxes)
    df_coords_1, df_labels_1 = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=df_boxes)
    dt_coords_1, dt_labels_1 = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=dt_boxes)

    # Use unfiltered LiDAR points to filter boxes
    df_idx = lidar_filter(df_boxes, df_coords_1, df_labels_1, threshold=0.2)
    dt_idx = lidar_filter(dt_boxes, dt_coords_1, dt_labels_1, threshold=0.2)
    df_boxes_1, df_labels_1, df_scores_1 = df_boxes[df_idx], df_labels_1[df_idx], df_scores[df_idx]
    dt_boxes_1, dt_labels_1, dt_scores_1, dt_classes_1 = dt_boxes[dt_idx], dt_labels_1[dt_idx], dt_scores[dt_idx], dt_classes[dt_idx]

    # Create detections objects
    hand_detections_1 = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                      class_id=np.zeros(len(hand_boxes), dtype='int'))
    df_detections_1 = sv.Detections(xyxy=df_boxes_1, confidence=df_scores_1,
                                    class_id=np.zeros(len(df_boxes_1), dtype='int'))
    dt_detections_1 = sv.Detections(xyxy=dt_boxes_1, confidence=dt_scores_1,
                                    class_id=dt_classes_1)
    dt_cnn_detections_1 = dt_cnn_detections[dt_idx]

    # Sample LiDAR points for SAM-2
    pos = 10
    neg = 10
    hand_coords_1, hand_labels_1 = sample_points(hand_coords_1, hand_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)
    if len(df_boxes_1) > 0:
      df_coords_1, df_labels_1 = sample_points(df_coords_1, df_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)
    if len(dt_boxes_1) > 0:
      dt_coords_1, dt_labels_1 = sample_points(dt_coords_1, dt_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)

    # Segment using SAM-2 on bounding boxes and LiDAR points
    hand_masks_1, hand_confidence_1 = segment_box_points(sam_predictor, hand_boxes, hand_coords_1, hand_labels_1, iterative=True)
    if len(df_boxes_1) > 0:
      df_masks_1, df_confidence_1 = segment_box_points(sam_predictor, df_boxes_1, df_coords_1, df_labels_1, iterative=True)
    else:
      df_masks_1, df_confidence_1 = np.empty((0,400,400)), np.empty((0))
    if len(dt_boxes_1) > 0:
      dt_masks_1, dt_confidence_1 = segment_box_points(sam_predictor, dt_boxes_1, dt_coords_1, dt_labels_1, iterative=True)
    else:
      dt_masks_1, dt_confidence_1 = np.empty((0,400,400)), np.empty((0))

    # Assign masks and confidence values to detections
    hand_detections_1.mask, hand_detections_1.confidence = hand_masks_1, hand_confidence_1
    df_detections_1.mask, df_detections_1.confidence = df_masks_1, df_confidence_1
    dt_detections_1.mask, dt_detections_1.confidence = dt_masks_1, dt_confidence_1

    # Save Detections
    with sv.CSVSink(os.path.join(experiments_folder, 'Random Seed Sampling', 'No Filter', f'Seed {i}', difficulty, filename+'.csv')) as sink:
      sink.append(hand_detections_1, {'method':'Manual + LiDAR'})
      sink.append(df_detections_1, {'method':'DeepForest + LiDAR'})
      sink.append(dt_detections_1, {'method':'Detectree2 + LiDAR'})
      sink.append(dt_cnn_detections_1, {'method':'Baseline Filtered'})

    # Save Masks
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'No Filter', f'Seed {i}', difficulty, filename+'.npy'), 'wb') as f:
      if len(hand_detections_1) > 0:
        np.save(f, hand_detections_1.mask)
      if len(df_detections_1) > 0:
        np.save(f, df_detections_1.mask)
      if len(dt_detections_1) > 0:
        np.save(f, dt_detections_1.mask)
      if len(dt_cnn_detections_1) > 0:
        np.save(f, dt_cnn_detections_1.mask)

  for i in range(4,6):
    # Use boxes to label original LiDAR points
    hand_coords_2, hand_labels_2 = rasterize_lidar_old(lidar_original, rgb_folder, filename, boxes=hand_boxes)
    df_coords_2, df_labels_2 = rasterize_lidar_old(lidar_original, rgb_folder, filename, boxes=df_boxes)
    dt_coords_2, dt_labels_2 = rasterize_lidar_old(lidar_original, rgb_folder, filename, boxes=dt_boxes)

    # Use original LiDAR points to filter boxes
    df_idx = lidar_filter(df_boxes, df_coords_2, df_labels_2, threshold=0.2)
    dt_idx = lidar_filter(dt_boxes, dt_coords_2, dt_labels_2, threshold=0.2)
    df_boxes_2, df_labels_2, df_scores_2 = df_boxes[df_idx], df_labels_2[df_idx], df_scores[df_idx]
    dt_boxes_2, dt_labels_2, dt_scores_2, dt_classes_2 = dt_boxes[dt_idx], dt_labels_2[dt_idx], dt_scores[dt_idx], dt_classes[dt_idx]

    # Create detections objects
    hand_detections_2 = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                      class_id=np.zeros(len(hand_boxes), dtype='int'))
    df_detections_2 = sv.Detections(xyxy=df_boxes_2, confidence=df_scores_2,
                                    class_id=np.zeros(len(df_boxes_2), dtype='int'))
    dt_detections_2 = sv.Detections(xyxy=dt_boxes_2, confidence=dt_scores_2,
                                    class_id=dt_classes_2)
    dt_cnn_detections_2 = dt_cnn_detections[dt_idx]

    # Sample LiDAR points for SAM-2
    pos = 10
    neg = 10
    hand_coords_2, hand_labels_2 = sample_points(hand_coords_2, hand_labels_2, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)
    if len(df_boxes_2) > 0:
      df_coords_2, df_labels_2 = sample_points(df_coords_2, df_labels_2, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)
    if len(dt_boxes_2) > 0:
      dt_coords_2, dt_labels_2 = sample_points(dt_coords_2, dt_labels_2, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)

    # Segment using SAM-2 on bounding boxes and LiDAR points
    hand_masks_2, hand_confidence_2 = segment_box_points(sam_predictor, hand_boxes, hand_coords_2, hand_labels_2, iterative=True)
    if len(df_boxes_2) > 0:
      df_masks_2, df_confidence_2 = segment_box_points(sam_predictor, df_boxes_2, df_coords_2, df_labels_2, iterative=True)
    else:
      df_masks_2, df_confidence_2 = np.empty((0,400,400)), np.empty((0))
    if len(dt_boxes_2) > 0:
      dt_masks_2, dt_confidence_2 = segment_box_points(sam_predictor, dt_boxes_2, dt_coords_2, dt_labels_2, iterative=True)
    else:
      dt_masks_2, dt_confidence_2 = np.empty((0,400,400)), np.empty((0))

    # Assign masks and confidence values to detections
    hand_detections_2.mask, hand_detections_2.confidence = hand_masks_2, hand_confidence_2
    df_detections_2.mask, df_detections_2.confidence = df_masks_2, df_confidence_2
    dt_detections_2.mask, dt_detections_2.confidence = dt_masks_2, dt_confidence_2

    # Save Detections
    with sv.CSVSink(os.path.join(experiments_folder, 'Random Seed Sampling', 'Height Threshold', f'Seed {i}', difficulty, filename+'.csv')) as sink:
      sink.append(hand_detections_2, {'method':'Manual + LiDAR'})
      sink.append(df_detections_2, {'method':'DeepForest + LiDAR'})
      sink.append(dt_detections_2, {'method':'Detectree2 + LiDAR'})
      sink.append(dt_cnn_detections_2, {'method':'Baseline Filtered'})

    # Save Masks
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'Height Threshold', f'Seed {i}', difficulty, filename+'.npy'), 'wb') as f:
      if len(hand_detections_2) > 0:
        np.save(f, hand_detections_2.mask)
      if len(df_detections_2) > 0:
        np.save(f, df_detections_2.mask)
      if len(dt_detections_2) > 0:
        np.save(f, dt_detections_2.mask)
      if len(dt_cnn_detections_2) > 0:
        np.save(f, dt_cnn_detections_2.mask)

  for i in range(4,6):
    # Use boxes to label original LiDAR points
    hand_coords_3, hand_labels_3 = rasterize_lidar_old(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
    df_coords_3, df_labels_3 = rasterize_lidar_old(lidar_folder, rgb_folder, filename, boxes=df_boxes)
    dt_coords_3, dt_labels_3 = rasterize_lidar_old(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

    # Use original LiDAR points to filter boxes
    df_idx = lidar_filter(df_boxes, df_coords_3, df_labels_3, threshold=0.2)
    dt_idx = lidar_filter(dt_boxes, dt_coords_3, dt_labels_3, threshold=0.2)
    df_boxes_3, df_labels_3, df_scores_3 = df_boxes[df_idx], df_labels_3[df_idx], df_scores[df_idx]
    dt_boxes_3, dt_labels_3, dt_scores_3, dt_classes_3 = dt_boxes[dt_idx], dt_labels_3[dt_idx], dt_scores[dt_idx], dt_classes[dt_idx]

    # Create detections objects
    hand_detections_3 = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                      class_id=np.zeros(len(hand_boxes), dtype='int'))
    df_detections_3 = sv.Detections(xyxy=df_boxes_3, confidence=df_scores_3,
                                    class_id=np.zeros(len(df_boxes_3), dtype='int'))
    dt_detections_3 = sv.Detections(xyxy=dt_boxes_3, confidence=dt_scores_3,
                                    class_id=dt_classes_3)
    dt_cnn_detections_3 = dt_cnn_detections[dt_idx]

    # Sample LiDAR points for SAM-2
    pos = 10
    neg = 10
    hand_coords_3, hand_labels_3 = sample_points(hand_coords_3, hand_labels_3, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)
    if len(df_boxes_3) > 0:
      df_coords_3, df_labels_3 = sample_points(df_coords_3, df_labels_3, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)
    if len(dt_boxes_3) > 0:
      dt_coords_3, dt_labels_3 = sample_points(dt_coords_3, dt_labels_3, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)

    # Segment using SAM-2 on bounding boxes and LiDAR points
    hand_masks_3, hand_confidence_3 = segment_box_points(sam_predictor, hand_boxes, hand_coords_3, hand_labels_3, iterative=True)
    if len(df_boxes_3) > 0:
      df_masks_3, df_confidence_3 = segment_box_points(sam_predictor, df_boxes_3, df_coords_3, df_labels_3, iterative=True)
    else:
      df_masks_3, df_confidence_3 = np.empty((0,400,400)), np.empty((0))
    if len(dt_boxes_3) > 0:
      dt_masks_3, dt_confidence_3 = segment_box_points(sam_predictor, dt_boxes_3, dt_coords_3, dt_labels_3, iterative=True)
    else:
      dt_masks_3, dt_confidence_3 = np.empty((0,400,400)), np.empty((0))

    # Assign masks and confidence values to detections
    hand_detections_3.mask, hand_detections_3.confidence = hand_masks_3, hand_confidence_3
    df_detections_3.mask, df_detections_3.confidence = df_masks_3, df_confidence_3
    dt_detections_3.mask, dt_detections_3.confidence = dt_masks_3, dt_confidence_3

    # Save Detections
    with sv.CSVSink(os.path.join(experiments_folder, 'Random Seed Sampling', 'Classified', f'Seed {i}', difficulty, filename+'.csv')) as sink:
      sink.append(hand_detections_3, {'method':'Manual + LiDAR'})
      sink.append(df_detections_3, {'method':'DeepForest + LiDAR'})
      sink.append(dt_detections_3, {'method':'Detectree2 + LiDAR'})
      sink.append(dt_cnn_detections_3, {'method':'Baseline Filtered'})

    # Save Masks
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'Classified', f'Seed {i}', difficulty, filename+'.npy'), 'wb') as f:
      if len(hand_detections_3) > 0:
        np.save(f, hand_detections_3.mask)
      if len(df_detections_3) > 0:
        np.save(f, df_detections_3.mask)
      if len(dt_detections_3) > 0:
        np.save(f, dt_detections_3.mask)
      if len(dt_cnn_detections_3) > 0:
        np.save(f, dt_cnn_detections_3.mask)

In [ ]:
#@title Calculate Per-Tree Averages
'''
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

lidar_methods = ['Manual + LiDAR', 'DeepForest + LiDAR', 'Detectree2 + LiDAR', 'Baseline Filtered']

difficulties = ['Easy','Medium', 'Hard']

seeds = ['Seed 1', 'Seed 2', 'Seed 3', 'Seed 4', 'Seed 5']

columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)

original_preds = {difficulty:{method:{i:dict() for i in seeds} for method in lidar_methods} for difficulty in difficulties}
classified_preds = {difficulty:{method:{i:dict() for i in seeds} for method in methods} for difficulty in difficulties}
unfiltered_preds = {difficulty:{method:{i:dict() for i in seeds} for method in lidar_methods} for difficulty in difficulties}
truths = {difficulty:dict() for difficulty in difficulties}

for filename, difficulty in test_imgs:
  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths[difficulty][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Height Threshold', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Height Threshold', difficulty, filename+'.csv'), index_col='method')
    for method in lidar_methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      original_preds[difficulty][method]['Seed 1'][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR', difficulty, filename+'.csv'), index_col='method')
    for method in methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      classified_preds[difficulty][method]['Seed 1'][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.csv'), index_col='method')
    for method in lidar_methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      unfiltered_preds[difficulty][method]['Seed 1'][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  for i in seeds[1:]:
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'Height Threshold', i, difficulty, filename+'.npy'), 'rb') as f:
      df = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Height Threshold', i, difficulty, filename+'.csv'), index_col='method')
      for method in lidar_methods:
        if method in df.index:
          boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
          confidence = np.array(df.loc[(method,'confidence')])
          class_id = np.array(df.loc[(method,'class_id')])
          mask = np.load(f)
          # Add extra dimension for images with single prediction
          if boxes.ndim == 1:
            boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
        else:
          boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
        original_preds[difficulty][method][i][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'Classified', i, difficulty, filename+'.npy'), 'rb') as f:
      df = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Classified', i, difficulty, filename+'.csv'), index_col='method')
      for method in methods:
        if method in df.index:
          boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
          confidence = np.array(df.loc[(method,'confidence')])
          class_id = np.array(df.loc[(method,'class_id')])
          mask = np.load(f)
          # Add extra dimension for images with single prediction
          if boxes.ndim == 1:
            boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
        else:
          boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
        classified_preds[difficulty][method][i][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'No Filter', i, difficulty, filename+'.npy'), 'rb') as f:
      df = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'No Filter', i, difficulty, filename+'.csv'), index_col='method')
      for method in lidar_methods:
        if method in df.index:
          boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
          confidence = np.array(df.loc[(method,'confidence')])
          class_id = np.array(df.loc[(method,'class_id')])
          mask = np.load(f)
          # Add extra dimension for images with single prediction
          if boxes.ndim == 1:
            boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
        else:
          boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
        unfiltered_preds[difficulty][method][i][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)

# Per Image Metrics
index = [(difficulty, method, filename, i) for method in lidar_methods for filename, difficulty in test_imgs for i in seeds]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Filename', 'Seed'])
original_metrics_images = pd.DataFrame(index=index, columns=columns)
classified_metrics_images = pd.DataFrame(index=index, columns=columns)
unfiltered_metrics_images = pd.DataFrame(index=index, columns=columns)

for filename, difficulty in test_imgs:
  true_detections = truths[difficulty][filename]
  for method in lidar_methods:
    for i in seeds:
      original_detections = original_preds[difficulty][method][i][filename]
      classified_detections = classified_preds[difficulty][method][i][filename]
      unfiltered_detections = unfiltered_preds[difficulty][method][i][filename]
      original_metrics_images.loc[difficulty, method, filename, i] = per_tree_metrics([true_detections], [original_detections])
      classified_metrics_images.loc[difficulty, method, filename, i] = per_tree_metrics([true_detections], [classified_detections])
      unfiltered_metrics_images.loc[difficulty, method, filename, i] = per_tree_metrics([true_detections], [unfiltered_detections])
original_metrics_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - Avg by Image.csv'))
classified_metrics_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - Avg by Image.csv'))
unfiltered_metrics_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - Avg by Image.csv'))

# Per Difficulty Metrics
index = [(difficulty, method, i) for difficulty in difficulties for method in lidar_methods for i in seeds]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Seed'])
original_metrics_difficulties = pd.DataFrame(index=index, columns=columns)
classified_metrics_difficulties = pd.DataFrame(index=index, columns=columns)
unfiltered_metrics_difficulties = pd.DataFrame(index=index, columns=columns)

for diff in difficulties:
  for method in lidar_methods:
    for i in seeds:
      true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs if difficulty==diff]
      original_detections = [original_preds[difficulty][method][i][filename] for filename, difficulty in test_imgs if difficulty==diff]
      classified_detections = [classified_preds[difficulty][method][i][filename] for filename, difficulty in test_imgs if difficulty==diff]
      unfiltered_detections = [unfiltered_preds[difficulty][method][i][filename] for filename, difficulty in test_imgs if difficulty==diff]
      original_metrics_difficulties.loc[diff,method, i] = per_tree_metrics(true_detections, original_detections)
      classified_metrics_difficulties.loc[diff,method, i] = per_tree_metrics(true_detections, classified_detections)
      unfiltered_metrics_difficulties.loc[diff,method, i] = per_tree_metrics(true_detections, unfiltered_detections)
original_metrics_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - Avg by Difficulty.csv'))
classified_metrics_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - Avg by Difficulty.csv'))
unfiltered_metrics_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - Avg by Difficulty.csv'))

# Per Method Metrics
index = [(method, i) for method in lidar_methods for i in seeds]
index = pd.MultiIndex.from_tuples(index, names=['Method', 'Seed'])
original_metrics_methods = pd.DataFrame(index=index, columns=columns)
classified_metrics_methods = pd.DataFrame(index=index, columns=columns)
unfiltered_metrics_methods = pd.DataFrame(index=index, columns=columns)

for method in lidar_methods:
  for i in seeds:
    true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs]
    original_detections = [original_preds[difficulty][method][i][filename] for filename, difficulty in test_imgs]
    classified_detections = [classified_preds[difficulty][method][i][filename] for filename, difficulty in test_imgs]
    unfiltered_detections = [unfiltered_preds[difficulty][method][i][filename] for filename, difficulty in test_imgs]
    original_metrics_methods.loc[method, i] = per_tree_metrics(true_detections, original_detections)
    classified_metrics_methods.loc[method, i] = per_tree_metrics(true_detections, classified_detections)
    unfiltered_metrics_methods.loc[method, i] = per_tree_metrics(true_detections, unfiltered_detections)
original_metrics_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - Avg by Method.csv'))
classified_metrics_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - Avg by Method.csv'))
unfiltered_metrics_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - Avg by Method.csv'))
'''
original_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - Avg by Method.csv'), index_col=[0,1], header=[0,1])
classified_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - Avg by Method.csv'), index_col=[0,1], header=[0,1])
unfiltered_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - Avg by Method.csv'), index_col=[0,1], header=[0,1])

display(original_metrics_methods.style.format('{:.1%}'))
print()
display(classified_metrics_methods.style.format('{:.1%}'))
print()
display(unfiltered_metrics_methods.style.format('{:.1%}'))

In [ ]:
#@title Calculate Per-Tree Standard Deviations
'''
original_metrics_images = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])
classified_metrics_images = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])
unfiltered_metrics_images = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])

original_metrics_difficulties = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])
classified_metrics_difficulties = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])
unfiltered_metrics_difficulties = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])

original_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - Avg by Method.csv'), index_col=[0,1], header=[0,1])
classified_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - Avg by Method.csv'), index_col=[0,1], header=[0,1])
unfiltered_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - Avg by Method.csv'), index_col=[0,1], header=[0,1])


index_images = [(difficulty, method, filename) for method in lidar_methods for filename, difficulty in test_imgs]
index_difficulties = [(difficulty, method) for method in lidar_methods for filename, difficulty in test_imgs]

original_std_images = original_metrics_images.groupby(by=['Difficulty','Method','Filename']).std().reindex(index_images)
classified_std_images = classified_metrics_images.groupby(by=['Difficulty','Method','Filename']).std().reindex(index_images)
unfiltered_std_images = unfiltered_metrics_images.groupby(by=['Difficulty','Method','Filename']).std().reindex(index_images)
original_std_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - STD per Image.csv'))
classified_std_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - STD per Image.csv'))
unfiltered_std_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - STD per Image.csv'))

original_std_difficulties = original_metrics_difficulties.groupby(by=['Difficulty','Method']).std().reindex(index_difficulties)
classified_std_difficulties = classified_metrics_difficulties.groupby(by=['Difficulty','Method']).std().reindex(index_difficulties)
unfiltered_std_difficulties = unfiltered_metrics_difficulties.groupby(by=['Difficulty','Method']).std().reindex(index_difficulties)
original_std_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - STD per Difficulty.csv'))
classified_std_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - STD per Difficulty.csv'))
unfiltered_std_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - STD per Difficulty.csv'))

original_std_methods = original_metrics_methods.groupby(by=['Method']).std().reindex(lidar_methods)
classified_std_methods = classified_metrics_methods.groupby(by=['Method']).std().reindex(lidar_methods)
unfiltered_std_methods = unfiltered_metrics_methods.groupby(by=['Method']).std().reindex(lidar_methods)
original_std_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - STD per Method.csv'))
classified_std_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - STD per Method.csv'))
unfiltered_std_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - STD per Method.csv'))
'''
original_std_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Height Threshold - STD by Method.csv'), index_col=[0,1], header=[0,1])
classified_std_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR - STD by Method.csv'), index_col=[0,1], header=[0,1])
unfiltered_std_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Tree Metrics', 'Classified LiDAR (No Filter) - STD by Method.csv'), index_col=[0,1], header=[0,1])

display(original_std_methods.style.format('{:.1%}'))
print()
display(classified_std_methods.style.format('{:.1%}'))
print()
display(unfiltered_std_methods.style.format('{:.1%}'))

In [ ]:
#@title Calculate Per-Image Averages
'''
methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

lidar_methods = ['Manual + LiDAR', 'DeepForest + LiDAR', 'Detectree2 + LiDAR', 'Baseline Filtered']

difficulties = ['Easy','Medium', 'Hard']

seeds = ['Seed 1', 'Seed 2', 'Seed 3', 'Seed 4', 'Seed 5']

columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)

original_preds = {difficulty:{method:{i:dict() for i in seeds} for method in lidar_methods} for difficulty in difficulties}
classified_preds = {difficulty:{method:{i:dict() for i in seeds} for method in methods} for difficulty in difficulties}
unfiltered_preds = {difficulty:{method:{i:dict() for i in seeds} for method in lidar_methods} for difficulty in difficulties}
truths = {difficulty:dict() for difficulty in difficulties}

for filename, difficulty in test_imgs:
  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths[difficulty][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Height Threshold', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'Height Threshold', difficulty, filename+'.csv'), index_col='method')
    for method in lidar_methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      original_preds[difficulty][method]['Seed 1'][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR', difficulty, filename+'.csv'), index_col='method')
    for method in methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      classified_preds[difficulty][method]['Seed 1'][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Preprocessing', 'No Filter', difficulty, filename+'.csv'), index_col='method')
    for method in lidar_methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      unfiltered_preds[difficulty][method]['Seed 1'][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  for i in seeds[1:]:
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'Height Threshold', i, difficulty, filename+'.npy'), 'rb') as f:
      df = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Height Threshold', i, difficulty, filename+'.csv'), index_col='method')
      for method in lidar_methods:
        if method in df.index:
          boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
          confidence = np.array(df.loc[(method,'confidence')])
          class_id = np.array(df.loc[(method,'class_id')])
          mask = np.load(f)
          # Add extra dimension for images with single prediction
          if boxes.ndim == 1:
            boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
        else:
          boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
        original_preds[difficulty][method][i][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'Classified', i, difficulty, filename+'.npy'), 'rb') as f:
      df = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Classified', i, difficulty, filename+'.csv'), index_col='method')
      for method in methods:
        if method in df.index:
          boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
          confidence = np.array(df.loc[(method,'confidence')])
          class_id = np.array(df.loc[(method,'class_id')])
          mask = np.load(f)
          # Add extra dimension for images with single prediction
          if boxes.ndim == 1:
            boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
        else:
          boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
        classified_preds[difficulty][method][i][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
    with open(os.path.join(experiments_folder, 'Random Seed Sampling', 'No Filter', i, difficulty, filename+'.npy'), 'rb') as f:
      df = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'No Filter', i, difficulty, filename+'.csv'), index_col='method')
      for method in lidar_methods:
        if method in df.index:
          boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
          confidence = np.array(df.loc[(method,'confidence')])
          class_id = np.array(df.loc[(method,'class_id')])
          mask = np.load(f)
          # Add extra dimension for images with single prediction
          if boxes.ndim == 1:
            boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
        else:
          boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
        unfiltered_preds[difficulty][method][i][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)

# Per Image Metrics
index = [(difficulty, method, filename, i) for method in lidar_methods for filename, difficulty in test_imgs for i in seeds]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Filename', 'Seed'])
original_metrics_images = pd.DataFrame(index=index, columns=columns)
classified_metrics_images = pd.DataFrame(index=index, columns=columns)
unfiltered_metrics_images = pd.DataFrame(index=index, columns=columns)

for filename, difficulty in test_imgs:
  true_detections = truths[difficulty][filename]
  for method in lidar_methods:
    for i in seeds:
      original_detections = original_preds[difficulty][method][i][filename]
      classified_detections = classified_preds[difficulty][method][i][filename]
      unfiltered_detections = unfiltered_preds[difficulty][method][i][filename]

      true_boxes, true_masks = true_detections.xyxy, true_detections.mask
      original_boxes, original_masks = original_detections.xyxy, original_detections.mask
      classified_boxes, classified_masks = classified_detections.xyxy, classified_detections.mask
      unfiltered_boxes, unfiltered_masks = unfiltered_detections.xyxy, unfiltered_detections.mask

      original_detect_metrics = detection_metrics(true_masks, original_masks)
      classified_detect_metrics = detection_metrics(true_masks, classified_masks)
      unfiltered_detect_metrics = detection_metrics(true_masks, unfiltered_masks)

      true_idx, original_idx = hungarian_matching(true_boxes, original_boxes, threshold=0.5)
      original_iou = compute_iou(true_masks[true_idx], original_masks[original_idx], reduction='mean')
      original_segment_metrics = segmentation_metrics(true_masks[true_idx], original_masks[original_idx], reduction='mean')

      true_idx, classified_idx = hungarian_matching(true_boxes, classified_boxes, threshold=0.5)
      classified_iou = compute_iou(true_masks[true_idx], classified_masks[classified_idx], reduction='mean')
      classified_segment_metrics = segmentation_metrics(true_masks[true_idx], classified_masks[classified_idx], reduction='mean')

      true_idx, unfiltered_idx = hungarian_matching(true_boxes, unfiltered_boxes, threshold=0.5)
      unfiltered_iou = compute_iou(true_masks[true_idx], unfiltered_masks[unfiltered_idx], reduction='mean')
      unfiltered_segment_metrics = segmentation_metrics(true_masks[true_idx], unfiltered_masks[unfiltered_idx], reduction='mean')

      original_metrics_images.loc[difficulty, method, filename, i] = original_detect_metrics + original_segment_metrics + (original_iou,)
      classified_metrics_images.loc[difficulty, method, filename, i] = classified_detect_metrics + classified_segment_metrics + (classified_iou,)
      unfiltered_metrics_images.loc[difficulty, method, filename, i] = unfiltered_detect_metrics + unfiltered_segment_metrics + (unfiltered_iou,)
original_metrics_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - Avg by Image.csv'))
classified_metrics_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - Avg by Image.csv'))
unfiltered_metrics_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - Avg by Image.csv'))

# Per Difficulty Metrics
index = [(difficulty, method, i) for difficulty in difficulties for method in lidar_methods for i in seeds]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Seed'])
original_metrics_difficulties = original_metrics_images.groupby(by=['Difficulty','Method', 'Seed']).mean().reindex(index)
classified__metrics_difficulties = classified_metrics_images.groupby(by=['Difficulty','Method', 'Seed']).mean().reindex(index)
unfiltered_metrics_difficulties = unfiltered_metrics_images.groupby(by=['Difficulty','Method', 'Seed']).mean().reindex(index)
original_metrics_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - Avg by Difficulty.csv'))
classified_metrics_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - Avg by Difficulty.csv'))
unfiltered_metrics_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - Avg by Difficulty.csv'))

# Per Method Metrics
index = [(method, i) for method in lidar_methods for i in seeds]
index = pd.MultiIndex.from_tuples(index, names=['Method', 'Seed'])
original_metrics_methods = original_metrics_images.groupby(by=['Method', 'Seed']).mean().reindex(index)
classified_metrics_methods = classified_metrics_images.groupby(by=['Method', 'Seed']).mean().reindex(index)
unfiltered_metrics_methods = unfiltered_metrics_images.groupby(by=['Method', 'Seed']).mean().reindex(index)
original_metrics_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - Avg by Method.csv'))
classified_metrics_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - Avg by Method.csv'))
unfiltered_metrics_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - Avg by Method.csv'))
'''
original_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - Avg by Method.csv'), index_col=[0,1], header=[0,1])
classified_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - Avg by Method.csv'), index_col=[0,1], header=[0,1])
unfiltered_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - Avg by Method.csv'), index_col=[0,1], header=[0,1])

display(original_metrics_methods.style.format('{:.1%}'))
print()
display(classified_metrics_methods.style.format('{:.1%}'))
print()
display(unfiltered_metrics_methods.style.format('{:.1%}'))

In [ ]:
#@title Calculate Per-Image Standard Deviations

original_metrics_images = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])
classified_metrics_images = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])
unfiltered_metrics_images = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])

original_metrics_difficulties = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])
classified_metrics_difficulties = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])
unfiltered_metrics_difficulties = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])

original_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - Avg by Method.csv'), index_col=[0,1], header=[0,1])
classified_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - Avg by Method.csv'), index_col=[0,1], header=[0,1])
unfiltered_metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - Avg by Method.csv'), index_col=[0,1], header=[0,1])

index_images = [(difficulty, method, filename) for method in lidar_methods for filename, difficulty in test_imgs]
index_difficulties = [(difficulty, method) for method in lidar_methods for filename, difficulty in test_imgs]

original_std_images = original_metrics_images.groupby(by=['Difficulty','Method','Filename']).std().reindex(index_images)
classified_std_images = classified_metrics_images.groupby(by=['Difficulty','Method','Filename']).std().reindex(index_images)
unfiltered_std_images = unfiltered_metrics_images.groupby(by=['Difficulty','Method','Filename']).std().reindex(index_images)
original_std_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - STD per Image.csv'))
classified_std_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - STD per Image.csv'))
unfiltered_std_images.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - STD per Image.csv'))

original_std_difficulties = original_metrics_difficulties.groupby(by=['Difficulty','Method']).std().reindex(index_difficulties)
classified_std_difficulties = classified_metrics_difficulties.groupby(by=['Difficulty','Method']).std().reindex(index_difficulties)
unfiltered_std_difficulties = unfiltered_metrics_difficulties.groupby(by=['Difficulty','Method']).std().reindex(index_difficulties)
original_std_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - STD per Difficulty.csv'))
classified_std_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - STD per Difficulty.csv'))
unfiltered_std_difficulties.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - STD per Difficulty.csv'))

original_std_methods = original_metrics_methods.groupby(by=['Method']).std().reindex(lidar_methods)
classified_std_methods = classified_metrics_methods.groupby(by=['Method']).std().reindex(lidar_methods)
unfiltered_std_methods = unfiltered_metrics_methods.groupby(by=['Method']).std().reindex(lidar_methods)
original_std_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Height Threshold - STD per Method.csv'))
classified_std_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR - STD per Method.csv'))
unfiltered_std_methods.to_csv(os.path.join(experiments_folder, 'Random Seed Sampling', 'Per Image Metrics', 'Classified LiDAR (No Filter) - STD per Method.csv'))

display(original_std_methods)
print()
display(classified_std_methods)
print()
display(unfiltered_std_methods)

In [ ]:
#@title Random Seed Sampling Image

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(12,8))

for filename, difficulty in [('2018_SJER_3_254000_4108000_image_700', 'Easy')]:
  for i in range(1,6):
    # Get image
    img = ds.get_image(filename)
    rgb_img = img['rgb']
    bgr_img = rgb_img[:,:,::-1].copy()
    sam_predictor.set_image(rgb_img)

    # Get ground truths
    true_boxes = img['annotation']
    true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))
    box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
    true_boxes, true_masks = true_boxes[box_idx], true_masks[mask_idx]
    true_detections = sv.Detections(xyxy=true_boxes, mask=true_masks, confidence=np.ones(len(true_boxes)),
                                                class_id=np.zeros(len(true_boxes), dtype='int'))

    # Collect hand-drawn bounding boxes
    hand_boxes = img['annotation']

    # Create detections objects for boxes
    hand_detections = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                      class_id=np.zeros(len(hand_boxes), dtype='int'))

    # Extract boxes and scores for convenience
    hand_boxes, hand_scores = hand_detections.xyxy, hand_detections.confidence

    # Use boxes to label unfiltered LiDAR points
    hand_coords_1, hand_labels_1 = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)

    # Create detections objects
    hand_detections_1 = sv.Detections(xyxy=hand_boxes, confidence=np.ones(len(hand_boxes)),
                                      class_id=np.zeros(len(hand_boxes), dtype='int'))

    # Sample LiDAR points for SAM-2
    pos = 10
    neg = 10
    hand_coords_1, hand_labels_1 = sample_points(hand_coords_1, hand_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=i)

    # Segment using SAM-2 on bounding boxes and LiDAR points
    hand_masks_1, hand_confidence_1 = segment_box_points(sam_predictor, hand_boxes, hand_coords_1, hand_labels_1, iterative=True)

    # Assign masks and confidence values to detections
    hand_detections_1.mask, hand_detections_1.confidence = hand_masks_1, hand_confidence_1

    # Plot Images
    poly_annotator = sv.PolygonAnnotator(thickness=2, color=sv.Color.RED)
    mask_annotator = sv.MaskAnnotator()

    hand_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[2])
    hand_img = mask_annotator.annotate(hand_img, detections=hand_detections_1[2])[:,:,::-1]

    j = i // 3
    k = i % 3
    axs[j,k] = show_as_points(hand_img, hand_detections_1, hand_coords_1[2:3], hand_labels_1[2:3], ax=axs[j,k], show_negative=True, marker_size=50)
    axs[j,k].set_title(f'Seed {i}', fontsize=14)

true_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[2])[:,:,::-1]
axs[0,0].imshow(true_img)
axs[0,0].set_title('Base Image', fontsize=14)

for i in range(2):
  for j in range(3):
    axs[i,j].set_frame_on(False)
    axs[i,j].get_xaxis().set_ticks([])
    axs[i,j].get_yaxis().set_ticks([])
plt.tight_layout()
plt.savefig(f'{filename} Random Seed Sampling.jpg')
plt.show()

In [ ]:
#@title Compare Bounding Box Sizes

methods = ['Manual Boxes', 'DeepForest Boxes', 'Detectree2 Boxes']

difficulties = ['Easy','Medium', 'Hard']

manual_boxes, df_boxes, dt_boxes = [],[],[]
all_boxes = [manual_boxes, df_boxes, dt_boxes]

for filename, difficulty in test_imgs:
  df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.csv'), index_col='method')
  for i, method in enumerate(methods):
    if method in df.index:
      boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
      # Add extra dimension for images with single prediction
      if boxes.ndim == 1:
        boxes = boxes[np.newaxis]
    else:
      continue
    all_boxes[i].append(boxes)

manual_boxes = np.concatenate(manual_boxes)
df_boxes = np.concatenate(df_boxes)
dt_boxes = np.concatenate(dt_boxes)

manual_boxes = [(box[2]-box[0]) * (box[3]-box[1]) for box in manual_boxes]
df_boxes = [(box[2]-box[0]) * (box[3]-box[1]) for box in df_boxes]
dt_boxes = [(box[2]-box[0]) * (box[3]-box[1]) for box in dt_boxes]
'''
fig, axs = plt.subplots(3,1, sharex=True, sharey=True, figsize=(5,5))
axs[0].hist(manual_boxes)
axs[1].hist(df_boxes)
axs[2].hist(dt_boxes)
axs[0].set_title('Manual Boxes')
axs[1].set_title('DeepForest Boxes')
axs[2].set_title('Detectree2 Boxes')
axs[1].set_ylabel('Box Count')
axs[2].set_xlabel('Box Area (pixels)')
for i in range(3):
  axs[i].set_xlim(0,25000)
plt.tight_layout()
plt.savefig('Bounding Box Size Histogram')
plt.show()
print()
'''
fig, axs = plt.subplots(figsize=(3.55,3.45))
axs.boxplot([manual_boxes, df_boxes, dt_boxes])
axs.set_title('Bounding Box Sizes')
axs.set_xticks([1,2,3],['Manual', 'DeepForest', 'Detectree2'])
plt.tight_layout()
plt.savefig('Bounding Box Size Box Plots.pdf', dpi=500)
plt.show()
print()
fig, axs = plt.subplots(figsize=(3.55,3.45))
axs.boxplot([manual_boxes, df_boxes, dt_boxes])
axs.set_title('Bounding Box Sizes')
axs.set_xticks([1,2,3],['Manual', 'DeepForest', 'Detectree2'])
axs.set_ylim(-1000,30000)
plt.tight_layout()
plt.savefig('Bounding Box Size Box Plots (Clipped).pdf', dpi=500)
plt.show()

In [ ]:
#@title Box to Mask Comparison

methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
           'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']

box_methods = ['Manual Boxes', 'DeepForest Boxes', 'Detectree2 Boxes', 'Baseline']

difficulties = ['Easy','Medium', 'Hard']

columns = [('Box','Precision'), ('Box','Recall'),('Box','F1'),
           ('Mask','Precision'), ('Mask','Recall'), ('Mask','F1')]
columns = pd.MultiIndex.from_tuples(columns)

preds = {difficulty:{method:dict() for method in methods} for difficulty in difficulties}
truths = {difficulty:dict() for difficulty in difficulties}

for filename, difficulty in test_imgs:
  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths[difficulty][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)
  with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.npy'), 'rb') as f:
    df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.csv'), index_col='method')
    for method in methods:
      if method in df.index:
        boxes = np.array(df.loc[(method,['x_min','y_min','x_max','y_max'])])
        confidence = np.array(df.loc[(method,'confidence')])
        class_id = np.array(df.loc[(method,'class_id')])
        mask = np.load(f)
        # Add extra dimension for images with single prediction
        if boxes.ndim == 1:
          boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
      else:
        boxes, confidence, class_id, mask = np.empty((0,4)), np.empty((0)), np.empty((0)), np.empty((0,400,400))
      preds[difficulty][method][filename] = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)

# Per Image Metrics
index = [(difficulty, method, filename) for method in box_methods for filename, difficulty in test_imgs]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'Filename'])
metrics_images = pd.DataFrame(index=index, columns=columns)

for filename, difficulty in test_imgs:
  true_detections = truths[difficulty][filename]
  for method in box_methods:
    pred_detections = preds[difficulty][method][filename]
    metrics_images.loc[difficulty, method, filename] = box_mask_metrics([true_detections], [pred_detections])
metrics_images.to_csv(os.path.join(experiments_folder, 'Box to Mask Comparison', 'Per Tree Metrics', 'Avg by Image.csv'))

# Per Difficulty Metrics
index = [(difficulty, method) for difficulty in difficulties for method in box_methods]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method'])
metrics_difficulty = pd.DataFrame(index=index, columns=columns)

for diff in difficulties:
  for method in box_methods:
    true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs if difficulty==diff]
    pred_detections = [preds[difficulty][method][filename] for filename, difficulty in test_imgs if difficulty==diff]
    metrics_difficulty.loc[diff,method] = box_mask_metrics(true_detections, pred_detections)
metrics_difficulty.to_csv(os.path.join(experiments_folder, 'Box to Mask Comparison', 'Per Tree Metrics', 'Avg by Difficulty.csv'))

# Per Method Metrics
metrics_methods = pd.DataFrame(index=box_methods, columns=columns)

for method in box_methods:
  true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs]
  pred_detections = [preds[difficulty][method][filename] for filename, difficulty in test_imgs]
  metrics_methods.loc[method] = box_mask_metrics(true_detections, pred_detections)
metrics_methods.to_csv(os.path.join(experiments_folder, 'Box to Mask Comparison', 'Per Tree Metrics', 'Avg by Method.csv'))

metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'Box to Mask Comparison', 'Per Tree Metrics', 'Avg by Method.csv'), index_col=[0], header=[0,1])
metrics_difficulty = pd.read_csv(os.path.join(experiments_folder, 'Box to Mask Comparison', 'Per Tree Metrics', 'Avg by Difficulty.csv'), index_col=[0,1], header=[0,1])
metrics_images = pd.read_csv(os.path.join(experiments_folder, 'Box to Mask Comparison', 'Per Tree Metrics', 'Avg by Image.csv'), index_col=[0,1,2], header=[0,1])

display(metrics_methods.style.format('{:.1%}'))
print()
display(metrics_difficulty.style.format('{:.1%}'))
print()
display(metrics_images.style.format('{:.1%}'))

In [ ]:
#@title Calculate Mask Size with vs without LiDAR
from collections import defaultdict
import json
import pickle
'''
hand_box_sizes = defaultdict(list)
df_box_sizes = defaultdict(list)
dt_box_sizes = defaultdict(list)

hand_point_sizes = defaultdict(list)
df_point_sizes = defaultdict(list)
dt_point_sizes = defaultdict(list)

true_sizes = defaultdict(list)

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)

  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()

  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()

  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]

  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter automatically detected boxes
  df_idx = lidar_filter(df_boxes, df_coords, df_labels)
  df_boxes, df_labels = df_boxes[df_idx], df_labels[df_idx]
  dt_idx = lidar_filter(dt_boxes, dt_coords, dt_labels)
  dt_boxes, dt_labels = dt_boxes[dt_idx], dt_labels[dt_idx]

  # Sample LiDAR points for SAM-2 (given boxes)
  pos = 10
  neg = 10
  hand_coords, hand_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes) > 0:
    df_coords, df_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes) > 0:
    dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using Boxes alone (mask threshold of 0.2)
  hand_box_masks, hand_box_confidence = segment_boxes(sam_predictor, hand_boxes)
  if len(df_boxes) > 0:
    df_box_masks, df_box_confidence = segment_boxes(sam_predictor, df_boxes)
  else:
    df_box_masks, df_box_confidence = np.empty((0,400,400)), np.empty((0))
  if len(dt_boxes) > 0:
    dt_box_masks, dt_box_confidence = segment_boxes(sam_predictor, dt_boxes)
  else:
    dt_box_masks, dt_box_confidence = np.empty((0,400,400)), np.empty((0))

  hand_box_sizes['all'] += [mask.sum() for mask in hand_box_masks]
  df_box_sizes['all'] += [mask.sum() for mask in df_box_masks]
  dt_box_sizes['all'] += [mask.sum() for mask in dt_box_masks]

  # Match to True Masks and Save Comparable Sizes
  true_idx, hand_idx = hungarian_matching(true_masks, hand_box_masks, threshold=0.5)
  hand_box_sizes['true'] += [mask.sum() for mask in hand_box_masks[hand_idx]]
  true_sizes['Hand Box'] += [mask.sum() for mask in true_masks[true_idx]]
  true_idx, df_idx = hungarian_matching(true_masks, df_box_masks, threshold=0.5)
  df_box_sizes['true'] += [mask.sum() for mask in df_box_masks[df_idx]]
  true_sizes['DF Box'] += [mask.sum() for mask in true_masks[true_idx]]
  true_idx, dt_idx = hungarian_matching(true_masks, dt_box_masks, threshold=0.5)
  dt_box_sizes['true'] += [mask.sum() for mask in dt_box_masks[dt_idx]]
  true_sizes['DT Box'] += [mask.sum() for mask in true_masks[true_idx]]

  # Segment using Boxes + LiDAR (mask threshold of 0.2)
  hand_point_masks, hand_point_confidence = segment_box_points(sam_predictor, hand_boxes, hand_coords, hand_labels, iterative=True)
  if len(df_boxes) > 0:
    df_point_masks, df_point_confidence = segment_box_points(sam_predictor, df_boxes, df_coords, df_labels, iterative=True)
  else:
    df_point_masks, df_point_confidence = np.empty((0,400,400)), np.empty((0))
  if len(dt_boxes) > 0:
    dt_point_masks, dt_point_confidence = segment_box_points(sam_predictor, dt_boxes, dt_coords, dt_labels, iterative=True)
  else:
    dt_point_masks, dt_point_confidence = np.empty((0,400,400)), np.empty((0))

  hand_point_sizes['all'] += [mask.sum() for mask in hand_point_masks]
  df_point_sizes['all'] += [mask.sum() for mask in df_point_masks]
  dt_point_sizes['all'] += [mask.sum() for mask in dt_point_masks]

  # Match to True Masks and Save Comparable Sizes
  true_idx, hand_idx = hungarian_matching(true_masks, hand_point_masks, threshold=0.5)
  hand_point_sizes['true'] += [mask.sum() for mask in hand_point_masks[hand_idx]]
  true_sizes['Hand Point'] += [mask.sum() for mask in true_masks[true_idx]]
  true_idx, df_idx = hungarian_matching(true_masks, df_point_masks, threshold=0.5)
  df_point_sizes['true'] += [mask.sum() for mask in df_point_masks[df_idx]]
  true_sizes['DF Point'] += [mask.sum() for mask in true_masks[true_idx]]
  true_idx, dt_idx = hungarian_matching(true_masks, dt_point_masks, threshold=0.5)
  dt_point_sizes['true'] += [mask.sum() for mask in dt_point_masks[dt_idx]]
  true_sizes['DT Point'] += [mask.sum() for mask in true_masks[true_idx]]

with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'True Sizes.pkl'), 'wb') as f:
  pickle.dump(true_sizes, f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'Manual Boxes Sizes.pkl'), 'wb') as f:
  pickle.dump(hand_box_sizes, f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'Manual + LiDAR Sizes.pkl'), 'wb') as f:
  pickle.dump(hand_point_sizes, f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'DeepForest Boxes.pkl'), 'wb') as f:
  pickle.dump(df_box_sizes, f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'DeepForest + LiDAR.pkl'), 'wb') as f:
  pickle.dump(df_point_sizes, f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'Detectree2 Boxes.pkl'), 'wb') as f:
  pickle.dump(dt_box_sizes, f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'Detectree2 + LiDAR.pkl'), 'wb') as f:
  pickle.dump(dt_point_sizes, f)
'''
print()

In [ ]:
#@title Display Mask Size with vs without LiDAR

# This is close to the largest true segmentation
size_limit = 20000

with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'True Sizes.pkl'), 'rb') as f:
  true_sizes = pickle.load(f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'Manual Boxes Sizes.pkl'), 'rb') as f:
  hand_box_sizes = pickle.load(f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'Manual + LiDAR Sizes.pkl'), 'rb') as f:
  hand_point_sizes = pickle.load(f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'DeepForest Boxes.pkl'), 'rb') as f:
  df_box_sizes = pickle.load(f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'DeepForest + LiDAR.pkl'), 'rb') as f:
  df_point_sizes = pickle.load(f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'Detectree2 Boxes.pkl'), 'rb') as f:
  dt_box_sizes = pickle.load(f)
with open(os.path.join(experiments_folder, 'Mask Size with vs without LiDAR', 'Detectree2 + LiDAR.pkl'), 'rb') as f:
  dt_point_sizes = pickle.load(f)

fig, axs = plt.subplots(nrows=3, ncols=3, sharex=True, sharey=True, figsize=(7.5,7.5))
axs[0,0].scatter(hand_box_sizes['all'], hand_point_sizes['all'], s=10)
axs[0,1].scatter(df_box_sizes['all'], df_point_sizes['all'], s=10)
axs[0,2].scatter(dt_box_sizes['all'], dt_point_sizes['all'], s=10)
axs[1,0].scatter(true_sizes['Hand Point'], hand_point_sizes['true'], s=10)
axs[1,1].scatter(true_sizes['DF Point'], df_point_sizes['true'], s=10)
axs[1,2].scatter(true_sizes['DT Point'], dt_point_sizes['true'], s=10)
axs[2,0].scatter(true_sizes['Hand Box'], hand_box_sizes['true'], s=10)
axs[2,1].scatter(true_sizes['DF Box'], df_box_sizes['true'], s=10)
axs[2,2].scatter(true_sizes['DT Box'], dt_box_sizes['true'], s=10)

#fig.suptitle('Mask Size with vs without LiDAR')
axs[0,0].set_title('Manual Boxes')
axs[0,1].set_title('DeepForest Boxes')
axs[0,2].set_title('Detectree2 Boxes')

axs[0,0].set_ylabel('With LiDAR')
axs[1,0].set_ylabel('With LiDAR')
axs[2,0].set_ylabel('Without LiDAR')
for i in range(3):
  #axs[0,i].set_ylabel('With LiDAR')
  axs[0,i].set_xlabel('Without LiDAR')
  #axs[1,i].set_ylabel('With LiDAR')
  axs[1,i].set_xlabel('Ground Truth')
  #axs[2,i].set_ylabel('Without LiDAR')
  axs[2,i].set_xlabel('Ground Truth')
  for j in range(3):
    axs[i,j].set_ylim(0,size_limit)
    axs[i,j].set_xlim(0,size_limit)
    axs[i,j].plot([0, 1], [0, 1], transform=axs[i,j].transAxes, linestyle='dashed')

plt.tight_layout()
plt.savefig('Mask Sizes.pdf', dpi=500)
plt.show()

In [ ]:
#@title LiDAR as Filter vs Prompt

methods = ['DeepForest + LiDAR', 'Detectree2 + LiDAR']

difficulties = ['Easy','Medium', 'Hard']

lidar_methods = ['Filter','Prompt']

columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)

preds = {difficulty:{method:{lidar:dict() for lidar in lidar_methods} for method in methods} for difficulty in difficulties}
truths = {difficulty:dict() for difficulty in difficulties}

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect Ground Truths
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes, true_masks = true_boxes[box_idx], true_masks[mask_idx]
  true_detections = sv.Detections(xyxy=true_boxes, mask=true_masks, confidence=np.ones(len(true_boxes)),
                                               class_id=np.zeros(len(true_boxes), dtype='int'))
  truths[difficulty][filename] = true_detections

  # Generate automatic bounding boxes using DeepForest
  df_preds = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  if df_preds is not None:
    df_preds = df_preds[df_preds['score'] >= 0.2]
    df_scores = df_preds['score'].to_numpy()
    df_boxes = df_preds[['xmin','ymin','xmax','ymax']].to_numpy()
  else:
    df_scores = np.empty((0))
    df_boxes = np.empty((0,4))

  # Generate automatic bounding boxes using Detectree2
  dt_preds = dt_model(bgr_img)
  if dt_preds is not None:
    dt_boxes = dt_preds['instances'].pred_boxes.tensor.cpu().numpy()
    dt_scores = dt_preds['instances'].scores.cpu().numpy()
    dt_classes = dt_preds['instances'].pred_classes.cpu().numpy()
    # Filter with confidence threshold of 0.2
    dt_idx = dt_scores >= 0.2
    dt_boxes = dt_boxes[dt_idx]
    dt_scores = dt_scores[dt_idx]
    dt_classes = dt_classes[dt_idx]
  else:
    dt_boxes = np.empty((0,4))
    dt_scores = np.empty((0))
    dt_classes = np.empty((0))

  # Create detections objects for boxes
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores,
                                  class_id=np.zeros(len(df_boxes), dtype='int'))
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores,
                                  class_id=dt_classes)

  # Apply custom non-max suppression to automatic boxes
  df_idx = custom_nms(df_detections, threshold=0.5)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  df_detections = df_detections[df_idx]
  dt_detections = dt_detections[dt_idx]

  # Extract boxes and scores for convenience
  df_boxes, df_scores = df_detections.xyxy, df_detections.confidence
  dt_boxes, dt_scores = dt_detections.xyxy, dt_detections.confidence

  # Use boxes to label LiDAR points
  df_coords, df_labels = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter boxes
  df_idx = lidar_filter(df_boxes, df_coords, df_labels, threshold=0.2)
  dt_idx = lidar_filter(dt_boxes, dt_coords, dt_labels, threshold=0.2)
  df_boxes_1, df_labels_1, df_scores_1 = df_boxes[df_idx], df_labels[df_idx], df_scores[df_idx]
  dt_boxes_1, dt_labels_1, dt_scores_1, dt_classes_1 = dt_boxes[dt_idx], dt_labels[dt_idx], dt_scores[dt_idx], dt_classes[dt_idx]

  # Create detections objects
  df_detections_1 = sv.Detections(xyxy=df_boxes_1, confidence=df_scores_1,
                                  class_id=np.zeros(len(df_boxes_1), dtype='int'))
  dt_detections_1 = sv.Detections(xyxy=dt_boxes_1, confidence=dt_scores_1,
                                  class_id=dt_classes_1)

  # Segment using SAM-2 on bounding boxes
  if len(df_boxes_1) > 0:
    df_masks_1, df_confidence_1 = segment_boxes(sam_predictor, df_boxes_1)
  else:
    df_masks_1, df_confidence_1 = np.empty((0,400,400)), np.empty((0))
  if len(dt_boxes_1) > 0:
    dt_masks_1, dt_confidence_1 = segment_boxes(sam_predictor, dt_boxes_1)
  else:
    dt_masks_1, dt_confidence_1 = np.empty((0,400,400)), np.empty((0))

  # Assign masks and confidence values to detections
  df_detections_1.mask, df_detections_1.confidence = df_masks_1, df_confidence_1
  dt_detections_1.mask, dt_detections_1.confidence = dt_masks_1, dt_confidence_1

  # Filter masks with confidence threshold of 0.2
  df_idx = df_confidence_1 >= 0.2
  dt_idx = dt_confidence_1 >= 0.2
  df_detections_1 = df_detections_1[df_idx]
  dt_detections_1 = dt_detections_1[dt_idx]

  preds[difficulty]['DeepForest + LiDAR']['Filter'][filename] = df_detections_1
  preds[difficulty]['Detectree2 + LiDAR']['Filter'][filename] = dt_detections_1

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  if len(df_boxes) > 0:
    df_coords_2, df_labels_2 = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes) > 0:
    dt_coords_2, dt_labels_2 = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2 on bounding boxes and LiDAR points
  if len(df_boxes) > 0:
    df_masks_2, df_confidence_2 = segment_box_points(sam_predictor, df_boxes, df_coords_2, df_labels_2, iterative=True)
  else:
    df_masks_2, df_confidence_2 = np.empty((0,400,400)), np.empty((0))
  if len(dt_boxes) > 0:
    dt_masks_2, dt_confidence_2 = segment_box_points(sam_predictor, dt_boxes, dt_coords_2, dt_labels_2, iterative=True)
  else:
    dt_masks_2, dt_confidence_2 = np.empty((0,400,400)), np.empty((0))

  # Assign masks and confidence values to detections
  df_detections_2, dt_detections_2 = df_detections, dt_detections
  df_detections_2.mask, df_detections_2.confidence = df_masks_2, df_confidence_2
  dt_detections_2.mask, dt_detections_2.confidence = dt_masks_2, dt_confidence_2

  # Filter masks with confidence threshold of 0.2
  df_idx = df_confidence_2 >= 0.2
  dt_idx = dt_confidence_2 >= 0.2
  df_detections_2 = df_detections_2[df_idx]
  dt_detections_2 = dt_detections_2[dt_idx]

  preds[difficulty]['DeepForest + LiDAR']['Prompt'][filename] = df_detections_2
  preds[difficulty]['Detectree2 + LiDAR']['Prompt'][filename] = dt_detections_2

# Per Image Metrics
index = [(difficulty, method, lidar, filename) for method in methods for filename, difficulty in test_imgs for lidar in lidar_methods]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'LiDAR', 'Filename'])
metrics_images = pd.DataFrame(index=index, columns=columns)

for filename, difficulty in test_imgs:
  true_detections = truths[difficulty][filename]
  for method in methods:
    for lidar in lidar_methods:
      pred_detections = preds[difficulty][method][lidar][filename]
      metrics_images.loc[difficulty, method, lidar, filename] = per_tree_metrics([true_detections], [pred_detections])
metrics_images.to_csv(os.path.join(experiments_folder, 'LiDAR Filter vs Prompt', 'Per Tree Metrics', 'Avg by Image.csv'))

# Per Difficulty Metrics
index = [(difficulty, method, lidar) for difficulty in difficulties for method in methods for lidar in lidar_methods]
index = pd.MultiIndex.from_tuples(index, names=['Difficulty', 'Method', 'LiDAR'])
metrics_difficulty = pd.DataFrame(index=index, columns=columns)

for diff in difficulties:
  for method in methods:
    for lidar in lidar_methods:
      true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs if difficulty==diff]
      pred_detections = [preds[difficulty][method][lidar][filename] for filename, difficulty in test_imgs if difficulty==diff]
      metrics_difficulty.loc[diff,method,lidar] = per_tree_metrics(true_detections, pred_detections)
metrics_difficulty.to_csv(os.path.join(experiments_folder, 'LiDAR Filter vs Prompt', 'Per Tree Metrics', 'Avg by Difficulty.csv'))

# Per Method Metrics
index = [(method, lidar) for method in methods for lidar in lidar_methods]
index = pd.MultiIndex.from_tuples(index, names=['Method', 'LiDAR'])
metrics_methods = pd.DataFrame(index=index, columns=columns)

for method in methods:
  for lidar in lidar_methods:
    true_detections = [truths[difficulty][filename] for filename, difficulty in test_imgs]
    pred_detections = [preds[difficulty][method][lidar][filename] for filename, difficulty in test_imgs]
    metrics_methods.loc[method,lidar] = per_tree_metrics(true_detections, pred_detections)
metrics_methods.to_csv(os.path.join(experiments_folder, 'LiDAR Filter vs Prompt', 'Per Tree Metrics', 'Avg by Method.csv'))

metrics_methods = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Filter vs Prompt', 'Per Tree Metrics', 'Avg by Method.csv'), index_col=[0,1], header=[0,1])
metrics_difficulty = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Filter vs Prompt', 'Per Tree Metrics', 'Avg by Difficulty.csv'), index_col=[0,1,2], header=[0,1])
metrics_images = pd.read_csv(os.path.join(experiments_folder, 'LiDAR Filter vs Prompt', 'Per Tree Metrics', 'Avg by Image.csv'), index_col=[0,1,2,3], header=[0,1])

display(metrics_methods.style.format('{:.1%}'))
print()
display(metrics_difficulty.style.format('{:.1%}'))
print()
display(metrics_images.style.format('{:.1%}'))

In [ ]:
#@title Detectree2 Low Density Example

methods = ['Detectree2 Boxes', 'Detectree2 + LiDAR']

filename, difficulty = '2018_TEAK_3_318000_4107000_image_718', 'Easy'
# Get image
img = ds.get_image(filename)
rgb_img = img['rgb']
bgr_img = rgb_img[:,:,::-1].copy()

with open(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.npy'), 'rb') as f:
  df = pd.read_csv(os.path.join(experiments_folder, 'Main Results - Classified LiDAR (No Filter)', difficulty, filename+'.csv'), index_col='method')
  boxes_1 = np.array(df.loc[('Detectree2 Boxes',['x_min','y_min','x_max','y_max'])])
  confidence_1 = np.array(df.loc[('Detectree2 Boxes','confidence')])
  class_id_1 = np.array(df.loc[('Detectree2 Boxes','class_id')])
  # Add extra dimension for images with single prediction
  if boxes_1.ndim == 1:
    boxes_1, confidence_1, class_id_1 = boxes_1[np.newaxis], confidence_1[np.newaxis], class_id_1[np.newaxis]
  preds_1 = sv.Detections(xyxy=boxes_1, confidence=confidence_1, class_id=class_id_1)

  boxes_2 = np.array(df.loc[('Detectree2 + LiDAR',['x_min','y_min','x_max','y_max'])])
  confidence_2 = np.array(df.loc[('Detectree2 + LiDAR','confidence')])
  class_id_2 = np.array(df.loc[('Detectree2 + LiDAR','class_id')])
  # Add extra dimension for images with single prediction
  if boxes_2.ndim == 1:
    boxes_2, confidence_2, class_id_2 = boxes_2[np.newaxis], confidence_2[np.newaxis], class_id_2[np.newaxis]
  preds_2 = sv.Detections(xyxy=boxes_2, confidence=confidence_2, class_id=class_id_2)

box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.BLUE)
preds_1 = box_annotator.annotate(bgr_img.copy(), detections=preds_1)[:,:,::-1]
preds_2 = box_annotator.annotate(bgr_img.copy(), detections=preds_2)[:,:,::-1]

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(5.5,2.75))
axs[0].imshow(preds_1)
axs[1].imshow(preds_2)
axs[0].set_title('Detectree2 Boxes')
axs[1].set_title('Detectree2 + LiDAR')
axs[0].axis('off')
axs[1].axis('off')
plt.tight_layout()
plt.savefig('Detectree2 Low Density.tif', dpi=500)
plt.show()

In [ ]:
#@title Table 1 Images

for filename, difficulty in [('2018_TEAK_3_318000_4107000_image_718','Easy'),('2018_SJER_3_254000_4108000_image_700','Easy'),
                             ('TEAK_049_2018','Medium'),('SJER_025_2018','Medium'),('LENO_066_2019','Hard'),('ABBY_076_2019','Hard')]:
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()

  with open(os.path.join(annotation_folder, difficulty, filename+'.npy'), 'rb') as f:
    mask = np.load(f)
  df = pd.read_csv(os.path.join(annotation_folder, difficulty, filename+'.csv'))
  boxes = np.array(df[['x_min','y_min','x_max','y_max']])
  confidence = np.array(df['confidence'])
  class_id = np.array(df['class_id'])
  if boxes.ndim == 1:
    boxes, confidence, class_id = boxes[np.newaxis], confidence[np.newaxis], class_id[np.newaxis]
  truths = sv.Detections(xyxy=boxes, confidence=confidence, class_id=class_id, mask=mask)

  poly_annotator = sv.PolygonAnnotator(thickness=1, color=sv.Color.BLUE)
  true_img = poly_annotator.annotate(bgr_img, detections=truths)[:,:,::-1]

  fig, axs = plt.subplots(figsize=(2,2))
  axs.imshow(true_img)
  plt.axis('off')
  plt.tight_layout()
  plt.savefig(f'{filename} segmentation.tif', dpi=500)
  plt.show()
  print()

In [ ]:
#@title Another Image

methods = ['Detectree2 Boxes', 'Detectree2 + LiDAR']

filename, difficulty = '2018_SJER_3_252000_4106000_image_234', 'Easy'
for filename, difficulty in [('2018_SJER_3_252000_4106000_image_234', 'Easy')]:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()

  # Generate automatic bounding boxes using Detectree2
  dt_preds = dt_model(bgr_img)
  if dt_preds is not None:
    dt_boxes = dt_preds['instances'].pred_boxes.tensor.cpu().numpy()
    dt_scores = dt_preds['instances'].scores.cpu().numpy()
    dt_classes = dt_preds['instances'].pred_classes.cpu().numpy()
    dt_cnn_masks = dt_preds['instances'].pred_masks.cpu().numpy()
    # Filter with confidence threshold of 0.2
    dt_idx = dt_scores >= 0.2
    dt_boxes = dt_boxes[dt_idx]
    dt_scores = dt_scores[dt_idx]
    dt_classes = dt_classes[dt_idx]
    dt_cnn_masks = dt_cnn_masks[dt_idx]
  else:
    dt_boxes = np.empty((0,4))
    dt_scores = np.empty((0))
    dt_classes = np.empty((0))
    dt_cnn_masks = np.empty(0,400,400)

  # Create detections objects for boxes
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores,
                                  class_id=dt_classes)

  # Apply custom non-max suppression to automatic boxes
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_detections = dt_detections[dt_idx]

  # Extract boxes and scores for convenience
  dt_boxes, dt_scores = dt_detections.xyxy, dt_detections.confidence

  # Use boxes to label unfiltered LiDAR points
  dt_coords_1, dt_labels_1 = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=dt_boxes)

  # Use unfiltered LiDAR points to filter boxes
  dt_idx = lidar_filter(dt_boxes, dt_coords_1, dt_labels_1, threshold=0.2)
  dt_boxes_1, dt_labels_1, dt_scores_1, dt_classes_1 = dt_boxes[dt_idx], dt_labels_1[dt_idx], dt_scores[dt_idx], dt_classes[dt_idx]

  # Create detections objects
  dt_detections_1 = sv.Detections(xyxy=dt_boxes_1, confidence=dt_scores_1,
                                  class_id=dt_classes_1)

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  if len(dt_boxes_1) > 0:
    dt_coords_1, dt_labels_1 = sample_points(dt_coords_1, dt_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=5)

box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.RED)
preds_1 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_1)[:,:,::-1]
show_as_points(preds_1, dt_detections_1, dt_coords_1, dt_labels_1, show_negative=True, show_boxes=True, save=True)


In [ ]:
#@title Another Other Image

methods = ['Detectree2 Boxes', 'Detectree2 + LiDAR']

filename, difficulty = '2018_SJER_3_252000_4106000_image_234', 'Easy'
for filename, difficulty in [('2018_SJER_3_252000_4106000_image_234', 'Easy')]:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()

  # Generate automatic bounding boxes using Detectree2
  dt_preds = dt_model(bgr_img)
  if dt_preds is not None:
    dt_boxes = dt_preds['instances'].pred_boxes.tensor.cpu().numpy()
    dt_scores = dt_preds['instances'].scores.cpu().numpy()
    dt_classes = dt_preds['instances'].pred_classes.cpu().numpy()
    dt_cnn_masks = dt_preds['instances'].pred_masks.cpu().numpy()
    # Filter with confidence threshold of 0.2
    dt_idx = dt_scores >= 0.2
    dt_boxes = dt_boxes[dt_idx]
    dt_scores = dt_scores[dt_idx]
    dt_classes = dt_classes[dt_idx]
    dt_cnn_masks = dt_cnn_masks[dt_idx]
  else:
    dt_boxes = np.empty((0,4))
    dt_scores = np.empty((0))
    dt_classes = np.empty((0))
    dt_cnn_masks = np.empty(0,400,400)

  # Create detections objects for boxes
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores,
                                  class_id=dt_classes)

  # Apply custom non-max suppression to automatic boxes
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_detections = dt_detections[dt_idx]

  # Extract boxes and scores for convenience
  dt_boxes, dt_scores = dt_detections.xyxy, dt_detections.confidence

  # Use boxes to label unfiltered LiDAR points
  dt_coords_0, dt_labels_0 = rasterize_lidar(lidar_unfiltered, rgb_folder, filename, boxes=dt_boxes)

  # Use unfiltered LiDAR points to filter boxes
  dt_idx = lidar_filter(dt_boxes, dt_coords_0, dt_labels_0, threshold=0.2)
  dt_boxes_1, dt_labels_1, dt_scores_1, dt_classes_1 = dt_boxes[dt_idx], dt_labels_0[dt_idx], dt_scores[dt_idx], dt_classes[dt_idx]

  # Create detections objects
  dt_detections_1 = sv.Detections(xyxy=dt_boxes_1, confidence=dt_scores_1,
                                  class_id=dt_classes_1)

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  if len(dt_boxes_1) > 0:
    dt_coords_2, dt_labels_2 = sample_points(dt_coords_0, dt_labels_1, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=5)

box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.BLUE)
preds_0 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections)[:,:,::-1]
preds_1 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_1)[:,:,::-1]

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(7.5,2.5))
axs[0] = show_as_mask(preds_0.copy(), dt_detections, dt_coords_0, dt_labels_0, ax=axs[0])
axs[1] = show_as_mask(preds_1.copy(), dt_detections_1, dt_coords_0, dt_labels_1, ax=axs[1])
axs[2] = show_as_points(preds_1.copy(), dt_detections_1, dt_coords_2, dt_labels_2, show_negative=True, ax=axs[2], marker_size=15)
plt.tight_layout()
plt.savefig('LiDAR.tif', dpi=500)
plt.show()

In [ ]:
#@title LiDAR as Filter (Old)

filter_methods = ['Manual Boxes', 'DeepForest Filtered', 'Detectree2 Filtered', 'Baseline Filtered']
filter_index = [(img[1], method, img[0]) for method in filter_methods for img in test_imgs]
filter_index = pd.MultiIndex.from_tuples(filter_index, names=['Difficulty', 'Method', 'Filename'])
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)
filter_metrics = pd.DataFrame(index=filter_index, columns=columns)

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect Manual bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)

  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()

  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()

  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Use boxes to label LiDAR points
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter boxes
  df_idx = lidar_filter(df_boxes, df_coords, df_labels)
  df_boxes, df_labels = df_boxes[df_idx], df_labels[df_idx]
  dt_idx = lidar_filter(dt_boxes, dt_coords, dt_labels)
  dt_boxes, dt_labels, dt_cnn_masks = dt_boxes[dt_idx], dt_labels[dt_idx], dt_cnn_masks[dt_idx]

  # Segment using SAM-2 (mask threshold of 0.2)
  hand_masks = segment_boxes(sam_predictor, hand_boxes, threshold=0.2)
  if len(df_boxes) > 0:
    df_masks = segment_boxes(sam_predictor, df_boxes, threshold=0.2)
  else:
    df_masks = np.empty((0,400,400))
  if len(dt_boxes) > 0:
    dt_masks = segment_boxes(sam_predictor, dt_boxes, threshold=0.2)
  else:
    dt_masks = np.empty((0,400,400))

  # Match predicted masks to boxes (in case of filtered out masks)
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes, df_masks = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]

  # Calculate detection based metrics for masks using 50% threshold
  hand_detect_metrics = detection_metrics(true_masks, hand_masks)
  df_detect_metrics = detection_metrics(true_masks, df_masks)
  dt_detect_metrics = detection_metrics(true_masks, dt_masks)
  dt_cnn_detect_metrics = detection_metrics(true_masks, dt_cnn_masks)

  # For correct detections (based on boxes), calculate IoU and Segmentation metrics
  true_idx, hand_idx = hungarian_matching(true_boxes, hand_boxes, threshold=0.5)
  hand_iou_metric = compute_iou(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')
  hand_seg_metrics = segmentation_metrics(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')

  true_idx, df_idx = hungarian_matching(true_boxes, df_boxes, threshold=0.5)
  df_iou_metric = compute_iou(true_masks[true_idx], df_masks[df_idx], reduction='mean')
  df_seg_metrics = segmentation_metrics(true_masks[true_idx], df_masks[df_idx], reduction='mean')

  true_idx, dt_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
  dt_iou_metric = compute_iou(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')
  dt_seg_metrics = segmentation_metrics(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')

  true_idx, dt_cnn_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
  dt_cnn_iou_metric = compute_iou(true_masks[true_idx], dt_cnn_masks[dt_cnn_idx], reduction='mean')
  dt_cnn_seg_metrics = segmentation_metrics(true_masks[true_idx], dt_cnn_masks[dt_cnn_idx], reduction='mean')

  # Store metrics
  filter_metrics.loc[difficulty, 'Manual Boxes', filename] = hand_detect_metrics + hand_seg_metrics + (hand_iou_metric,)
  filter_metrics.loc[difficulty, 'DeepForest Filtered', filename] = df_detect_metrics + df_seg_metrics + (df_iou_metric,)
  filter_metrics.loc[difficulty, 'Detectree2 Filtered', filename] = dt_detect_metrics + dt_seg_metrics + (dt_iou_metric,)
  filter_metrics.loc[difficulty, 'Baseline Filtered', filename] = dt_cnn_detect_metrics + dt_cnn_seg_metrics + (dt_cnn_iou_metric,)

# Save Metrics
#filter_metrics.to_csv('Filtered Box Workflow Metrics.csv')

filter_order = [(difficulty, method) for difficulty in ['Easy','Medium','Hard'] for method in filter_methods]
filter_diff_metrics = filter_metrics.groupby(by=['Difficulty','Method']).mean().reindex(filter_order)
display(filter_diff_metrics)
print()
filter_avg_metrics = filter_metrics.groupby(by=['Method']).mean().reindex(filter_methods)
display(filter_avg_metrics)

In [ ]:
#@title LiDAR as Prompt (Old)
no_filter_methods = ['Manual + LiDAR No Filter', 'DeepForest + LiDAR No Filter', 'Detectree2 + LiDAR No Filter']
no_filter_index = [(img[1], method, img[0]) for method in no_filter_methods for img in test_imgs]
no_filter_index = pd.MultiIndex.from_tuples(no_filter_index, names=['Difficulty', 'Method', 'Filename'])
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)
no_filter_metrics = pd.DataFrame(index=no_filter_index, columns=columns)

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect Manual bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)

  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()

  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()

  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Sample LiDAR points for SAM-2 (given boxes)
  pos = 10
  neg = 10
  hand_coords, hand_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes) > 0:
    df_coords, df_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes) > 0:
    dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2 (mask threshold of 0.2)
  hand_masks = segment_box_points(sam_predictor, hand_boxes, hand_coords, hand_labels, threshold=0.2, iterative=True)
  if len(df_boxes) > 0:
    df_masks = segment_box_points(sam_predictor, df_boxes, df_coords, df_labels, threshold=0.2, iterative=True)
  else:
    df_masks = np.empty((0,400,400))
  if len(dt_boxes) > 0:
    dt_masks = segment_box_points(sam_predictor, dt_boxes, dt_coords, dt_labels, threshold=0.2, iterative=True)
  else:
    dt_masks = np.empty((0,400,400))

  # Match predicted masks to boxes (in case of filtered out masks)
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes, hand_masks = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes, df_masks = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]

  # Calculate detection based metrics for masks using 50% threshold
  hand_detect_metrics = detection_metrics(true_masks, hand_masks)
  df_detect_metrics = detection_metrics(true_masks, df_masks)
  dt_detect_metrics = detection_metrics(true_masks, dt_masks)

  # For correct detections (based on boxes), calculate IoU and Segmentation metrics
  true_idx, hand_idx = hungarian_matching(true_boxes, hand_boxes, threshold=0.5)
  hand_iou_metric = compute_iou(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')
  hand_seg_metrics = segmentation_metrics(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')

  true_idx, df_idx = hungarian_matching(true_boxes, df_boxes, threshold=0.5)
  df_iou_metric = compute_iou(true_masks[true_idx], df_masks[df_idx], reduction='mean')
  df_seg_metrics = segmentation_metrics(true_masks[true_idx], df_masks[df_idx], reduction='mean')

  true_idx, dt_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
  dt_iou_metric = compute_iou(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')
  dt_seg_metrics = segmentation_metrics(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')

  # Store metrics
  no_filter_metrics.loc[difficulty, 'Manual + LiDAR No Filter', filename] = hand_detect_metrics + hand_seg_metrics + (hand_iou_metric,)
  no_filter_metrics.loc[difficulty, 'DeepForest + LiDAR No Filter', filename] = df_detect_metrics + df_seg_metrics + (df_iou_metric,)
  no_filter_metrics.loc[difficulty, 'Detectree2 + LiDAR No Filter', filename] = dt_detect_metrics + dt_seg_metrics + (dt_iou_metric,)

# Save Metrics
#no_filter_metrics.to_csv('Box + LiDAR No Filter Workflow Metrics.csv')

no_filter_order = [(difficulty, method) for difficulty in ['Easy','Medium','Hard'] for method in no_filter_methods]
no_filter_diff_metrics = no_filter_metrics.groupby(by=['Difficulty','Method']).mean().reindex(no_filter_order)
display(no_filter_diff_metrics)
print()
no_filter_avg_metrics = no_filter_metrics.groupby(by=['Method']).mean().reindex(no_filter_methods)
display(no_filter_avg_metrics)

## Boxes with LiDAR Filter, 10 Positive Points (Iterative)

In [ ]:
pos_point_methods = ['Manual + LiDAR (Pos)', 'DeepForest + LiDAR (Pos)', 'Detectree2 + LiDAR (Pos)', 'Baseline Filtered']
pos_point_index = [(img[1], method, img[0]) for method in pos_point_methods for img in test_imgs]
pos_point_index = pd.MultiIndex.from_tuples(pos_point_index, names=['Difficulty', 'Method', 'Filename'])
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)
pos_point_metrics = pd.DataFrame(index=pos_point_index, columns=columns)

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)

  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()

  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()

  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter automatically detected boxes
  df_idx = lidar_filter(df_boxes, df_coords, df_labels)
  df_boxes, df_labels = df_boxes[df_idx], df_labels[df_idx]
  dt_idx = lidar_filter(dt_boxes, dt_coords, dt_labels)
  dt_boxes, dt_labels, dt_cnn_masks = dt_boxes[dt_idx], dt_labels[dt_idx], dt_cnn_masks[dt_idx]

  # Sample LiDAR points for SAM-2 (given boxes)
  pos = 10
  neg = 0
  hand_coords, hand_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes) > 0:
    df_coords, df_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes) > 0:
    dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2 (mask threshold of 0.2)
  hand_masks = segment_box_points(sam_predictor, hand_boxes, hand_coords, hand_labels, threshold=0.2, iterative=True)
  if len(df_boxes) > 0:
    df_masks = segment_box_points(sam_predictor, df_boxes, df_coords, df_labels, threshold=0.2, iterative=True)
  else:
    df_masks = np.empty((0,400,400))
  if len(dt_boxes) > 0:
    dt_masks = segment_box_points(sam_predictor, dt_boxes, dt_coords, dt_labels, threshold=0.2, iterative=True)
  else:
    dt_masks = np.empty((0,400,400))

  # Match predicted masks to boxes (in case of filtered out masks)
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes, hand_masks = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes, df_masks = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]

  # Calculate detection based metrics for masks using 50% threshold
  hand_detect_metrics = detection_metrics(true_masks, hand_masks)
  df_detect_metrics = detection_metrics(true_masks, df_masks)
  dt_detect_metrics = detection_metrics(true_masks, dt_masks)
  dt_cnn_detect_metrics = detection_metrics(true_masks, dt_cnn_masks)

  # For correct detections (based on boxes), calculate IoU and Segmentation metrics
  true_idx, hand_idx = hungarian_matching(true_boxes, hand_boxes, threshold=0.5)
  hand_iou_metric = compute_iou(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')
  hand_seg_metrics = segmentation_metrics(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')

  true_idx, df_idx = hungarian_matching(true_boxes, df_boxes, threshold=0.5)
  df_iou_metric = compute_iou(true_masks[true_idx], df_masks[df_idx], reduction='mean')
  df_seg_metrics = segmentation_metrics(true_masks[true_idx], df_masks[df_idx], reduction='mean')

  true_idx, dt_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
  dt_iou_metric = compute_iou(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')
  dt_seg_metrics = segmentation_metrics(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')

  true_idx, dt_cnn_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
  dt_cnn_iou_metric = compute_iou(true_masks[true_idx], dt_cnn_masks[dt_cnn_idx], reduction='mean')
  dt_cnn_seg_metrics = segmentation_metrics(true_masks[true_idx], dt_cnn_masks[dt_cnn_idx], reduction='mean')

  # Store metrics
  pos_point_metrics.loc[difficulty, 'Manual + LiDAR (Pos)', filename] = hand_detect_metrics + hand_seg_metrics + (hand_iou_metric,)
  pos_point_metrics.loc[difficulty, 'DeepForest + LiDAR (Pos)', filename] = df_detect_metrics + df_seg_metrics + (df_iou_metric,)
  pos_point_metrics.loc[difficulty, 'Detectree2 + LiDAR (Pos)', filename] = dt_detect_metrics + dt_seg_metrics + (dt_iou_metric,)
  pos_point_metrics.loc[difficulty, 'Baseline Filtered', filename] = dt_cnn_detect_metrics + dt_cnn_seg_metrics + (dt_cnn_iou_metric,)

# Save Metrics
pos_point_metrics.to_csv('Box + LiDAR (Pos) Workflow Metrics.csv')

pos_point_order = [(difficulty, method) for difficulty in ['Easy','Medium','Hard'] for method in pos_point_methods]
pos_point_diff_metrics = pos_point_metrics.groupby(by=['Difficulty','Method']).mean().reindex(pos_point_order)
display(pos_point_diff_metrics)
print()
pos_point_avg_metrics = pos_point_metrics.groupby(by=['Method']).mean().reindex(pos_point_methods)
display(pos_point_avg_metrics)

## 10 Positive and 10 Negative Points (Iterative)

In [ ]:
metrics['hand_only_points'] = {'Easy':dict(), 'Medium':dict(), 'Hard':dict()}
metrics['df_only_points'] = {'Easy':dict(), 'Medium':dict(), 'Hard':dict()}
metrics['dt_only_points'] = {'Easy':dict(), 'Medium':dict(), 'Hard':dict()}

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()
  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()
  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]
  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Use boxes to label LiDAR points
  #start_time = timeit.default_timer()
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)
  #print(f'Lidar label time: {timeit.default_timer() - start_time}')

  # Use labeled LiDAR points to filter boxes
  idx = lidar_filter(hand_boxes, hand_coords, hand_labels)
  hand_boxes, hand_labels = hand_boxes[idx], hand_labels[idx]
  idx = lidar_filter(df_boxes, df_coords, df_labels)
  df_boxes, df_labels = df_boxes[idx], df_labels[idx]
  idx = lidar_filter(dt_boxes, dt_coords, dt_labels)
  dt_boxes, dt_labels, dt_cnn_masks = dt_boxes[idx], dt_labels[idx], dt_cnn_masks[idx]

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  hand_coords, hand_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  df_coords, df_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2
  hand_masks = segment_points(sam_predictor, hand_coords, hand_labels, threshold=0.2, iterative=True)
  df_masks = segment_points(sam_predictor, df_coords, df_labels, threshold=0.2, iterative=True)
  dt_masks = segment_points(sam_predictor, dt_coords, dt_labels, threshold=0.2, iterative=True)

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes, hand_masks = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes, df_masks = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]

  # Calculate detection based metrics using 50% threshold on masks
  hand_detect_metrics = detection_metrics(true_masks, hand_masks)
  df_detect_metrics = detection_metrics(true_masks, df_masks)
  dt_detect_metrics = detection_metrics(true_masks, dt_masks)
  dt_cnn_detect_metrics = detection_metrics(true_masks, dt_cnn_masks)

  # For correct detections (based on boxes), calculate IoU and pixel metrics
  true_idx, hand_idx = hungarian_matching(true_boxes, hand_boxes, threshold=0.5)
  hand_iou_metric = compute_iou(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')
  hand_pixel_metrics = pixel_metrics(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')

  true_idx, df_idx = hungarian_matching(true_boxes, df_boxes, threshold=0.5)
  df_iou_metric = compute_iou(true_masks[true_idx], df_masks[df_idx], reduction='mean')
  df_pixel_metrics = pixel_metrics(true_masks[true_idx], df_masks[df_idx], reduction='mean')

  true_idx, dt_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
  dt_iou_metric = compute_iou(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')
  dt_pixel_metrics = pixel_metrics(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')

  true_idx, dt_cnn_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
  dt_cnn_iou_metric = compute_iou(true_masks[true_idx], dt_cnn_masks[dt_cnn_idx], reduction='mean')
  dt_cnn_pixel_metrics = pixel_metrics(true_masks[true_idx], dt_cnn_masks[dt_cnn_idx], reduction='mean')

  # Store metrics
  metrics['hand_only_points'][difficulty][filename] = hand_detect_metrics + (hand_iou_metric,) + hand_pixel_metrics
  metrics['df_only_points'][difficulty][filename] = df_detect_metrics + (df_iou_metric,) + df_pixel_metrics
  metrics['dt_only_points'][difficulty][filename] = dt_detect_metrics + (dt_iou_metric,) + dt_pixel_metrics

# Put metrics in array indexed by (in order of dimension) Prompt, Difficulty, Image, and Metric
only_points_metrics_array = np.zeros(shape=(3,3,6,7))
for i, method in enumerate(['hand_only_points', 'df_only_points', 'dt_only_points']):
  for j, difficulty in enumerate(['Easy', 'Medium', 'Hard']):
    for k, img in enumerate(metrics[method][difficulty].keys()):
      only_points_metrics_array[i,j,k] = metrics[method][difficulty][img]

# Separate out Easy, Medium, Hard, and all metrics, and take mean across images
easy_metrics = np.nanmean(only_points_metrics_array[:,0], axis=1)
medium_metrics = np.nanmean(only_points_metrics_array[:,1], axis=1)
hard_metrics = np.nanmean(only_points_metrics_array[:,2], axis=1)
all_metrics = np.nanmean(only_points_metrics_array.reshape((3,18,7)), axis=1)

# Display as Pandas DataFrames
print()
print('Easy Images')
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Pixel','IoU'), ('Pixel','Precision'), ('Pixel','Recall'), ('Pixel','F1')]
columns = pd.MultiIndex.from_tuples(columns)
index = ['Hand-Drawn Points', 'DeepForest Points', 'Detectree-2 Points']
easy_df = pd.DataFrame(easy_metrics, index=index, columns=columns)
display(easy_df)

print()
print('Medium Images')
medium_df = pd.DataFrame(medium_metrics, index=index, columns=columns)
display(medium_df)

print()
print('Hard Images')
hard_df = pd.DataFrame(hard_metrics, index=index, columns=columns)
display(hard_df)

print()
print('All Images')
all_df = pd.DataFrame(all_metrics, index=index, columns=columns)
display(all_df)

# Box to Mask Comparison

In [ ]:
comparison_folder = '/content/drive/MyDrive/Sam+LiDAR/Experiments/Box to Mask Comparison'
box_methods = ['Manual Boxes', 'DeepForest Boxes','Detectree2 Boxes', 'Baseline']
box_mask_metrics = pd.read_csv(os.path.join(comparison_folder, 'Box to Mask Comparison.csv'), index_col=[0,1,2], header=[0,1])
box_mask_avg = box_mask_metrics.groupby(by=['Method']).mean().reindex(box_methods)
box_mask_std = box_mask_metrics.groupby(by=['Method']).std().reindex(box_methods)
display(box_mask_avg)
print()
display(box_mask_std)

# LiDAR as Filter vs Prompt

In [ ]:
filter_folder = '/content/drive/MyDrive/Sam+LiDAR/Experiments/Filter vs Point Prompt Tests'
filter_methods = ['Manual Boxes', 'DeepForest Filtered', 'Detectree2 Filtered', 'Baseline Filtered']
prompt_methods = ['Manual + LiDAR No Filter', 'DeepForest + LiDAR No Filter', 'Detectree2 + LiDAR No Filter']
filter_metrics = pd.read_csv(os.path.join(filter_folder, 'LiDAR as Filter.csv'), index_col=[0,1,2], header=[0,1])
prompt_metrics = pd.read_csv(os.path.join(filter_folder, 'LiDAR as Prompt.csv'), index_col=[0,1,2], header=[0,1])
filter_avg = filter_metrics.groupby(by=['Method']).mean().reindex(filter_methods).loc[['DeepForest Filtered', 'Detectree2 Filtered']]
prompt_avg = prompt_metrics.groupby(by=['Method']).mean().reindex(prompt_methods).loc[['DeepForest + LiDAR No Filter', 'Detectree2 + LiDAR No Filter']]
display(filter_avg)
print()
display(prompt_avg)

# Detectree2 Confidence Thresholds

In [ ]:
thresholds = [0.1, 0.2]
detectree_methods = ['Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']
detectree_index = [(img[1], method, thresh, img[0]) for method in detectree_methods for thresh in thresholds for img in test_imgs]
detectree_index = pd.MultiIndex.from_tuples(detectree_index, names=['Difficulty', 'Method', 'Threshold', 'Filename'])
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)
detectree_metrics = pd.DataFrame(index=detectree_index, columns=columns)
display(detectree_metrics)
print()

difficulty_order = [(difficulty, method, thresh) for difficulty in ['Easy','Medium','Hard'] for method in detectree_methods for thresh in thresholds]
detectree_diff_metrics = detectree_metrics.groupby(by=['Difficulty','Method', 'Threshold']).mean().reindex(difficulty_order)
display(detectree_diff_metrics)
print()

avg_order = [(method, thresh) for method in detectree_methods for thresh in thresholds]
detectree_avg_metrics = detectree_metrics.groupby(by=['Method', 'Threshold']).mean().reindex(avg_order)
display(detectree_avg_metrics)

In [ ]:
threshold_folder = '/content/drive/MyDrive/Sam+LiDAR/Experiments/Detectree2 Threshold Tests'
thresholds = [0.1, 0.2]
detectree_methods = ['Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']
detectree_index = [(img[1], method, thresh, img[0]) for method in detectree_methods for thresh in thresholds for img in test_imgs]
detectree_index = pd.MultiIndex.from_tuples(detectree_index, names=['Difficulty', 'Method', 'Threshold', 'Filename'])
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)
detectree_metrics = pd.DataFrame(index=detectree_index, columns=columns)

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  baseline_masks = dt_outputs['instances'].pred_masks.cpu().numpy()

  for thresh in thresholds:
    dt_idx = np.nonzero(dt_scores >= thresh)
    dt_boxes = dt_boxes[dt_idx]
    dt_scores = dt_scores[dt_idx]
    dt_classes = dt_classes[dt_idx]
    baseline_masks = baseline_masks[dt_idx]

    # Apply custom non-max suppression
    dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
    dt_idx = custom_nms(dt_detections, threshold=0.5)
    dt_boxes = dt_boxes[dt_idx]
    dt_scores = dt_scores[dt_idx]
    dt_classes = dt_classes[dt_idx]
    baseline_masks = baseline_masks[dt_idx]

    # Segment using SAM-2 (mask threshold of 0.2)
    dt_masks = segment_boxes(sam_predictor, dt_boxes, threshold=0.2)

    # Match predicted masks to boxes (in case of filtered out masks)
    box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
    dt_boxes, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]

    # Calculate detection based metrics for boxes using 50% threshold
    dt_box_metrics = detection_metrics(true_boxes, dt_boxes)

    # Calculate detection based metrics for masks using 50% threshold
    dt_detect_metrics = detection_metrics(true_masks, dt_masks)
    baseline_detect_metrics = detection_metrics(true_masks, baseline_masks)

    # For correct detections (based on boxes), calculate IoU and Segmentation metrics
    true_idx, dt_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
    dt_iou_metric = compute_iou(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')
    dt_seg_metrics = segmentation_metrics(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')

    true_idx, baseline_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
    baseline_iou_metric = compute_iou(true_masks[true_idx], baseline_masks[baseline_idx], reduction='mean')
    baseline_seg_metrics = segmentation_metrics(true_masks[true_idx], baseline_masks[baseline_idx], reduction='mean')

    # Store metrics
    detectree_metrics.loc[difficulty, 'Detectree2 Boxes', thresh, filename] = dt_detect_metrics + dt_seg_metrics + (dt_iou_metric,)
    detectree_metrics.loc[difficulty, 'Baseline', thresh, filename] = baseline_detect_metrics + baseline_seg_metrics + (baseline_iou_metric,)

    # Use boxes to label LiDAR points
    dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

    # Use labeled LiDAR points to filter automatically detected boxes
    dt_idx = lidar_filter(dt_boxes, dt_coords, dt_labels)
    dt_lidar_boxes, dt_labels, baseline_filter_masks = dt_boxes[dt_idx], dt_labels[dt_idx], baseline_masks[dt_idx]

    # Sample LiDAR points for SAM-2 (given boxes)
    pos = 10
    neg = 10
    if len(dt_lidar_boxes) > 0:
      dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

    # Segment using SAM-2 (mask threshold of 0.2)
    if len(dt_lidar_boxes) > 0:
      dt_lidar_masks = segment_box_points(sam_predictor, dt_lidar_boxes, dt_coords, dt_labels, threshold=0.2, iterative=True)
    else:
      dt_lidar_masks = np.empty((0,400,400))

    # Match predicted masks to boxes (in case of filtered out masks)
    box_idx, mask_idx = hungarian_matching(dt_lidar_boxes, dt_lidar_masks)
    dt_lidar_boxes, dt_lidar_masks = dt_lidar_boxes[box_idx], dt_lidar_masks[mask_idx]

    # Calculate detection based metrics for masks using 50% threshold
    dt_lidar_detect_metrics = detection_metrics(true_masks, dt_lidar_masks)
    baseline_filter_detect_metrics = detection_metrics(true_masks, baseline_filter_masks)

    # For correct detections (based on boxes), calculate IoU and Segmentation metrics
    true_idx, dt_idx = hungarian_matching(true_boxes, dt_lidar_boxes, threshold=0.5)
    dt_lidar_iou_metric = compute_iou(true_masks[true_idx], dt_lidar_masks[dt_idx], reduction='mean')
    dt_lidar_seg_metrics = segmentation_metrics(true_masks[true_idx], dt_lidar_masks[dt_idx], reduction='mean')

    true_idx, baseline_idx = hungarian_matching(true_boxes, dt_lidar_boxes, threshold=0.5)
    baseline_filter_iou_metric = compute_iou(true_masks[true_idx], baseline_filter_masks[baseline_idx], reduction='mean')
    baseline_filter_seg_metrics = segmentation_metrics(true_masks[true_idx], baseline_filter_masks[baseline_idx], reduction='mean')

    # Store metrics
    detectree_metrics.loc[difficulty, 'Detectree2 + LiDAR', thresh, filename] = dt_lidar_detect_metrics + dt_lidar_seg_metrics + (dt_lidar_iou_metric,)
    detectree_metrics.loc[difficulty, 'Baseline Filtered', thresh, filename] = baseline_filter_detect_metrics + baseline_filter_seg_metrics + (baseline_filter_iou_metric,)

# Save Metrics
detectree_metrics.to_csv(os.path.join(threshold_folder, 'Detectree2 Threshold Tests - All Results.csv'))

difficulty_order = [(difficulty, method, thresh) for difficulty in ['Easy','Medium','Hard'] for method in detectree_methods for thresh in thresholds]
detectree_diff_metrics = detectree_metrics.groupby(by=['Difficulty','Method', 'Threshold']).mean().reindex(difficulty_order)
detectree_diff_metrics.to_csv(os.path.join(threshold_folder, 'Detectree2 Threshold Tests - Avg by Difficulty.csv'))
display(detectree_diff_metrics)
print()

avg_order = [(method, thresh) for method in detectree_methods for thresh in thresholds]
detectree_avg_metrics = detectree_metrics.groupby(by=['Method', 'Threshold']).mean().reindex(avg_order)
detectree_avg_metrics.to_csv(os.path.join(threshold_folder, 'Detectree2 Threshold Tests - Avg by Method.csv'))
display(detectree_avg_metrics)

In [ ]:
# 0.2 Threshold for Easy, 0.1 for Medium and Hard

detectree_easy = detectree_diff_metrics.loc[('Easy', detectree_methods, 0.2)].droplevel('Threshold')
detectree_medium = detectree_diff_metrics.loc[('Medium', detectree_methods, 0.1)].droplevel('Threshold')
detectree_hard = detectree_diff_metrics.loc[('Hard', detectree_methods, 0.1)].droplevel('Threshold')

detectree_best = pd.concat([detectree_easy, detectree_medium, detectree_hard]).reindex(difficulty_order)
detectree_best = detectree_best.groupby(by=['Method']).mean().reindex(detectree_methods)
detectree_best.to_csv(os.path.join(threshold_folder, 'Detectree2 Threshold Tests - Best Avg.csv'))
detectree_best

In [ ]:
styler1 = detectree_best.style \
  .format('{:.1%}') \
  .set_table_styles([{'selector':'th','props':'text-align: left'}], overwrite=False) \
  .hide(axis=0, names=True)

styler1


In [ ]:
styler2 = detectree_avg_metrics.loc[(detectree_methods, 0.1),].droplevel('Threshold').style \
  .format('{:.1%}') \
  .set_table_styles([{'selector':'th','props':'text-align: left'}], overwrite=False) \
  .hide(axis=0, names=True)

styler2

In [ ]:
styler3 = detectree_avg_metrics.loc[(detectree_methods, 0.2),].droplevel('Threshold').style \
  .format('{:.1%}') \
  .set_table_styles([{'selector':'th','props':'text-align: left'}], overwrite=False) \
  .hide(axis=0, names=True)

styler3

In [ ]:
styler3 = detectree_avg_metrics.style \
  .format('{:.1%}') \
  .set_table_styles([{'selector':'th','props':'text-align: left'}], overwrite=False) \

styler3

In [ ]:
fig, axs = plt.subplots(18,3, figsize=(9,54))

for i, (filename, difficulty) in enumerate(test_imgs):
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_detections = sv.Detections(xyxy=true_boxes, confidence=np.ones(len(true_boxes)), class_id=np.zeros(len(true_boxes),dtype='int32'))

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()

  dt1_idx = np.nonzero(dt_scores >= 0.2)
  dt1_boxes = dt_boxes[dt1_idx]
  dt1_scores = dt_scores[dt1_idx]
  dt1_classes = dt_classes[dt1_idx]

  # Apply custom non-max suppression
  dt1_detections = sv.Detections(xyxy=dt1_boxes, confidence=dt1_scores, class_id=dt1_classes)
  dt1_idx = custom_nms(dt1_detections, threshold=0.5)
  dt1_detections = dt1_detections[dt1_idx]

  dt2_idx = np.nonzero(dt_scores >= 0.2)
  dt2_boxes = dt_boxes[dt2_idx]
  dt2_scores = dt_scores[dt2_idx]
  dt2_classes = dt_classes[dt2_idx]

  # Apply custom non-max suppression
  dt2_detections = sv.Detections(xyxy=dt2_boxes, confidence=dt2_scores, class_id=dt2_classes)
  dt2_idx = custom_nms(dt2_detections, threshold=0.5)
  dt2_detections = dt2_detections[dt2_idx]

  box_annotator = sv.BoxAnnotator(color=sv.Color.RED, thickness=1)
  true_img = box_annotator.annotate(bgr_img.copy(), detections=true_detections)[:,:,::-1]
  dt1_img = box_annotator.annotate(bgr_img.copy(), detections=dt1_detections)[:,:,::-1]
  dt2_img = box_annotator.annotate(bgr_img.copy(), detections=dt2_detections)[:,:,::-1]

  axs[i,0].imshow(true_img)
  axs[i,0].axis('off')
  axs[i,1].imshow(dt1_img)
  axs[i,1].axis('off')
  axs[i,2].imshow(dt2_img)
  axs[i,2].axis('off')

axs[0,0].set_title('True Boxes')
axs[0,1].set_title('Detectree2 Filter = 0.1')
axs[0,2].set_title('Detectree2 Filter = 0.2')
plt.tight_layout()
plt.savefig(os.path.join(threshold_folder, 'Detectree2 Thresholds.jpg'))
plt.show()

In [ ]:
  dt2_idx = np.nonzero(dt_scores >= 0.2)
  dt2_boxes = dt_boxes[dt2_idx]
  dt2_scores = dt_scores[dt2_idx]
  dt2_classes = dt_classes[dt2_idx]

  # Apply custom non-max suppression
  dt2_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt2_idx = custom_nms(dt_detections, threshold=0.5)
  dt2_boxes = dt_boxes[dt2_idx]
  dt2_scores = dt_scores[dt2_idx]
  dt2_classes = dt_classes[dt2_idx]

# Main Metrics

In [ ]:
metrics_folder = '/content/drive/MyDrive/Sam+LiDAR/Experiments/Main Metrics'
'''
main_methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
                'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']
main_order = [(difficulty, method, name) for _, difficulty in test_imgs for method in main_methods for name, _ in test_imgs]

metrics = pd.read_csv('/content/Box Workflow Metrics.csv', index_col=[0,1,2], header=[0,1])
point_metrics = pd.read_csv('/content/Box + LiDAR Workflow Metrics.csv', index_col=[0,1,2], header=[0,1])

main_metrics = pd.concat((metrics, point_metrics)).loc[(['Easy','Medium','Hard'], main_methods),]
main_metrics.to_csv(os.path.join(metrics_folder, 'Main Metrics - All Results.csv'))
'''
main_metrics = pd.read_csv(os.path.join(metrics_folder, 'Main Metrics - All Results.csv'), index_col=[0,1,2], header=[0,1])
display(main_metrics)
print()
'''
diff_order = [(difficulty, method) for difficulty in ['Easy','Medium','Hard'] for method in main_methods]
main_diff_metrics = main_metrics.groupby(by=['Difficulty','Method']).mean().reindex(diff_order)
main_diff_metrics.to_csv(os.path.join(metrics_folder, 'Main Metrics - Avg by Difficulty.csv'))
'''
main_diff_metrics = pd.read_csv(os.path.join(metrics_folder, 'Main Metrics - Avg by Difficulty.csv'), index_col=[0,1], header=[0,1])
display(main_diff_metrics)
print()
'''
main_avg_metrics = main_metrics.groupby(by=['Method']).mean().reindex(main_methods)
main_avg_metrics.to_csv(os.path.join(metrics_folder, 'Main Metrics - Avg by Method.csv'))
'''
main_avg_metrics = pd.read_csv(os.path.join(metrics_folder, 'Main Metrics - Avg by Method.csv'), index_col=[0], header=[0,1])
display(main_avg_metrics)

In [ ]:
old_metrics_folder = '/content/drive/MyDrive/Sam+LiDAR/Experiments/Old Main Metrics'

old_metrics = pd.read_csv(os.path.join(old_metrics_folder, 'Main Metrics - All Results.csv'), index_col=[0,1,2], header=[0,1])
old_diff_metrics = pd.read_csv(os.path.join(old_metrics_folder, 'Main Metrics - Avg by Difficulty.csv'), index_col=[0,1], header=[0,1])
old_avg_metrics = pd.read_csv(os.path.join(old_metrics_folder, 'Main Metrics - Avg by Method.csv'), index_col=[0], header=[0,1])

lidar_methods = ['Manual + LiDAR', 'DeepForest + LiDAR', 'Detectree2 + LiDAR', 'Baseline Filtered']
difficulty_methods = [(difficulty, method) for difficulty in ['Easy','Medium','Hard'] for method in compare_methods]
difficulty_methods_names = [(difficulty, method, filename) for difficulty in ['Easy','Medium','Hard'] for method in compare_methods for filename, diff in test_imgs if diff == difficulty]

compare_metrics = main_metrics - old_metrics
compare_metrics = compare_metrics.loc[difficulty_methods_names]

compare_diff_metrics_1 = main_diff_metrics - old_diff_metrics
compare_diff_metrics_1 = compare_diff_metrics_1.loc[difficulty_methods]

compare_avg_metrics_1 = main_avg_metrics - old_avg_metrics
compare_avg_metrics_1 = compare_avg_metrics_1.loc[lidar_methods]

#display(compare_metrics)
#print()

compare_diff_metrics_2 = compare_metrics.groupby(by=['Difficulty','Method']).mean().reindex(compare_order)
display(compare_diff_metrics_1)
print()
display(compare_diff_metrics_2)
print()

compare_avg_metrics_2 = compare_metrics.groupby(by=['Method']).mean().reindex(compare_methods)
display(compare_avg_metrics_1)
print()
display(compare_avg_metrics_2)

In [ ]:
main_methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
                'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']
main_metrics.loc[('Medium', main_methods, 'DSNY_025_2018')]

In [ ]:
main_diff_metrics.loc[['Medium','Hard']].groupby(by='Method').mean().reindex(main_methods)

In [ ]:
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'), ('Segmentation','IoU')]

df = main_avg_metrics.loc[:,columns]

styles = {'Manual Boxes': [{'selector':'th','props':'background-color: white'},
                           {'selector':'td','props':'background-color: white'}],
          'Manual + LiDAR': [{'selector':'th','props':'background-color: white'},
                             {'selector':'td','props':'background-color: white'}],
          'DeepForest Boxes': [{'selector':'th','props':'background-color: #f2f2f2'},
                               {'selector':'td','props':'background-color: #f2f2f2'}],
          'DeepForest + LiDAR': [{'selector':'th','props':'background-color: #f2f2f2'},
                                 {'selector':'td','props':'background-color: #f2f2f2'}],
          'Detectree2 Boxes': [{'selector':'th','props':'background-color: white'},
                               {'selector':'td','props':'background-color: white'}],
          'Detectree2 + LiDAR': [{'selector':'th','props':'background-color: white'},
                                 {'selector':'td','props':'background-color: white'}],
          'Baseline': [{'selector':'th','props':'background-color: #f2f2f2'},
                       {'selector':'td','props':'background-color: #f2f2f2'}],
          'Baseline Filtered': [{'selector':'th','props':'background-color: #f2f2f2'},
                                {'selector':'td','props':'background-color: #f2f2f2'}]}


styler = df.style \
  .format('{:.1%}') \
  .set_table_styles(styles, axis=1) \
  .set_table_styles([{'selector':'th','props':'text-align: left'}], overwrite=False) \
  .hide(axis=0, names=True)

styler

In [ ]:
main_methods = ['Manual Boxes', 'Manual + LiDAR', 'DeepForest Boxes', 'DeepForest + LiDAR',
                'Detectree2 Boxes', 'Detectree2 + LiDAR', 'Baseline', 'Baseline Filtered']
stds = main_metrics.groupby(by='Method').std().reindex(main_methods)
stds_style = stds.style
stds_style.use(styler.export()).format('{:.1%}')
stds_style

In [ ]:
all_styler = main_metrics.loc['Easy'].style \
  .format('{:.1%}') \
  .set_table_styles([{'selector':'th','props':'text-align: left'}], overwrite=False) \
  .hide(axis=0, names=True)

all_styler

In [ ]:
all_styler = compare_metrics.loc['Easy'].style \
  .format('{:.1%}') \
  .set_table_styles([{'selector':'th','props':'text-align: left'}], overwrite=False) \
  .hide(axis=0, names=True)

all_styler

In [ ]:
df_easy = main_diff_metrics.loc['Easy'].style
df_easy.use(styler.export()).format('{:.2%}')
df_easy

In [ ]:
df_medium = main_diff_metrics.loc['Medium',columns].style
df_medium.use(styler.export()).format('{:.2%}')
df_medium

In [ ]:
df_hard = main_diff_metrics.loc['Hard',columns].style
df_hard.use(styler.export()).format('{:.2%}')
df_hard

# Alternative Metrics

In [ ]:

alt_metrics = pd.concat((metrics, filter_metrics, point_metrics.drop(labels='Detectree2 Filtered', level='Method')))
alt_methods = ['Hand-Drawn Boxes', 'Hand-Drawn Boxes + LiDAR',
               'DeepForest', 'DeepForest Filtered', 'DeepForest + LiDAR',
               'Detectree2', 'Detectree2 Filtered', 'Detectree2 + LiDAR',
               'Detectree2 Baseline', 'Detectree2 Baseline Filtered']
alt_order = [(difficulty, method) for difficulty in ['Easy','Medium','Hard'] for method in alt_methods]
alt_diff_metrics = alt_metrics.groupby(by=['Difficulty','Method']).mean().reindex(alt_order)
alt_diff_metrics.to_csv('Alternative Metrics by Difficulty.csv')

display(alt_diff_metrics)

print()

alt_avg_metrics = alt_metrics.groupby(by=['Method']).mean().reindex(alt_methods)
alt_avg_metrics.to_csv('Alternative Metrics.csv')

alt_avg_metrics = pd.read_csv('/content/Alternative Metrics.csv', index_col=[0], header=[0,1])
#alt_avg_metrics.rename(columns={'Pixel':'Segmentation'}, level=0, inplace=True)
display(alt_avg_metrics)

In [ ]:
box_mask_metrics = pd.read_csv('/content/Box to Mask Comparison.csv', index_col=[0,1,2], header=[0,1])
df = box_mask_metrics.groupby(by=['Method']).mean().reindex(['Hand-Drawn Boxes', 'DeepForest', 'Detectree2', 'Detectree2 Baseline'])
display(df)

In [ ]:
box_mask_metrics.groupby(by='Method').std().style.format('{:.1%}')

# Main Workflows with Images

In [ ]:
for filename, difficulty in [()]:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()
  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()
  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]
  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks_1 = dt_cnn_masks[dt_idx]
  dt_cnn_boxes_1 = dt_boxes.copy()

  # Segment using SAM-2
  hand_masks = segment_boxes(sam_predictor, hand_boxes, threshold=0.2)
  df_masks = segment_boxes(sam_predictor, df_boxes, threshold=0.2)
  dt_masks = segment_boxes(sam_predictor, dt_boxes, threshold=0.2)

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_1, hand_masks_1 = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes_1, df_masks_1 = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes_1, dt_masks_1 = dt_boxes[box_idx], dt_masks[mask_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter boxes
  idx = lidar_filter(hand_boxes, hand_coords, hand_labels, threshold=0.2)
  hand_boxes, hand_labels = hand_boxes[idx], hand_labels[idx]
  idx = lidar_filter(df_boxes, df_coords, df_labels, threshold=0.2)
  df_boxes, df_labels = df_boxes[idx], df_labels[idx]
  idx = lidar_filter(dt_boxes, dt_coords, dt_labels, threshold=0.2)
  dt_boxes, dt_labels = dt_boxes[idx], dt_labels[idx]
  dt_cnn_boxes_2, dt_cnn_masks_2 = dt_cnn_boxes_1[idx], dt_cnn_masks_1[idx]

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  if len(hand_boxes) > 0:
    hand_coords, hand_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes) > 0:
    df_coords, df_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes) > 0:
    dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2
  if len(hand_boxes) > 0:
    hand_masks = segment_box_points(sam_predictor, hand_boxes, hand_coords, hand_labels, threshold=0.2, iterative=True)
  else:
    hand_masks = np.empty((0,400,400))
  if len(df_boxes) > 0:
    df_masks = segment_box_points(sam_predictor, df_boxes, df_coords, df_labels, threshold=0.2, iterative=True)
  else:
    df_masks = np.empty((0,400,400))
  if len(dt_boxes) > 0:
    dt_masks = segment_box_points(sam_predictor, dt_boxes, dt_coords, dt_labels, threshold=0.2, iterative=True)
  else:
    dt_masks = np.empty((0,400,400))

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_2, hand_masks_2 = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes_2, df_masks_2 = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes_2, dt_masks_2 = dt_boxes[box_idx], dt_masks[mask_idx]

  true_detections = sv.Detections(xyxy=true_boxes,
                                  mask=true_masks,
                                  class_id=np.zeros(len(true_boxes), dtype='int'))

  hand_detections_1 = sv.Detections(xyxy=hand_boxes_1,
                                    mask=hand_masks_1,
                                    class_id=np.zeros(len(hand_boxes_1), dtype='int'))

  hand_detections_2 = sv.Detections(xyxy=hand_boxes_2,
                                    mask=hand_masks_2,
                                    class_id=np.zeros(len(hand_boxes_2), dtype='int'))

  df_detections_1 = sv.Detections(xyxy=df_boxes_1,
                                  mask=df_masks_1,
                                  class_id=np.zeros(len(df_boxes_1), dtype='int'))

  df_detections_2 = sv.Detections(xyxy=df_boxes_2,
                                  mask=df_masks_2,
                                  class_id=np.zeros(len(df_boxes_2), dtype='int'))

  dt_detections_1 = sv.Detections(xyxy=dt_boxes_1,
                                  mask=dt_masks_1,
                                  class_id=np.zeros(len(dt_boxes_1), dtype='int'))

  dt_detections_2 = sv.Detections(xyxy=dt_boxes_2,
                                  mask=dt_masks_2,
                                  class_id=np.zeros(len(dt_boxes_2), dtype='int'))

  dt_cnn_detections_1 = sv.Detections(xyxy=dt_cnn_boxes_1,
                                      mask=dt_cnn_masks_1,
                                      class_id=np.zeros(len(dt_cnn_boxes_1), dtype='int'))

  dt_cnn_detections_2 = sv.Detections(xyxy=dt_cnn_boxes_2,
                                      mask=dt_cnn_masks_2,
                                      class_id=np.zeros(len(dt_cnn_boxes_2), dtype='int'))

  box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.RED)
  poly_annotator = sv.PolygonAnnotator(thickness=1, color=sv.Color.RED)
  mask_annotator = sv.MaskAnnotator()
  true_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections)

  fig, axs = plt.subplots(nrows=3, ncols=4, figsize=(16, 12))
  hand_img_0 = box_annotator.annotate(bgr_img.copy(), detections=hand_detections_1)[:,:,::-1]
  df_img_0 = box_annotator.annotate(bgr_img.copy(), detections=df_detections_1)[:,:,::-1]
  dt_img_0 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_1)[:,:,::-1]
  dt_cnn_img_0 = box_annotator.annotate(bgr_img.copy(), detections=dt_cnn_detections_1)[:,:,::-1]

  hand_img_1 = mask_annotator.annotate(true_img.copy(), detections=hand_detections_1)[:,:,::-1]
  df_img_1 = mask_annotator.annotate(true_img.copy(), detections=df_detections_1)[:,:,::-1]
  dt_img_1 = mask_annotator.annotate(true_img.copy(), detections=dt_detections_1)[:,:,::-1]
  dt_cnn_img_1 = mask_annotator.annotate(true_img.copy(), detections=dt_cnn_detections_1)[:,:,::-1]

  hand_img_2 = mask_annotator.annotate(true_img.copy(), detections=hand_detections_2)[:,:,::-1]
  df_img_2 = mask_annotator.annotate(true_img.copy(), detections=df_detections_2)[:,:,::-1]
  dt_img_2 = mask_annotator.annotate(true_img.copy(), detections=dt_detections_2)[:,:,::-1]
  dt_cnn_img_2 = mask_annotator.annotate(true_img.copy(), detections=dt_cnn_detections_2)[:,:,::-1]

  axs[0,0].imshow(hand_img_0)
  axs[0,1].imshow(df_img_0)
  axs[0,2].imshow(dt_img_0)
  axs[0,3].imshow(dt_cnn_img_0)

  axs[1,0].imshow(hand_img_1)
  axs[1,1].imshow(df_img_1)
  axs[1,2].imshow(dt_img_1)
  axs[1,3].imshow(dt_cnn_img_1)

  axs[2,0].imshow(hand_img_2)
  axs[2,1].imshow(df_img_2)
  axs[2,2].imshow(dt_img_2)
  axs[2,3].imshow(dt_cnn_img_2)

  axs[0,0].set_title('Hand Drawn', fontsize=14)
  axs[0,1].set_title('DeepForest', fontsize=14)
  axs[0,2].set_title('Detectree-2', fontsize=14)
  axs[0,3].set_title('Detectree-2 Baseline', fontsize=14)
  axs[0,0].set_ylabel('Detections', fontsize=14)
  axs[1,0].set_ylabel('Box Prompts', fontsize=14)
  axs[2,0].set_ylabel('Box + LiDAR Prompts', fontsize=14)
  for i in range(3):
    for j in range(4):
      axs[i,j].set_frame_on(False)
      axs[i,j].get_xaxis().set_ticks([])
      axs[i,j].get_yaxis().set_ticks([])
  plt.tight_layout()
  plt.savefig(f'{filename} segmentation.jpg')
  plt.show()

In [ ]:
#@title Iterative Point Prompt predictions

test_imgs = [('2018_SJER_3_252000_4106000_image_234', 'Easy'), ('2018_SJER_3_254000_4108000_image_700', 'Easy'),
             ('2018_TEAK_3_318000_4107000_image_718', 'Easy'), ('DSNY_002_2018', 'Easy'), ('JERC_055_2018', 'Easy'),
             ('OSBS_051_2019', 'Easy'), ('ABBY_063_2019', 'Medium'), ('BONA_018_2019', 'Medium'), ('DSNY_025_2018', 'Medium'),
             ('SJER_025_2018', 'Medium'), ('TEAK_049_2018', 'Medium'), ('WREF_072_2019', 'Medium'), ('ABBY_076_2019', 'Hard'),
             ('BART_050_2019', 'Hard'), ('BONA_005_2019', 'Hard'), ('DELA_047_2019', 'Hard'), ('LENO_066_2019', 'Hard'),
             ('SERC_004_2019', 'Hard')]

viz_folder = '/content/drive/MyDrive/Sam+LiDAR/Visualizations/Manual + LiDAR Per Tree Predictions'

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Segment using SAM-2
  hand_masks = segment_boxes(sam_predictor, hand_boxes, threshold=0.2)

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_1, hand_masks_1 = hand_boxes[box_idx], hand_masks[mask_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  if len(hand_boxes) > 0:
    sample_coords, sample_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2
  if len(hand_boxes) > 0:
    hand_masks = segment_box_points(sam_predictor, hand_boxes, sample_coords, sample_labels, threshold=0.2, iterative=True)
  else:
    hand_masks = np.empty((0,400,400))

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_2, hand_masks_2 = hand_boxes[box_idx], hand_masks[mask_idx]

  true_detections = sv.Detections(xyxy=true_boxes,
                                  mask=true_masks,
                                  class_id=np.zeros(len(true_boxes), dtype='int'))

  hand_detections_1 = sv.Detections(xyxy=hand_boxes_1,
                                    mask=hand_masks_1,
                                    class_id=np.zeros(len(hand_boxes_2), dtype='int'))

  hand_detections_2 = sv.Detections(xyxy=hand_boxes_2,
                                    mask=hand_masks_2,
                                    class_id=np.zeros(len(hand_boxes_2), dtype='int'))

  box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.RED)
  poly_annotator = sv.PolygonAnnotator(thickness=1, color=sv.Color.RED)
  mask_annotator = sv.MaskAnnotator()
  true_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections)

  iou_1 = compute_iou(hand_masks_1, true_masks, reduction='none')
  iou_2 = compute_iou(hand_masks_2, true_masks, reduction='none')

  fig, axs = plt.subplots(nrows=len(true_detections), ncols=3, figsize=(12,4*len(true_detections)))
  # This isn't how f-strings are supposed to work, but for some reason they interact strangely with matplotlib.suptitle
  if len(filename) == 13:
    fig.suptitle(f'{filename:<80} {"Mean Box IoU:"} {np.mean(iou_1):.2f} {"Mean LiDAR IoU":>55} {np.mean(iou_2):.2f}', ha='left', x=0.03, y=1)
  elif len(filename) == 36:
    fig.suptitle(f'{filename:<57} {"Mean Box IoU:"} {np.mean(iou_1):.2f} {"Mean LiDAR IoU":>55} {np.mean(iou_2):.2f}', ha='left', x=0.03, y=1)

  if len(hand_detections_2) > 1:
    axs[0,0].set_title('Manual + LiDAR Prompts')
    for i in range(len(hand_detections_2)):
      true_img = poly_annotator.annotate(scene=bgr_img.copy(), detections=true_detections[i])
      box_img = mask_annotator.annotate(scene=true_img.copy(), detections=hand_detections_1[i])[:,:,::-1]
      lidar_img = mask_annotator.annotate(scene=true_img.copy(), detections=hand_detections_2[i])[:,:,::-1]
      axs[i,0] = show_as_points(rgb_img, true_detections[i], sample_coords[i:i+1], sample_labels[i:i+1], ax=axs[i,0],
                                show_positive=True, show_negative=True, show_boxes=True, marker_size=25)
      axs[i,1].imshow(box_img)
      axs[i,2].imshow(lidar_img)
      axs[i,1].axis('off')
      axs[i,2].axis('off')
      axs[i,1].set_title(f'Box IoU: {iou_1[i]:.2f}')
      axs[i,2].set_title(f'LiDAR IoU: {iou_2[i]:.2f}')

  else:
    axs[0].set_title('Manual + LiDAR Prompts')
    true_img = poly_annotator.annotate(scene=bgr_img.copy(), detections=true_detections)
    box_img = mask_annotator.annotate(scene=true_img.copy(), detections=hand_detections_1)[:,:,::-1]
    lidar_img = mask_annotator.annotate(scene=true_img.copy(), detections=hand_detections_2)[:,:,::-1]
    axs[0] = show_as_points(rgb_img, true_detections, sample_coords, sample_labels, ax=axs[0],
                            show_positive=True, show_negative=True, show_boxes=True, marker_size=25)
    axs[1].imshow(box_img)
    axs[2].imshow(lidar_img)
    axs[1].axis('off')
    axs[2].axis('off')
    axs[1].set_title(f'Box IoU: {iou_1[0]:.2f}')
    axs[2].set_title(f'LiDAR IoU: {iou_2[0]:.2f}')

  plt.tight_layout()
  plt.savefig(os.path.join(viz_folder, f'{filename}.jpg'))
  plt.show()

In [ ]:
for filename, difficulty in [('2018_SJER_3_252000_4106000_image_234', 'Easy')]:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()

  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]

  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]

  # Use boxes to label LiDAR points
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter boxes
  idx = lidar_filter(dt_boxes, dt_coords, dt_labels, threshold=0.2)
  dt_boxes_filtered, dt_labels_filtered = dt_boxes[idx], dt_labels[idx]

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  if len(dt_boxes) > 0:
    dt_coords_sampled, dt_labels_sampled = sample_points(dt_coords, dt_labels_filtered, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
    dt_coords_uniform, dt_labels_uniform = sample_points(dt_coords, dt_labels_filtered, pos_samples=pos, neg_samples=neg, distance_weight=False, seed=seed)

  # Trick show_as_mask into using color by adding extra row of labels to dt_labels_filtered (all zeros)
  dt_labels_trick = np.concatenate([dt_labels_filtered, np.zeros(dt_labels_filtered.shape, dtype='int')])
  dt_boxes_trick = np.concatenate([dt_boxes_filtered, [[-1,-1,-1,-1]]])

dt_detections = sv.Detections(xyxy=dt_boxes, class_id=np.zeros(len(dt_boxes), dtype='int'))
dt_detections_filtered = sv.Detections(xyxy=dt_boxes_filtered, class_id=np.zeros(len(dt_boxes_filtered), dtype='int'))
dt_detections_trick = sv.Detections(xyxy=dt_boxes_trick, class_id=np.zeros(len(dt_boxes_trick), dtype='int'))

#show_as_mask(rgb_img, dt_detections, dt_coords, dt_labels, show_boxes=True, save=True)
show_as_mask(rgb_img, dt_detections_trick, dt_coords, dt_labels_trick, show_boxes=True, save=True)
show_as_points(rgb_img, dt_detections_filtered, dt_coords_uniform, dt_labels_uniform, show_negative=True, save=True)
#show_as_points(rgb_img, dt_detections_filtered, dt_coords_sampled, dt_labels_sampled, show_negative=True, save=True)

In [ ]:
import rasterio
import supervision as sv
from supervision.detection.utils import box_iou_batch, polygon_to_mask

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  true_detections = sv.Detections(xyxy=true_boxes,
                                  mask=true_masks,
                                  class_id=np.zeros(len(true_boxes), dtype='int'))

  poly_annotator = sv.PolygonAnnotator(thickness=1, color=sv.Color.RED)

  fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))
  true_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections)[:,:,::-1]
  axs.imshow(true_img)
  axs.axis('off')

  plt.tight_layout()
  plt.savefig(f'{filename} segmentation.jpg')
  plt.show()

**Selected Images**

**Easy:** 2018_TEAK_3_318000_4107000_image_718

**LiDAR labels:**

Manual: 0

DeepForest: 0

Detectree2: 5?

**Medium:** DSNY_025_2018 or WREF_072_2019

**LiDAR labels:**

Manual: 35 or 5

DeepForest: 43 or 14

Detectree2: 4 or 5

**Hard:** DELA_047_2019

**LiDAR labels:**

Manual: 37

DeepForest: 11

Detectree2: 1

In [ ]:
for filename, difficulty in [('DSNY_025_2018', 'Medium')]:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()
  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()
  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]
  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks_1 = dt_cnn_masks[dt_idx]
  dt_cnn_boxes_1 = dt_boxes.copy()

  # Segment using SAM-2
  hand_masks = segment_boxes(sam_predictor, hand_boxes, threshold=0.2)
  df_masks = segment_boxes(sam_predictor, df_boxes, threshold=0.2)
  dt_masks = segment_boxes(sam_predictor, dt_boxes, threshold=0.2)

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_1, hand_masks_1 = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes_1, df_masks_1 = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes_1, dt_masks_1 = dt_boxes[box_idx], dt_masks[mask_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter boxes
  idx = lidar_filter(hand_boxes, hand_coords, hand_labels, threshold=0.2)
  hand_boxes, hand_labels = hand_boxes[idx], hand_labels[idx]
  idx = lidar_filter(df_boxes, df_coords, df_labels, threshold=0.2)
  df_boxes, df_labels = df_boxes[idx], df_labels[idx]
  idx = lidar_filter(dt_boxes, dt_coords, dt_labels, threshold=0.2)
  dt_boxes, dt_labels = dt_boxes[idx], dt_labels[idx]
  dt_cnn_boxes_2, dt_cnn_masks_2 = dt_cnn_boxes_1[idx], dt_cnn_masks_1[idx]

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  if len(hand_boxes) > 0:
    hand_coords, hand_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes) > 0:
    df_coords, df_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes) > 0:
    dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2
  if len(hand_boxes) > 0:
    hand_masks = segment_box_points(sam_predictor, hand_boxes, hand_coords, hand_labels, threshold=0.2, iterative=True)
  else:
    hand_masks = np.empty((0,400,400))
  if len(df_boxes) > 0:
    df_masks = segment_box_points(sam_predictor, df_boxes, df_coords, df_labels, threshold=0.2, iterative=True)
  else:
    df_masks = np.empty((0,400,400))
  if len(dt_boxes) > 0:
    dt_masks = segment_box_points(sam_predictor, dt_boxes, dt_coords, dt_labels, threshold=0.2, iterative=True)
  else:
    dt_masks = np.empty((0,400,400))

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_2, hand_masks_2 = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes_2, df_masks_2 = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes_2, dt_masks_2 = dt_boxes[box_idx], dt_masks[mask_idx]

  true_detections = sv.Detections(xyxy=true_boxes,
                                  mask=true_masks,
                                  class_id=np.zeros(len(true_boxes), dtype='int'))

  hand_detections_1 = sv.Detections(xyxy=hand_boxes_1,
                                    mask=hand_masks_1,
                                    class_id=np.zeros(len(hand_boxes_1), dtype='int'))

  hand_detections_2 = sv.Detections(xyxy=hand_boxes_2,
                                    mask=hand_masks_2,
                                    class_id=np.zeros(len(hand_boxes_2), dtype='int'))

  df_detections_1 = sv.Detections(xyxy=df_boxes_1,
                                  mask=df_masks_1,
                                  class_id=np.zeros(len(df_boxes_1), dtype='int'))

  df_detections_2 = sv.Detections(xyxy=df_boxes_2,
                                  mask=df_masks_2,
                                  class_id=np.zeros(len(df_boxes_2), dtype='int'))

  dt_detections_1 = sv.Detections(xyxy=dt_boxes_1,
                                  mask=dt_masks_1,
                                  class_id=np.zeros(len(dt_boxes_1), dtype='int'))

  dt_detections_2 = sv.Detections(xyxy=dt_boxes_2,
                                  mask=dt_masks_2,
                                  class_id=np.zeros(len(dt_boxes_2), dtype='int'))

  dt_cnn_detections_1 = sv.Detections(xyxy=dt_cnn_boxes_1,
                                      mask=dt_cnn_masks_1,
                                      class_id=np.zeros(len(dt_cnn_boxes_1), dtype='int'))

  dt_cnn_detections_2 = sv.Detections(xyxy=dt_cnn_boxes_2,
                                      mask=dt_cnn_masks_2,
                                      class_id=np.zeros(len(dt_cnn_boxes_2), dtype='int'))

  box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.BLACK)
  poly_annotator = sv.PolygonAnnotator(thickness=1, color=sv.Color.RED)
  mask_annotator = sv.MaskAnnotator()
  true_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections)

  fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(16, 8))
  hand_img_1 = box_annotator.annotate(true_img.copy(), detections=hand_detections_1)
  df_img_1 = box_annotator.annotate(true_img.copy(), detections=df_detections_1)
  dt_img_1 = box_annotator.annotate(true_img.copy(), detections=dt_detections_1)
  dt_cnn_img_1 = box_annotator.annotate(true_img.copy(), detections=dt_cnn_detections_1)

  hand_img_1 = mask_annotator.annotate(hand_img_1.copy(), detections=hand_detections_1)[:,:,::-1]
  df_img_1 = mask_annotator.annotate(df_img_1.copy(), detections=df_detections_1)[:,:,::-1]
  dt_img_1 = mask_annotator.annotate(dt_img_1.copy(), detections=dt_detections_1)[:,:,::-1]
  dt_cnn_img_1 = mask_annotator.annotate(dt_cnn_img_1.copy(), detections=dt_cnn_detections_1)[:,:,::-1]

  hand_img_2 = box_annotator.annotate(true_img.copy(), detections=hand_detections_2)
  df_img_2 = box_annotator.annotate(true_img.copy(), detections=df_detections_2)
  dt_img_2 = box_annotator.annotate(true_img.copy(), detections=dt_detections_2)
  dt_cnn_img_2 = box_annotator.annotate(true_img.copy(), detections=dt_cnn_detections_2)

  hand_img_2 = mask_annotator.annotate(hand_img_2.copy(), detections=hand_detections_2)[:,:,::-1]
  df_img_2 = mask_annotator.annotate(df_img_2.copy(), detections=df_detections_2)[:,:,::-1]
  dt_img_2 = mask_annotator.annotate(dt_img_2.copy(), detections=dt_detections_2)[:,:,::-1]
  dt_cnn_img_2 = mask_annotator.annotate(dt_cnn_img_2.copy(), detections=dt_cnn_detections_2)[:,:,::-1]

  hand_img_2 = show_as_points(hand_img_2, hand_detections_2, hand_coords[35:36], hand_labels[35:36], ax=axs[1,0], show_negative=True, marker_size=50)
  df_img_2 = show_as_points(df_img_2, df_detections_2, df_coords[43:44], df_labels[43:44], ax=axs[1,1], show_negative=True, marker_size=50)
  dt_img_2 = show_as_points(dt_img_2, dt_detections_2, dt_coords[4:5], dt_labels[4:5], ax=axs[1,2], show_negative=True, marker_size=50)

  axs[0,0].imshow(hand_img_1)
  axs[0,1].imshow(df_img_1)
  axs[0,2].imshow(dt_img_1)
  axs[0,3].imshow(dt_cnn_img_1)
  axs[1,3].imshow(dt_cnn_img_2)

  axs[0,0].set_title('Manual', fontsize=14)
  axs[0,1].set_title('DeepForest', fontsize=14)
  axs[0,2].set_title('Detectree2', fontsize=14)
  axs[0,3].set_title('Baseline', fontsize=14)
  axs[0,0].set_ylabel('Boxes', fontsize=14)
  axs[1,0].axis('on')
  axs[1,0].set_ylabel('Boxes + LiDAR', fontsize=14)
  for i in range(2):
    for j in range(4):
      axs[i,j].set_frame_on(False)
      axs[i,j].get_xaxis().set_ticks([])
      axs[i,j].get_yaxis().set_ticks([])
  plt.tight_layout()
  plt.savefig(f'{filename} segmentation.jpg')
  plt.show()

In [ ]:
tree_idx = {'2018_TEAK_3_318000_4107000_image_718':[0,0,0], 'WREF_072_2019': [5,14,15], 'DELA_047_2019': [37,11,1]}
#fig, axs = plt.subplots(nrows=3, ncols=6, figsize=(24, 12))
fig, axs = plt.subplots(nrows=6, ncols=3, figsize=(5,10))

for i, (filename, difficulty) in enumerate([('2018_TEAK_3_318000_4107000_image_718', 'Easy'),
                                            ('WREF_072_2019', 'Medium'),
                                            ('DELA_047_2019', 'Hard')]):
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)
  id = tree_idx[filename]

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()
  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()
  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]
  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks_1 = dt_cnn_masks[dt_idx]
  dt_cnn_boxes_1 = dt_boxes.copy()

  # Segment using SAM-2
  hand_masks = segment_boxes(sam_predictor, hand_boxes, threshold=0.2)
  df_masks = segment_boxes(sam_predictor, df_boxes, threshold=0.2)
  dt_masks = segment_boxes(sam_predictor, dt_boxes, threshold=0.2)

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_1, hand_masks_1 = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes_1, df_masks_1 = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes_1, dt_masks_1 = dt_boxes[box_idx], dt_masks[mask_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter boxes
  idx = lidar_filter(hand_boxes, hand_coords, hand_labels, threshold=0.2)
  hand_boxes, hand_labels = hand_boxes[idx], hand_labels[idx]
  idx = lidar_filter(df_boxes, df_coords, df_labels, threshold=0.2)
  df_boxes, df_labels = df_boxes[idx], df_labels[idx]
  idx = lidar_filter(dt_boxes, dt_coords, dt_labels, threshold=0.2)
  dt_boxes, dt_labels = dt_boxes[idx], dt_labels[idx]
  dt_cnn_boxes_2, dt_cnn_masks_2 = dt_cnn_boxes_1[idx], dt_cnn_masks_1[idx]

  # Sample LiDAR points for SAM-2
  pos = 10
  neg = 10
  if len(hand_boxes) > 0:
    hand_coords, hand_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(df_boxes) > 0:
    df_coords, df_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
  if len(dt_boxes) > 0:
    dt_coords, dt_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

  # Segment using SAM-2
  if len(hand_boxes) > 0:
    hand_masks = segment_box_points(sam_predictor, hand_boxes, hand_coords, hand_labels, threshold=0.2, iterative=True)
  else:
    hand_masks = np.empty((0,400,400))
  if len(df_boxes) > 0:
    df_masks = segment_box_points(sam_predictor, df_boxes, df_coords, df_labels, threshold=0.2, iterative=True)
  else:
    df_masks = np.empty((0,400,400))
  if len(dt_boxes) > 0:
    dt_masks = segment_box_points(sam_predictor, dt_boxes, dt_coords, dt_labels, threshold=0.2, iterative=True)
  else:
    dt_masks = np.empty((0,400,400))

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_2, hand_masks_2 = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes_2, df_masks_2 = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes_2, dt_masks_2 = dt_boxes[box_idx], dt_masks[mask_idx]

  true_detections = sv.Detections(xyxy=true_boxes,
                                  mask=true_masks,
                                  class_id=np.zeros(len(true_boxes), dtype='int'))

  hand_detections_1 = sv.Detections(xyxy=hand_boxes_1,
                                    mask=hand_masks_1,
                                    class_id=np.zeros(len(hand_boxes_1), dtype='int'))

  hand_detections_2 = sv.Detections(xyxy=hand_boxes_2,
                                    mask=hand_masks_2,
                                    class_id=np.zeros(len(hand_boxes_2), dtype='int'))

  df_detections_1 = sv.Detections(xyxy=df_boxes_1,
                                  mask=df_masks_1,
                                  class_id=np.zeros(len(df_boxes_1), dtype='int'))

  df_detections_2 = sv.Detections(xyxy=df_boxes_2,
                                  mask=df_masks_2,
                                  class_id=np.zeros(len(df_boxes_2), dtype='int'))

  dt_detections_1 = sv.Detections(xyxy=dt_boxes_1,
                                  mask=dt_masks_1,
                                  class_id=np.zeros(len(dt_boxes_1), dtype='int'))

  dt_detections_2 = sv.Detections(xyxy=dt_boxes_2,
                                  mask=dt_masks_2,
                                  class_id=np.zeros(len(dt_boxes_2), dtype='int'))
  '''
  dt_cnn_detections_1 = sv.Detections(xyxy=dt_cnn_boxes_1,
                                      mask=dt_cnn_masks_1,
                                      class_id=np.zeros(len(dt_cnn_boxes_1), dtype='int'))

  dt_cnn_detections_2 = sv.Detections(xyxy=dt_cnn_boxes_2,
                                      mask=dt_cnn_masks_2,
                                      class_id=np.zeros(len(dt_cnn_boxes_2), dtype='int'))
  '''
  if i == 0:
    box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.BLACK)
  else:
    box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.WHITE)
  poly_annotator = sv.PolygonAnnotator(thickness=2, color=sv.Color.RED)
  mask_annotator = sv.MaskAnnotator()
  true_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections)

  # Visualize
  hand_img_1 = box_annotator.annotate(true_img.copy(), detections=hand_detections_1)
  df_img_1 = box_annotator.annotate(true_img.copy(), detections=df_detections_1)
  dt_img_1 = box_annotator.annotate(true_img.copy(), detections=dt_detections_1)
  #dt_cnn_img_1 = box_annotator.annotate(true_img.copy(), detections=dt_cnn_detections_1)

  hand_img_1 = mask_annotator.annotate(hand_img_1.copy(), detections=hand_detections_1)[:,:,::-1]
  df_img_1 = mask_annotator.annotate(df_img_1.copy(), detections=df_detections_1)[:,:,::-1]
  dt_img_1 = mask_annotator.annotate(dt_img_1.copy(), detections=dt_detections_1)[:,:,::-1]
  #dt_cnn_img_1 = mask_annotator.annotate(dt_cnn_img_1.copy(), detections=dt_cnn_detections_1)[:,:,::-1]

  hand_img_2 = box_annotator.annotate(true_img.copy(), detections=hand_detections_2)
  df_img_2 = box_annotator.annotate(true_img.copy(), detections=df_detections_2)
  dt_img_2 = box_annotator.annotate(true_img.copy(), detections=dt_detections_2)
  #dt_cnn_img_2 = box_annotator.annotate(true_img.copy(), detections=dt_cnn_detections_2)

  hand_img_2 = mask_annotator.annotate(hand_img_2.copy(), detections=hand_detections_2)[:,:,::-1]
  df_img_2 = mask_annotator.annotate(df_img_2.copy(), detections=df_detections_2)[:,:,::-1]
  dt_img_2 = mask_annotator.annotate(dt_img_2.copy(), detections=dt_detections_2)[:,:,::-1]
  #dt_cnn_img_2 = mask_annotator.annotate(dt_cnn_img_2.copy(), detections=dt_cnn_detections_2)[:,:,::-1]
  '''
  hand_img_2 = show_as_points(hand_img_2, hand_detections_2, hand_coords[id[0]:id[0]+1], hand_labels[id[0]:id[0]+1], ax=axs[i,1], show_negative=True, marker_size=50)
  df_img_2 = show_as_points(df_img_2, df_detections_2, df_coords[id[1]:id[1]+1], df_labels[id[1]:id[1]+1], ax=axs[i,3], show_negative=True, marker_size=50)
  dt_img_2 = show_as_points(dt_img_2, dt_detections_2, dt_coords[id[2]:id[2]+1], dt_labels[id[2]:id[2]+1], ax=axs[i,5], show_negative=True, marker_size=50)

  axs[i,0].imshow(hand_img_1)
  axs[i,2].imshow(df_img_1)
  axs[i,4].imshow(dt_img_1)
  #axs[0,3].imshow(dt_cnn_img_1)
  #axs[1,3].imshow(dt_cnn_img_2)
  if i == 0:
    axs[i,0].set_ylabel(f'{filename[:-10]}\n{filename[-10:]}', fontsize=14)
  else:
    axs[i,0].set_ylabel(filename, fontsize=14)
  '''
  hand_img_2 = show_as_points(hand_img_2, hand_detections_2, hand_coords[id[0]:id[0]+1], hand_labels[id[0]:id[0]+1], ax=axs[1,i], show_negative=True, marker_size=10)
  df_img_2 = show_as_points(df_img_2, df_detections_2, df_coords[id[1]:id[1]+1], df_labels[id[1]:id[1]+1], ax=axs[3,i], show_negative=True, marker_size=10)
  dt_img_2 = show_as_points(dt_img_2, dt_detections_2, dt_coords[id[2]:id[2]+1], dt_labels[id[2]:id[2]+1], ax=axs[5,i], show_negative=True, marker_size=10)
  axs[0,i].imshow(hand_img_1)
  axs[2,i].imshow(df_img_1)
  axs[4,i].imshow(dt_img_1)

'''
axs[0,0].set_title('Manual Boxes', fontsize=14)
axs[0,1].set_title('Manual + LiDAR', fontsize=14)
axs[0,2].set_title('DeepForest Boxes', fontsize=14)
axs[0,3].set_title('DeepForest + LiDAR', fontsize=14)
axs[0,4].set_title('Detectree2 Boxes', fontsize=14)
axs[0,5].set_title('Detectree2 + LiDAR', fontsize=14)
for i in range(3):
  for j in range(6):
    axs[i,j].set_frame_on(False)
    axs[i,j].get_xaxis().set_ticks([])
    axs[i,j].get_yaxis().set_ticks([])
'''
axs[0,0].set_title('Easy', fontsize=12)
axs[0,1].set_title('Medium', fontsize=12)
axs[0,2].set_title('Hard', fontsize=12)
axs[0,0].set_ylabel('Manual Boxes', fontsize=9)
axs[1,0].set_ylabel('Manual + LiDAR', fontsize=9)
axs[2,0].set_ylabel('DeepForest Boxes', fontsize=9)
axs[3,0].set_ylabel('DeepForest + LiDAR', fontsize=9)
axs[4,0].set_ylabel('Detectree2 Boxes', fontsize=9)
axs[5,0].set_ylabel('Detectree2 + LiDAR', fontsize=9)
for i in range(6):
  for j in range(3):
    axs[i,j].axis(True)
    axs[i,j].set_frame_on(False)
    axs[i,j].get_xaxis().set_ticks([])
    axs[i,j].get_yaxis().set_ticks([])

plt.tight_layout()
plt.savefig(f'qualitative_segmentation.jpg')
plt.show()

In [ ]:
for i in range(len(dt_detections_2)):
  print(i)
  show_as_points(rgb_img, dt_detections_2, dt_coords[i:i+1], dt_labels[i:i+1], show_negative=True, marker_size=50)

In [ ]:
main_diff_metrics

In [ ]:
main_diff_metrics[('Detection','F1')]

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

plt.figure(figsize=(12, 7))
sns.set_style('whitegrid')

ax = sns.boxplot(data=main_metrics['Detection'], x='Method',y='F1', hue='Difficulty', width=0.2,
                  #boxprops={'facecolor': 'black'},
                  showmeans=True, meanline=False, meanprops={'marker': 'o', 'mfc': 'white', 'mec': 'black', 'ms': 10})
xmin, xmax = ax.get_xlim()
ax.set_xlim(xmin - 0.4, xmax + 0.4)
sns.despine(left=True)
plt.tight_layout()
plt.show()

In [ ]:
true_idx, hand_idx = hungarian_matching(true_boxes, hand_boxes_1, threshold=0.5)
hand_box_precision, hand_box_recall, hand_box_f1 = segmentation_metrics(true_masks[true_idx], hand_masks_1[hand_idx])
hand_box_iou = compute_iou(true_masks[true_idx], hand_masks_1[hand_idx])

true_idx, df_idx = hungarian_matching(true_boxes, df_boxes_1, threshold=0.5)
df_box_precision, df_box_recall, df_box_f1 = segmentation_metrics(true_masks[true_idx], df_masks_1[df_idx])
df_box_iou = compute_iou(true_masks[true_idx], df_masks_1[df_idx])

true_idx, dt_idx = hungarian_matching(true_boxes, dt_boxes_1, threshold=0.5)
dt_box_precision, dt_box_recall, dt_box_f1 = segmentation_metrics(true_masks[true_idx], dt_masks_1[dt_idx])
dt_box_iou = compute_iou(true_masks[true_idx], dt_masks_1[dt_idx])

In [ ]:
sorted(hand_box_recall)

In [ ]:
hand_box_precision

In [ ]:
hand_box_iou

In [ ]:
compute_iou(true_boxes[true_idx], hand_boxes_1[hand_idx])

In [ ]:
print('Manual Boxes')
fig, ax = plt.subplots(figsize=(4,4))
tree_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[0])
tree_img = box_annotator.annotate(tree_img, detections=hand_detections_1[0])
tree_img = mask_annotator.annotate(tree_img, detections=hand_detections_1[0])[:,:,::-1]
ax.imshow(tree_img)
plt.axis('off')
plt.tight_layout()
#plt.savefig(f'{filename}_Manual_Boxes.pdf')
plt.show()

iou = compute_iou(true_detections[0].mask, hand_detections_1[0].mask)
precision, recall, f1 = segmentation_metrics(true_detections[0].mask, hand_detections_1[0].mask)
print(f'Precision: {precision[0] * 100:.1f}%   Recall: {recall[0] * 100:.1f}%   F1: {f1[0] * 100:.1f}%   IoU: {iou[0] * 100:.1f}%')

In [ ]:
print('Manual + LiDAR')
fig, axs = plt.subplots(figsize=(4,4))
tree_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[0])
tree_img = box_annotator.annotate(tree_img, detections=hand_detections_2[0])
tree_img = mask_annotator.annotate(tree_img, detections=hand_detections_2[0])[:,:,::-1]
tree_img = show_as_points(tree_img, hand_detections_2[0], hand_coords[0:1], hand_labels[0:1],
                          show_negative=True, marker_size=50, ax=axs)
plt.tight_layout()
#plt.savefig(f'{filename}_Manual+LiDAR.pdf')
plt.show()

iou = compute_iou(true_detections[0].mask, hand_detections_2[0].mask)
precision, recall, f1 = segmentation_metrics(true_detections[0].mask, hand_detections_2[0].mask)
print(f'Precision: {precision[0] * 100:.1f}%   Recall: {recall[0] * 100:.1f}%   F1: {f1[0] * 100:.1f}%   IoU: {iou[0] * 100:.1f}%')
print()

In [ ]:
print('DeepForest Boxes')
fig, ax = plt.subplots(figsize=(4,4))
tree_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[0])
tree_img = box_annotator.annotate(tree_img, detections=df_detections_1[0])
tree_img = mask_annotator.annotate(tree_img, detections=df_detections_1[0])[:,:,::-1]
ax.imshow(tree_img)
plt.axis('off')
plt.tight_layout()
#plt.savefig(f'{filename}_DeepForest_Boxes.pdf')
plt.show()

iou = compute_iou(true_detections[0].mask, df_detections_1[0].mask)
precision, recall, f1 = segmentation_metrics(true_detections[0].mask, df_detections_1[0].mask)
print(f'Precision: {precision[0] * 100:.1f}%   Recall: {recall[0] * 100:.1f}%   F1: {f1[0] * 100:.1f}%   IoU: {iou[0] * 100:.1f}%')

In [ ]:
print('DeepForest + LiDAR')
fig, axs = plt.subplots(figsize=(4,4))
tree_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[0])
tree_img = box_annotator.annotate(tree_img, detections=df_detections_2[0])
tree_img = mask_annotator.annotate(tree_img, detections=df_detections_2[0])[:,:,::-1]
tree_img = show_as_points(tree_img, df_detections_2[0], hand_coords[0:1], hand_labels[0:1],
                          show_negative=True, marker_size=50, ax=axs)
plt.tight_layout()
#plt.savefig(f'{filename}_DeepForest+LiDAR.pdf')
plt.show()

iou = compute_iou(true_detections[0].mask, df_detections_2[0].mask)
precision, recall, f1 = segmentation_metrics(true_detections[0].mask, df_detections_2[0].mask)
print(f'Precision: {precision[0] * 100:.1f}%   Recall: {recall[0] * 100:.1f}%   F1: {f1[0] * 100:.1f}%   IoU: {iou[0] * 100:.1f}%')
print()

In [ ]:
print('Detectree2 Boxes')
fig, ax = plt.subplots(figsize=(4,4))
tree_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[0])
tree_img = box_annotator.annotate(tree_img, detections=dt_detections_1[0])
tree_img = mask_annotator.annotate(tree_img, detections=dt_detections_1[0])[:,:,::-1]
ax.imshow(tree_img)
plt.axis('off')
plt.tight_layout()
plt.savefig(f'{filename}_Detectree2_Boxes.png')
plt.show()

iou = compute_iou(true_detections[0].mask, dt_detections_1[0].mask)
precision, recall, f1 = segmentation_metrics(true_detections[0].mask, dt_detections_1[0].mask)
print(f'Precision: {precision[0] * 100:.1f}%   Recall: {recall[0] * 100:.1f}%   F1: {f1[0] * 100:.1f}%   IoU: {iou[0] * 100:.1f}%')

In [ ]:
print('Detectree2 + LiDAR')
fig, axs = plt.subplots(figsize=(4,4))
tree_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[0])
tree_img = box_annotator.annotate(tree_img, detections=dt_detections_2[0])
tree_img = mask_annotator.annotate(tree_img, detections=dt_detections_2[0])[:,:,::-1]
tree_img = show_as_points(tree_img, dt_detections_2[0], hand_coords[0:1], hand_labels[0:1],
                          show_negative=True, marker_size=50, ax=axs)
plt.tight_layout()
#plt.savefig(f'{filename}_Detectree2+LiDAR.png')
plt.show()

iou = compute_iou(true_detections[0].mask, dt_detections_2[0].mask)
precision, recall, f1 = segmentation_metrics(true_detections[0].mask, dt_detections_2[0].mask)
print(f'Precision: {precision[0] * 100:.1f}%   Recall: {recall[0] * 100:.1f}%   F1: {f1[0] * 100:.1f}%   IoU: {iou[0] * 100:.1f}%')
print()

In [ ]:
print('Detectree2 Boxes')
fig, axs = plt.subplots(1, 2, figsize=(8,4))
tree_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[0])
tree_img = box_annotator.annotate(tree_img, detections=dt_detections_1)
tree_img = mask_annotator.annotate(tree_img, detections=dt_detections_1)[:,:,::-1]
axs[0].axis('off')
axs[0].imshow(tree_img)

iou_1 = compute_iou(true_detections[0].mask, dt_detections_1[5].mask)
precision_1, recall_1, f1_1 = segmentation_metrics(true_detections[0].mask, dt_detections_1[5].mask)

print('Detectree2 + LiDAR')
tree_img = poly_annotator.annotate(bgr_img.copy(), detections=true_detections[0])
tree_img = box_annotator.annotate(tree_img, detections=dt_detections_2[0])
tree_img = mask_annotator.annotate(tree_img, detections=dt_detections_2[0])[:,:,::-1]
tree_img = show_as_points(tree_img, dt_detections_2[0], hand_coords[0:1], hand_labels[0:1],
                          show_negative=True, marker_size=50, ax=axs[1])
plt.tight_layout()
plt.savefig(f'{filename}_Detectree2.png')
plt.show()

iou_2 = compute_iou(true_detections[0].mask, dt_detections_2[0].mask)
precision_2, recall_2, f1_2 = segmentation_metrics(true_detections[0].mask, dt_detections_2[0].mask)

print(f'Precision: {precision_1[0] * 100:.1f}%   Recall: {recall_1[0] * 100:.1f}%   F1: {f1_1[0] * 100:.1f}%   IoU: {iou_1[0] * 100:.1f}%')
print(f'Precision: {precision_2[0] * 100:.1f}%   Recall: {recall_2[0] * 100:.1f}%   F1: {f1_2[0] * 100:.1f}%   IoU: {iou_2[0] * 100:.1f}%')

In [ ]:
len(hand_labels), len(df_labels), len(dt_labels)

In [ ]:
for i in range(len(dt_labels)):
  print(i)
  show_as_points(true_img[:,:,::-1].copy(), dt_detections_2, dt_coords[i:i+1], dt_labels[i:i+1], show_boxes=True)

In [ ]:
for i in range(len(df_labels)):
  print(i)
  show_as_points(true_img[:,:,::-1].copy(), df_detections_2, df_coords[i:i+1], df_labels[i:i+1], show_boxes=True)

In [ ]:
for i in range(15,len(hand_labels)):
  print(i)
  show_as_points(true_img[:,:,::-1].copy(), hand_detections_2, hand_coords[i:i+1], hand_labels[i:i+1], show_boxes=True)

In [ ]:
filename = '2018_SJER_3_252000_4106000_image_234'
img = ds.get_image(filename)
rgb_img = img['rgb']
bgr_img = rgb_img[:,:,::-1].copy()
box_annotator = sv.BoxAnnotator(color=sv.Color.RED, thickness=1)

# Generate automatic bounding boxes using Detectree2
dt_outputs = dt_model(bgr_img)
dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
dt_scores = dt_outputs['instances'].scores.cpu().numpy()
dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
dt_detections_1 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)

# Filter with confidence threshold of 0.2
dt_idx = np.nonzero(dt_scores >= 0.2)
dt_boxes = dt_boxes[dt_idx]
dt_scores = dt_scores[dt_idx]
dt_classes = dt_classes[dt_idx]
dt_detections_2 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)

# Apply custom non-max suppression
dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
dt_idx = custom_nms(dt_detections, threshold=0.5)
dt_boxes = dt_boxes[dt_idx]
dt_scores = dt_scores[dt_idx]
dt_classes = dt_classes[dt_idx]
dt_detections_3 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)

# Use boxes to label LiDAR points
dt_coords, dt_labels_1 = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

# Use labeled LiDAR points to filter boxes
idx = lidar_filter(dt_boxes, dt_coords, dt_labels_1, threshold=0.2)
dt_boxes, dt_labels_2 = dt_boxes[idx], dt_labels[idx]
dt_scores, dt_classes = dt_scores[idx], dt_classes[idx]
zero_boxes = np.zeros_like(dt_boxes)
zero_labels = np.zeros_like(dt_labels_2)
dt_boxes = np.concatenate((dt_boxes, zero_boxes))
dt_labels_2 = np.concatenate((dt_labels_2, zero_labels))

dt_detections_4 = sv.Detections(xyxy=dt_boxes, confidence=np.ones(len(dt_boxes)), class_id=np.zeros(len(dt_boxes), dtype='int'))

labels = [dt_labels_1, dt_labels_2]
detections = [dt_detections_3, dt_detections_4]
for i, detection in enumerate(detections):
  fig, axs = plt.subplots(1,1, figsize=(5,5))
  axs = show_as_mask(rgb_img, detection, dt_coords, labels[i], ax=axs, show_boxes=True)
  axs.axis('off')
  plt.tight_layout()
  plt.savefig(f'{filename} height filter {i}.jpg')
  plt.show()

# Number of Sampled LiDAR Points

In [ ]:
num_samples = list(range(5,51,5))

trials = {img[0]:{num:dict() for num in num_samples} for img in test_imgs}

In [ ]:
sample_folder = '/content/drive/MyDrive/Sam+LiDAR/Experiments/Sampled Points Tests'

sample_metrics = pd.read_csv(os.path.join(sample_folder, 'Box + LiDAR - Sampled Points Tests - All Results.csv'),
                             index_col=[0,1,2,3], header=[0,1])

sample_diff_metrics = pd.read_csv(os.path.join(sample_folder, 'Box + LiDAR - Sampled Points Tests - Avg by Difficulty.csv'),
                                  index_col=[0,1,2], header=[0,1])

sample_avg_metrics = pd.read_csv(os.path.join(sample_folder, 'Box + LiDAR - Sampled Points Tests - Avg by Method.csv'),
                                  index_col=[0,1], header=[0,1])

display(sample_avg_metrics)

In [ ]:
sample_folder = '/content/drive/MyDrive/Sam+LiDAR/Experiments/Sampled Points Tests'

num_samples = list(range(5,51,5))
point_methods = ['Manual + LiDAR', 'DeepForest + LiDAR', 'Detectree2 + LiDAR']
point_index = [(img[1], method, img[0], sample_size) for method in point_methods for img in test_imgs for sample_size in num_samples]
point_index = pd.MultiIndex.from_tuples(point_index, names=['Difficulty', 'Method', 'Filename', 'Sampled Points'])
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Segmentation','Precision'), ('Segmentation','Recall'), ('Segmentation','F1'),
           ('Segmentation','IoU')]
columns = pd.MultiIndex.from_tuples(columns)
point_metrics = pd.DataFrame(index=point_index, columns=columns)

for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)

  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()

  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()

  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]

  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter automatically detected boxes
  df_idx = lidar_filter(df_boxes, df_coords, df_labels)
  df_boxes, df_labels = df_boxes[df_idx], df_labels[df_idx]
  dt_idx = lidar_filter(dt_boxes, dt_coords, dt_labels)

  for sample_size in num_samples:
    # Sample LiDAR points for SAM-2 (given boxes)
    pos = sample_size
    neg = sample_size
    hand_sampled_coords, hand_sampled_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
    if len(df_boxes) > 0:
      df_sampled_coords, df_sampled_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
    if len(dt_boxes) > 0:
      dt_sampled_coords, dt_sampled_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

    # Segment using SAM-2 (mask threshold of 0.2)
    hand_masks = segment_box_points(sam_predictor, hand_boxes, hand_sampled_coords, hand_sampled_labels, threshold=0.2, iterative=True)
    if len(df_boxes) > 0:
      df_masks = segment_box_points(sam_predictor, df_boxes, df_sampled_coords, df_sampled_labels, threshold=0.2, iterative=True)
    else:
      df_masks = np.empty((0,400,400))
    if len(dt_boxes) > 0:
      dt_masks = segment_box_points(sam_predictor, dt_boxes, dt_sampled_coords, dt_sampled_labels, threshold=0.2, iterative=True)
    else:
      dt_masks = np.empty((0,400,400))

    # Match predicted masks to boxes (in case of filtered out masks)
    box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
    hand_boxes, hand_masks = hand_boxes[box_idx], hand_masks[mask_idx]
    box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
    df_boxes, df_masks = df_boxes[box_idx], df_masks[mask_idx]
    box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
    dt_boxes, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]

    # Calculate detection based metrics for masks using 50% threshold
    hand_detect_metrics = detection_metrics(true_masks, hand_masks)
    df_detect_metrics = detection_metrics(true_masks, df_masks)
    dt_detect_metrics = detection_metrics(true_masks, dt_masks)

    # For correct detections (based on boxes), calculate IoU and Segmentation metrics
    true_idx, hand_idx = hungarian_matching(true_boxes, hand_boxes, threshold=0.5)
    hand_iou_metric = compute_iou(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')
    hand_seg_metrics = segmentation_metrics(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')

    true_idx, df_idx = hungarian_matching(true_boxes, df_boxes, threshold=0.5)
    df_iou_metric = compute_iou(true_masks[true_idx], df_masks[df_idx], reduction='mean')
    df_seg_metrics = segmentation_metrics(true_masks[true_idx], df_masks[df_idx], reduction='mean')

    true_idx, dt_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
    dt_iou_metric = compute_iou(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')
    dt_seg_metrics = segmentation_metrics(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')

    # Store metrics
    point_metrics.loc[difficulty, 'Manual + LiDAR', filename, sample_size] = hand_detect_metrics + hand_seg_metrics + (hand_iou_metric,)
    point_metrics.loc[difficulty, 'DeepForest + LiDAR', filename, sample_size] = df_detect_metrics + df_seg_metrics + (df_iou_metric,)
    point_metrics.loc[difficulty, 'Detectree2 + LiDAR', filename, sample_size] = dt_detect_metrics + dt_seg_metrics + (dt_iou_metric,)

# Save Metrics
point_metrics.to_csv(os.path.join(sample_folder, 'Box + LiDAR - Sampled Points Tests - All Results.csv'))

point_order = [(difficulty, method, sample_size) for difficulty in ['Easy','Medium','Hard'] for method in point_methods for sample_size in num_samples]
point_diff_metrics = point_metrics.groupby(by=['Difficulty','Method','Sampled Points']).mean().reindex(point_order)
point_diff_metrics.to_csv(os.path.join(sample_folder, 'Box + LiDAR - Sampled Points Tests - Avg by Difficulty.csv'))

avg_order = [(method, sample_size) for method in point_methods for sample_size in num_samples]
point_avg_metrics = point_metrics.groupby(by=['Method', 'Sampled Points']).mean().reindex(avg_order)
point_avg_metrics.to_csv(os.path.join(sample_folder, 'Box + LiDAR - Sampled Points Tests - Avg by Method.csv'))

display(point_avg_metrics)

In [ ]:
sample_folder = '/content/drive/MyDrive/Sam+LiDAR/Experiments/Sampled Points Tests'

point_order = [(difficulty, method, sample_size) for difficulty in ['Easy','Medium','Hard'] for method in point_methods for sample_size in num_samples]
point_diff_stds = point_metrics.groupby(by=['Difficulty','Method','Sampled Points']).std().reindex(point_order)
point_diff_stds.to_csv(os.path.join(sample_folder, 'Box + LiDAR - Sampled Points Tests - Std by Difficulty.csv'))

avg_order = [(method, sample_size) for method in point_methods for sample_size in num_samples]
point_avg_stds = point_metrics.groupby(by=['Method', 'Sampled Points']).std().reindex(avg_order)
point_avg_stds.to_csv(os.path.join(sample_folder, 'Box + LiDAR - Sampled Points Tests - Std by Method.csv'))

In [ ]:
point_metrics.loc['Easy']

In [ ]:
avg_order = [(method, sample_size) for method in point_methods for sample_size in num_samples]
avg_order

In [ ]:
for filename, difficulty in test_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()
  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()
  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]
  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Use boxes to label LiDAR points
  #start_time = timeit.default_timer()
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)
  #print(f'Lidar label time: {timeit.default_timer() - start_time}')

  # Use labeled LiDAR points to filter boxes
  idx = lidar_filter(hand_boxes, hand_coords, hand_labels)
  hand_boxes, hand_labels = hand_boxes[idx], hand_labels[idx]
  idx = lidar_filter(df_boxes, df_coords, df_labels)
  df_boxes, df_labels = df_boxes[idx], df_labels[idx]
  idx = lidar_filter(dt_boxes, dt_coords, dt_labels)
  dt_boxes, dt_labels, dt_cnn_masks = dt_boxes[idx], dt_labels[idx], dt_cnn_masks[idx]

  for num in num_samples:
    # Sample LiDAR points for SAM-2
    pos = num
    neg = num
    hand_sample_coords, hand_sample_labels = sample_points(hand_coords, hand_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
    df_sample_coords, df_sample_labels = sample_points(df_coords, df_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)
    dt_sample_coords, dt_sample_labels = sample_points(dt_coords, dt_labels, pos_samples=pos, neg_samples=neg, distance_weight=True, neg_sample_spread=1, seed=seed)

    # Segment using SAM-2
    hand_masks = segment_box_points(sam_predictor, hand_boxes, hand_sample_coords, hand_sample_labels, threshold=0.2, iterative=True)
    df_masks = segment_box_points(sam_predictor, df_boxes, df_sample_coords, df_sample_labels, threshold=0.2, iterative=True)
    dt_masks = segment_box_points(sam_predictor, dt_boxes, dt_sample_coords, dt_sample_labels, threshold=0.2, iterative=True)

    # Match predicted masks to boxes
    box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
    hand_boxes, hand_masks = hand_boxes[box_idx], hand_masks[mask_idx]
    box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
    df_boxes, df_masks = df_boxes[box_idx], df_masks[mask_idx]
    box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
    dt_boxes, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]

    # Calculate detection based metrics using 50% threshold on masks
    hand_detect_metrics = detection_metrics(true_masks, hand_masks)
    df_detect_metrics = detection_metrics(true_masks, df_masks)
    dt_detect_metrics = detection_metrics(true_masks, dt_masks)
    dt_cnn_detect_metrics = detection_metrics(true_masks, dt_cnn_masks)

    # For correct detections (based on boxes), calculate IoU and pixel metrics
    true_idx, hand_idx = hungarian_matching(true_boxes, hand_boxes, threshold=0.5)
    hand_iou_metric = compute_iou(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')
    hand_pixel_metrics = pixel_metrics(true_masks[true_idx], hand_masks[hand_idx], reduction='mean')

    true_idx, df_idx = hungarian_matching(true_boxes, df_boxes, threshold=0.5)
    df_iou_metric = compute_iou(true_masks[true_idx], df_masks[df_idx], reduction='mean')
    df_pixel_metrics = pixel_metrics(true_masks[true_idx], df_masks[df_idx], reduction='mean')

    true_idx, dt_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
    dt_iou_metric = compute_iou(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')
    dt_pixel_metrics = pixel_metrics(true_masks[true_idx], dt_masks[dt_idx], reduction='mean')

    true_idx, dt_cnn_idx = hungarian_matching(true_boxes, dt_boxes, threshold=0.5)
    dt_cnn_iou_metric = compute_iou(true_masks[true_idx], dt_cnn_masks[dt_cnn_idx], reduction='mean')
    dt_cnn_pixel_metrics = pixel_metrics(true_masks[true_idx], dt_cnn_masks[dt_cnn_idx], reduction='mean')

    # Store metrics
    trials[filename][num]['hand_points'] = hand_detect_metrics + (hand_iou_metric,) + hand_pixel_metrics
    trials[filename][num]['df_points'] = df_detect_metrics + (df_iou_metric,) + df_pixel_metrics
    trials[filename][num]['dt_points'] = dt_detect_metrics + (dt_iou_metric,) + dt_pixel_metrics

# Put metrics in array indexed by (in order of dimension) Image, Number Samples, Prompt, and Metric
trials_array = np.zeros(shape=(3,3,6,7))
for i, img in enumerate(test_imgs):
  for j, num in enumerate(num_samples):
    for k, prompt in enumerate(['hand_points', 'df_points', 'dt_points']):
      trials_array[i,j,k] = trials[img[0]][num][prompt]

# Separate out Easy, Medium, Hard, and all metrics, and take mean across images
avg_trials_array = np.nanmean(trials_array, axis=0)

In [ ]:
hand_labels.shape

In [ ]:
easy_imgs = [img for img in test_imgs if img[1]=='Easy']

for filename, difficulty in easy_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()
  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()
  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]
  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  dt_detections_1 = sv.Detections(xyxy=dt_boxes,
                                  confidence=np.ones(len(dt_boxes)),
                                  class_id=np.zeros(len(dt_boxes), dtype='int64'))

  # Use labeled LiDAR points to filter boxes
  hand_boxes_2, hand_labels_2 = lidar_filter(hand_boxes, hand_coords, hand_labels)
  df_boxes_2, df_labels_2 = lidar_filter(df_boxes, df_coords, df_labels)
  dt_boxes_2, dt_labels_2 = lidar_filter(dt_boxes, dt_coords, dt_labels)

  dt_detections_2 = sv.Detections(xyxy=dt_boxes_2,
                                  confidence=np.ones(len(dt_boxes_2)),
                                  class_id=np.zeros(len(dt_boxes_2), dtype='int64'))

  # Segment using SAM-2
  hand_masks = segment_boxes(sam_predictor, hand_boxes, threshold=0.2)
  df_masks = segment_boxes(sam_predictor, df_boxes, threshold=0.2)
  dt_masks = segment_boxes(sam_predictor, dt_boxes, threshold=0.2)

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes_3, hand_masks = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes_3, df_masks = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes_3, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]
  dt_coords_3, dt_labels_3 = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes_3)

  dt_detections_3 = sv.Detections(xyxy=dt_boxes_3,
                                  confidence=np.ones(len(dt_boxes_3)),
                                  class_id=np.zeros(len(dt_boxes_3), dtype='int64'))

  # Visualize rasterized LiDAR points
  fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(10,5))
  axs[0] = show_as_mask(rgb_img, dt_detections_1, dt_coords, dt_labels, ax=axs[0], show_boxes=True)
  axs[1] = show_as_mask(rgb_img, dt_detections_2, dt_coords, dt_labels_2, ax=axs[1], show_boxes=True)
  axs[2] = show_as_mask(rgb_img, dt_detections_3, dt_coords_3, dt_labels_3, ax=axs[2], show_boxes=True)
  axs[0].set_title('Original Predictions')
  axs[1].set_title('LiDAR Filter')
  axs[2].set_title('Confidence Filter')
  plt.tight_layout()
  plt.show()
  plt.close()

In [ ]:
  hand_detect_metrics_2 = detection_metrics_2(true_masks, hand_masks_2)
  df_detect_metrics_2 = detection_metrics_2(true_masks, df_masks_2)
  dt_detect_metrics_2 = detection_metrics_2(true_masks, dt_masks_2)

In [ ]:
easy_imgs = [img for img in test_imgs if img[1]=='Easy']

for filename, difficulty in easy_imgs:
  # Get image
  img = ds.get_image(filename)
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  sam_predictor.set_image(rgb_img)

  # Collect true bounding boxes and masks
  true_boxes = img['annotation']
  true_masks = get_masks(filename, rgb_folder, os.path.join(shape_folder, f'{difficulty}_annotate'))

  # Align true bounding boxes and masks
  box_idx, mask_idx = hungarian_matching(true_boxes, true_masks)
  true_boxes = true_boxes[box_idx]
  true_masks = true_masks[mask_idx]

  # Collect hand-drawn bounding boxes
  hand_boxes = img['annotation']

  # Generate automatic bounding boxes using DeepForest
  df_boxes = df_model.predict_image(image=rgb_img.astype('float32'), return_plot=False)
  # Filter with confidence threshold of 0.2
  df_boxes = df_boxes[df_boxes['score'] >= 0.2]
  df_scores = df_boxes['score'].to_numpy()
  df_boxes = df_boxes[['xmin','ymin','xmax','ymax']].to_numpy()
  # Apply custom non-max suppression
  df_detections = sv.Detections(xyxy=df_boxes, confidence=df_scores, class_id=np.zeros(len(df_scores),dtype='int'))
  df_idx = custom_nms(df_detections, threshold=0.5)
  df_boxes = df_boxes[df_idx]
  df_scores = df_scores[df_idx]

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_cnn_masks = dt_outputs['instances'].pred_masks.cpu().numpy()
  # Filter with confidence threshold of 0.2
  dt_idx = np.nonzero(dt_scores >= 0.2)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]
  # Apply custom non-max suppression
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_cnn_masks = dt_cnn_masks[dt_idx]

  # Segment using SAM-2
  hand_masks = segment_boxes(sam_predictor, hand_boxes, threshold=0.2)
  df_masks = segment_boxes(sam_predictor, df_boxes, threshold=0.2)
  dt_masks = segment_boxes(sam_predictor, dt_boxes, threshold=0.2)

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes, hand_masks)
  hand_boxes, hand_masks = hand_boxes[box_idx], hand_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes, df_masks)
  df_boxes, df_masks = df_boxes[box_idx], df_masks[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes, dt_masks)
  dt_boxes, dt_masks = dt_boxes[box_idx], dt_masks[mask_idx]

  # Use boxes to label LiDAR points
  hand_coords, hand_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=hand_boxes)
  df_coords, df_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=df_boxes)
  dt_coords, dt_labels = rasterize_lidar(lidar_folder, rgb_folder, filename, boxes=dt_boxes)

  # Use labeled LiDAR points to filter boxes
  hand_boxes_2, hand_labels_2 = lidar_filter(hand_boxes, hand_coords, hand_labels)
  df_boxes_2, df_labels_2 = lidar_filter(df_boxes, df_coords, df_labels)
  dt_boxes_2, dt_labels_2 = lidar_filter(dt_boxes, dt_coords, dt_labels)


  # Segment using SAM-2
  hand_masks_2 = segment_boxes(sam_predictor, hand_boxes_2, threshold=0.2)
  df_masks_2 = segment_boxes(sam_predictor, df_boxes_2, threshold=0.2)
  dt_masks_2 = segment_boxes(sam_predictor, dt_boxes_2, threshold=0.2)

  # Match predicted masks to boxes
  box_idx, mask_idx = hungarian_matching(hand_boxes_2, hand_masks_2)
  hand_boxes_2, hand_masks_2 = hand_boxes_2[box_idx], hand_masks_2[mask_idx]
  box_idx, mask_idx = hungarian_matching(df_boxes_2, df_masks_2)
  df_boxes_2, df_masks_2 = df_boxes_2[box_idx], df_masks_2[mask_idx]
  box_idx, mask_idx = hungarian_matching(dt_boxes_2, dt_masks_2)
  dt_boxes_2, dt_masks_2 = dt_boxes_2[box_idx], dt_masks_2[mask_idx]

  hand_detect_metrics = detection_metrics(true_masks, hand_masks)
  df_detect_metrics = detection_metrics(true_masks, df_masks)
  dt_detect_metrics = detection_metrics(true_masks, dt_masks)

  hand_detect_metrics_2 = detection_metrics(true_masks, hand_masks_2)
  df_detect_metrics_2 = detection_metrics(true_masks, df_masks_2)
  dt_detect_metrics_2 = detection_metrics(true_masks, dt_masks_2)

  dt_detections_1 = sv.Detections(xyxy=true_boxes,
                                  confidence=np.ones(len(true_boxes)),
                                  class_id=np.zeros(len(true_boxes), dtype='int'),
                                  mask=true_masks)

  dt_detections_2 = sv.Detections(xyxy=dt_boxes,
                                  confidence=np.ones(len(dt_boxes)),
                                  class_id=np.zeros(len(dt_boxes), dtype='int'),
                                  mask=dt_masks.astype('bool'))

  dt_detections_3 = sv.Detections(xyxy=dt_boxes_2,
                                  confidence=np.ones(len(dt_boxes_2)),
                                  class_id=np.zeros(len(dt_boxes_2), dtype='int'),
                                  mask=dt_masks_2.astype('bool'))

  box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.RED)
  mask_annotator = sv.MaskAnnotator()

  # Visualize rasterized LiDAR points
  fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(10,5))
  img_1 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_1)
  img_1 = mask_annotator.annotate(img_1, detections=dt_detections_1)[:,:,::-1]
  img_2 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_2)
  img_2 = mask_annotator.annotate(img_2, detections=dt_detections_2)[:,:,::-1]
  img_3 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_3)
  img_3 = mask_annotator.annotate(img_3, detections=dt_detections_3)[:,:,::-1]
  axs[0].imshow(img_1)
  axs[1].imshow(img_2)
  axs[2].imshow(img_3)
  axs[0].set_title('Ground Truth')
  axs[1].set_title('Confidence Filter')
  axs[2].set_title('LiDAR Filter')
  for ax in axs:
    ax.axis('off')
  plt.tight_layout()
  plt.show()
  plt.close()

  print(dt_detect_metrics)
  print(dt_detect_metrics_2)

In [ ]:
dt_detections_1

In [ ]:
dt_detections_2

In [ ]:
dt_detections_3

In [ ]:
  img_3 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_3)
  img_3 = mask_annotator.annotate(img_3, detections=dt_detections_3)

In [ ]:
def lidar_filter(boxes, coordinates, labels):
    '''
    Remove boxes with low numbers of corresponding Lidar points. This is calculated by
    taking the average number of points per pixel in each box (a value between 0 and 1
    for every box), calculating the Inter-Quartile Region (IQR) for all boxes in the
    image, and removing boxes where the average number of points per pixel falls more
    than 1.5 x IQR below the lowest quartile.

    params:
      boxes (ndarray): N x 4 array of bounding boxes, where N is the number of boxes
      coordinates (ndarray): 1 x P x 2 array of X,Y coordinates, where P is the number
                             of points.
      labels (ndarray): N x P array of binary labels, where the order of N is based on
                        the order of the N bounding boxes and the order of P is based on
                        the order of the P coordinates.

    returns:
      boxes (ndarray): The remaining boxes after filtering
      labels (ndarray): The remaining labels after filtering
    '''
    def box_area(box):
        return (box[2] - box[0]) * (box[3] - box[1])

    # Remove boxes with no positive LiDAR points
    idx_filter = [i for i in range(len(labels)) if labels[i].sum() > 0]
    boxes = boxes[idx_filter]
    labels = labels[idx_filter]

    # Find average number of points per pixel in remaining boxes, calculate IQR
    avg_points = [labels[i].sum()/box_area(boxes[i]) for i in range(len(boxes))]
    q75, q25 = np.percentile(avg_points, [75 ,25])
    iqr = q75 - q25

    # Remove boxes with abnoramlly low numbers of points
    idx_filter = [i for i in range(len(labels)) if avg_points[i] >= q25 - 1.5 * iqr]
    boxes = boxes[idx_filter]
    labels = labels[idx_filter]

    return boxes, labels

In [ ]:
hand_boxes_2, hand_labels_2 = lidar_filter(hand_boxes, hand_coords, hand_labels)
hand_boxes_2, hand_labels_2

In [ ]:
cfg = setup_cfg(update_model='/content/detectree2_weights/230729_05dates.pth')
dt_model=DefaultPredictor(cfg)

easy_imgs = [img for img in test_imgs if img[1]=='Easy']

for filename in easy_imgs:
  img = ds.get_image(filename[0])
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  true_boxes = img['annotation']
  print(filename[0], filename[1])
  print('True Boxes:', len(true_boxes))

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_masks_cnn = dt_outputs['instances'].pred_masks.cpu().numpy()
  dt_detections_0 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_0 = len(dt_detections_0)
  dt_threshold = 0.2
  dt_idx = np.nonzero(dt_scores >= dt_threshold)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_masks_cnn = dt_masks_cnn[dt_idx]
  dt_detections_1 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_1 = len(dt_detections_1)
  #dt_detections = np.hstack((dt_boxes, dt_scores[...,None]))
  #dt_idx = non_max_suppression(dt_detections, iou_threshold=0.5)
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_masks_cnn = dt_masks_cnn[dt_idx]
  dt_detections_2 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_2 = len(dt_detections_2)

  true_detections = sv.Detections(xyxy=true_boxes, confidence=np.ones(len(true_boxes)), class_id=np.zeros(len(true_boxes)))

  box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.RED)

  fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(15,5))
  boxes_0 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_0)[:,:,::-1]
  boxes_1 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_1)[:,:,::-1]
  boxes_2 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_2)[:,:,::-1]
  boxes_true = box_annotator.annotate(bgr_img.copy(), detections=true_detections)[:,:,::-1]
  axs[0].imshow(boxes_0)
  axs[0].set_title(f'DT Model Initial Predictions: {length_0}')
  axs[1].imshow(boxes_1)
  axs[1].set_title(f'DT Model Confidence Threshold: {length_1}')
  axs[2].imshow(boxes_2)
  axs[2].set_title(f'DT Model Non-Max Suppression: {length_2}')
  axs[3].imshow(boxes_true)
  axs[3].set_title(f'True Boxes: {len(true_boxes)}')
  for i in range(len(axs)):
    axs[i].axis('off')
  plt.tight_layout()
  plt.show()

In [ ]:
cfg = setup_cfg(update_model='/content/detectree2_weights/230729_05dates.pth')
dt_model=DefaultPredictor(cfg)

medium_imgs = [img for img in test_imgs if img[1]=='Medium']

for filename in medium_imgs:
  img = ds.get_image(filename[0])
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  true_boxes = img['annotation']
  print(filename[0], filename[1])
  print('True Boxes:', len(true_boxes))

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_masks_cnn = dt_outputs['instances'].pred_masks.cpu().numpy()
  dt_detections_0 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_0 = len(dt_detections_0)
  dt_threshold = 0.2
  dt_idx = np.nonzero(dt_scores >= dt_threshold)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_masks_cnn = dt_masks_cnn[dt_idx]
  dt_detections_1 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_1 = len(dt_detections_1)
  #dt_detections = np.hstack((dt_boxes, dt_scores[...,None]))
  #dt_idx = non_max_suppression(dt_detections, iou_threshold=0.5)
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_masks_cnn = dt_masks_cnn[dt_idx]
  dt_detections_2 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_2 = len(dt_detections_2)

  true_detections = sv.Detections(xyxy=true_boxes, confidence=np.ones(len(true_boxes)), class_id=np.zeros(len(true_boxes)))

  box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.RED)

  fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(15,5))
  boxes_0 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_0)[:,:,::-1]
  boxes_1 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_1)[:,:,::-1]
  boxes_2 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_2)[:,:,::-1]
  boxes_true = box_annotator.annotate(bgr_img.copy(), detections=true_detections)[:,:,::-1]
  axs[0].imshow(boxes_0)
  axs[0].set_title(f'DT Model Initial Predictions: {length_0}')
  axs[1].imshow(boxes_1)
  axs[1].set_title(f'DT Model Confidence Threshold: {length_1}')
  axs[2].imshow(boxes_2)
  axs[2].set_title(f'DT Model Non-Max Suppression: {length_2}')
  axs[3].imshow(boxes_true)
  axs[3].set_title(f'True Boxes: {len(true_boxes)}')
  for i in range(len(axs)):
    axs[i].axis('off')
  plt.tight_layout()
  plt.show()

In [ ]:
cfg = setup_cfg(update_model='/content/detectree2_weights/230729_05dates.pth')
dt_model=DefaultPredictor(cfg)

hard_imgs = [img for img in test_imgs if img[1]=='Hard']

for filename in hard_imgs:
  img = ds.get_image(filename[0])
  rgb_img = img['rgb']
  bgr_img = rgb_img[:,:,::-1].copy()
  true_boxes = img['annotation']
  print(filename[0], filename[1])
  print('True Boxes:', len(true_boxes))

  # Generate automatic bounding boxes using Detectree2
  dt_outputs = dt_model(bgr_img)
  dt_boxes = dt_outputs['instances'].pred_boxes.tensor.cpu().numpy()
  dt_scores = dt_outputs['instances'].scores.cpu().numpy()
  dt_classes = dt_outputs['instances'].pred_classes.cpu().numpy()
  dt_masks_cnn = dt_outputs['instances'].pred_masks.cpu().numpy()
  dt_detections_0 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_0 = len(dt_detections_0)
  dt_threshold = 0.2
  dt_idx = np.nonzero(dt_scores >= dt_threshold)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_masks_cnn = dt_masks_cnn[dt_idx]
  dt_detections_1 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_1 = len(dt_detections_1)
  #dt_detections = np.hstack((dt_boxes, dt_scores[...,None]))
  #dt_idx = non_max_suppression(dt_detections, iou_threshold=0.5)
  dt_detections = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  dt_idx = custom_nms(dt_detections, threshold=0.5)
  dt_boxes = dt_boxes[dt_idx]
  dt_scores = dt_scores[dt_idx]
  dt_classes = dt_classes[dt_idx]
  dt_masks_cnn = dt_masks_cnn[dt_idx]
  dt_detections_2 = sv.Detections(xyxy=dt_boxes, confidence=dt_scores, class_id=dt_classes)
  length_2 = len(dt_detections_2)

  true_detections = sv.Detections(xyxy=true_boxes, confidence=np.ones(len(true_boxes)), class_id=np.zeros(len(true_boxes)))

  box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.RED)

  fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(15,5))
  boxes_0 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_0)[:,:,::-1]
  boxes_1 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_1)[:,:,::-1]
  boxes_2 = box_annotator.annotate(bgr_img.copy(), detections=dt_detections_2)[:,:,::-1]
  boxes_true = box_annotator.annotate(bgr_img.copy(), detections=true_detections)[:,:,::-1]
  axs[0].imshow(boxes_0)
  axs[0].set_title(f'DT Model Initial Predictions: {length_0}')
  axs[1].imshow(boxes_1)
  axs[1].set_title(f'DT Model Confidence Threshold: {length_1}')
  axs[2].imshow(boxes_2)
  axs[2].set_title(f'DT Model Non-Max Suppression: {length_2}')
  axs[3].imshow(boxes_true)
  axs[3].set_title(f'True Boxes: {len(true_boxes)}')
  for i in range(len(axs)):
    axs[i].axis('off')
  plt.tight_layout()
  plt.show()

# All Results

In [ ]:
# Show metrics
columns = [('Detection','Precision'), ('Detection','Recall'),('Detection','F1'),
           ('Pixel','IoU'), ('Pixel','Precision'), ('Pixel','Recall'), ('Pixel','F1')]
columns = pd.MultiIndex.from_tuples(columns)
index = ['Hand-Drawn Boxes', 'Hand-Drawn Boxes + Points', 'Hand-Drawn Boxes + Points (Positive)',
         'Auto-Drawn Boxes', 'Auto-Drawn Boxes + Points', 'Auto-Drawn Boxes + Points (Positive)',
         'LiDAR Points', 'LiDAR-Drawn Boxes + Points', 'LiDAR-Drawn Boxes + Points (Positive)',
         'LiDAR Baseline']
metrics = pd.DataFrame.from_dict(all_metrics, orient='index', columns=columns)
metrics = metrics.reindex(index)
metrics

In [ ]:
filename = 'BART_050_2019'
chm_folder = '/content/weecology-NeonTreeEvaluation-d0b90bc/evaluation/CHM'

with rasterio.open(os.path.join(chm_folder, filename+'_CHM.tif')) as rast_img:
  print(rast_img.shape)

In [ ]:
import shutil
shutil.copyfile(os.path.join(chm_folder, filename+'_CHM.tif'), f'/content/{filename}_CHM.tif')

In [ ]:
shutil.copyfile(os.path.join(lidar_folder, filename+'.laz'), f'/content/{filename}.laz')